In [ ]:
!pip install -U transformers trl peft bitsandbytes datasets accelerate rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 147.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 52.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
  Attempting uninstal

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re
import json
import math
from collections import Counter

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import load_dataset
from rank_bm25 import BM25Okapi

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)


DATA_DIR = "/content/drive/MyDrive/hedge_run"

INPUT_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_1000.json"
)

PROGRESS_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_verification_progress.json"
)

OUTPUT_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_verified_1000.json"
)

OUTPUT_CSV = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_features_17.csv"
)


NLI_MODEL_ID = (
    "MoritzLaurer/"
    "DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
)


# Same retrieval size as generation.
RETRIEVE_K = 10


SUPPORT_THRESHOLD = 0.50
CONTRADICTION_THRESHOLD = 0.50


# NLI inference.
NLI_BATCH_SIZE = 10
NLI_MAX_LENGTH = 512


SAVE_EVERY = 10


LIMIT = None


if not os.path.exists(INPUT_JSON):

    raise FileNotFoundError(
        INPUT_JSON
    )


with open(
    INPUT_JSON,
    "r",
    encoding="utf-8",
) as f:

    generated = json.load(f)


assert len(generated) == 1000


print(
    "Loaded frozen Hotpot generations:",
    len(generated)
)


print(
    "dev:",
    sum(
        r["split"] == "dev"
        for r in generated
    )
)


print(
    "test:",
    sum(
        r["split"] == "test"
        for r in generated
    )
)


assert (
    sum(
        r["split"] == "dev"
        for r in generated
    )
    ==
    800
)


assert (
    sum(
        r["split"] == "test"
        for r in generated
    )
    ==
    200
)




assert all(
    r["generation_protocol"]["version"]
    ==
    "passive_iterative_v2"

    for r in generated
)




print(
    "\nLoading HotpotQA distractor validation split..."
)


hotpot = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor",
    split="validation",
)


question_to_example = {
    ex["question"]: ex
    for ex in hotpot
}


missing_context = [
    r["question"]
    for r in generated
    if r["question"] not in question_to_example
]


if missing_context:

    raise RuntimeError(
        f"Could not recover context for "
        f"{len(missing_context)} questions."
    )


print(
    "Recovered all Hotpot contexts."
)



def bm25_tokens(
    text,
):

    return re.findall(
        r"\w+",
        str(text).lower(),
    )


def flatten_hotpot_context(
    example,
):
    """
    Return ONLY context sentences.

    Supporting-fact labels are deliberately ignored.
    """

    groups = (
        example[
            "context"
        ][
            "sentences"
        ]
    )

    sentences = []

    for group in groups:

        for sentence in group:

            sentence = (
                str(sentence)
                .strip()
            )

            if sentence:

                sentences.append(
                    sentence
                )

    return sentences


def build_bm25_for_example(
    example,
):

    sentences = (
        flatten_hotpot_context(
            example
        )
    )

    if not sentences:

        return (
            [],
            None,
        )

    bm25 = BM25Okapi(
        [
            bm25_tokens(sentence)
            for sentence in sentences
        ]
    )

    return (
        sentences,
        bm25,
    )


def retrieve_top_k(
    sentences,
    bm25,
    query,
    k=RETRIEVE_K,
):

    if (
        not sentences
        or bm25 is None
    ):

        return []

    query_tokens = (
        bm25_tokens(
            query
        )
    )

    scores = (
        bm25.get_scores(
            query_tokens
        )
    )

    order = (
        np.argsort(
            scores
        )[::-1][:k]
    )

    return [
        sentences[
            int(i)
        ]
        for i in order
    ]


print(
    "\nLoading NLI verifier..."
)


nli_tokenizer = (
    AutoTokenizer.from_pretrained(
        NLI_MODEL_ID
    )
)


if torch.cuda.is_available():

    NLI_DEVICE = torch.device(
        "cuda"
    )

    nli_dtype = torch.float16

else:

    NLI_DEVICE = torch.device(
        "cpu"
    )

    nli_dtype = torch.float32


try:

    nli_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            NLI_MODEL_ID,
            dtype=nli_dtype,
        )
    )

except TypeError:

    nli_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            NLI_MODEL_ID,
            torch_dtype=nli_dtype,
        )
    )


nli_model = (
    nli_model.to(
        NLI_DEVICE
    )
)

nli_model.eval()


print(
    "NLI device:",
    NLI_DEVICE
)


print(
    "NLI labels:",
    nli_model.config.id2label
)



id2label = {
    int(k):
        str(v).lower()
    for k, v
    in nli_model.config.id2label.items()
}


def find_label_index(
    target,
):

    target = target.lower()

    for idx, label in id2label.items():

        if target in label:

            return idx

    raise RuntimeError(
        f"Cannot find '{target}' in "
        f"id2label={id2label}"
    )


ENT_IDX = find_label_index(
    "entail"
)

CON_IDX = find_label_index(
    "contrad"
)

NEU_IDX = find_label_index(
    "neutral"
)


print(
    "ENT index:",
    ENT_IDX
)

print(
    "NEU index:",
    NEU_IDX
)

print(
    "CON index:",
    CON_IDX
)


@torch.no_grad()
def nli_score_pairs(
    premises,
    hypothesis,
):
    """
    Score multiple evidence sentences against ONE hypothesis.

    Returns one dictionary per evidence sentence.
    """

    if not premises:
        return []


    results = []


    for start in range(
        0,
        len(premises),
        NLI_BATCH_SIZE,
    ):

        batch_premises = premises[
            start:
            start + NLI_BATCH_SIZE
        ]


        hypotheses = [
            hypothesis
        ] * len(
            batch_premises
        )


        encoded = nli_tokenizer(
            batch_premises,
            hypotheses,
            padding=True,
            truncation=True,
            max_length=NLI_MAX_LENGTH,
            return_tensors="pt",
        )


        encoded = {
            key:
                value.to(
                    NLI_DEVICE
                )
            for key, value
            in encoded.items()
        }


        logits = (
            nli_model(
                **encoded
            )
            .logits
            .float()
        )


        probs = (
            torch.softmax(
                logits,
                dim=-1,
            )
            .cpu()
            .numpy()
        )


        for evidence, p in zip(
            batch_premises,
            probs,
        ):

            results.append(
                {
                    "evidence":
                        evidence,

                    "entailment":
                        float(
                            p[
                                ENT_IDX
                            ]
                        ),

                    "neutral":
                        float(
                            p[
                                NEU_IDX
                            ]
                        ),

                    "contradiction":
                        float(
                            p[
                                CON_IDX
                            ]
                        ),
                }
            )


    return results


def aggregate_nli(
    pair_scores,
):

    if not pair_scores:

        return {
            "support":
                0.0,

            "contradiction":
                0.0,

            "neutral":
                1.0,

            "best_support_evidence":
                "",

            "best_contradiction_evidence":
                "",
        }


    support_idx = int(
        np.argmax(
            [
                x[
                    "entailment"
                ]
                for x
                in pair_scores
            ]
        )
    )


    contradiction_idx = int(
        np.argmax(
            [
                x[
                    "contradiction"
                ]
                for x
                in pair_scores
            ]
        )
    )


    neutral_idx = int(
        np.argmax(
            [
                x[
                    "neutral"
                ]
                for x
                in pair_scores
            ]
        )
    )


    return {
        "support":
            float(
                pair_scores[
                    support_idx
                ][
                    "entailment"
                ]
            ),

        "contradiction":
            float(
                pair_scores[
                    contradiction_idx
                ][
                    "contradiction"
                ]
            ),

        "neutral":
            float(
                pair_scores[
                    neutral_idx
                ][
                    "neutral"
                ]
            ),

        "best_support_evidence":
            pair_scores[
                support_idx
            ][
                "evidence"
            ],

        "best_contradiction_evidence":
            pair_scores[
                contradiction_idx
            ][
                "evidence"
            ],
    }



def step_label(
    support,
    contradiction,
):

    if (
        support
        >=
        SUPPORT_THRESHOLD
    ):

        return "supported"


    if (
        contradiction
        >=
        CONTRADICTION_THRESHOLD
    ):

        return "contradicted"


    return "unclear"


def verify_reasoning_step(
    step_index,
    step_text,
    sentences,
    bm25,
):

    retrieved = retrieve_top_k(
        sentences,
        bm25,
        step_text,
        RETRIEVE_K,
    )


    pair_scores = nli_score_pairs(
        retrieved,
        step_text,
    )


    agg = aggregate_nli(
        pair_scores
    )


    label = step_label(
        agg["support"],
        agg["contradiction"],
    )


    return {
        "step_index":
            int(
                step_index
            ),

        "step":
            step_text,

        "retrieval_query":
            step_text,

        "retrieved_evidence":
            retrieved,

        "support":
            float(
                agg[
                    "support"
                ]
            ),

        "contradiction":
            float(
                agg[
                    "contradiction"
                ]
            ),

        "neutral":
            float(
                agg[
                    "neutral"
                ]
            ),

        "label":
            label,

        "best_support_evidence":
            agg[
                "best_support_evidence"
            ],

        "best_contradiction_evidence":
            agg[
                "best_contradiction_evidence"
            ],
    }


#11 REASONING FEATURES
# Cmax  = max contradiction
# Cmean = mean contradiction
# Icon  = any contradicted
# Fcon  = fraction contradicted
# Smin  = minimum support
# Smean = mean support
# Fsup  = fraction supported
# Sspread = max support - min support
# Funclear = fraction unclear
# Cconflict = max support * max contradiction
# LR = number of reasoning steps


def reasoning_features(
    step_reports,
):

    m = len(
        step_reports
    )


    if m == 0:

        return {
            "max_contradiction_score":
                0.0,

            "any_contradicted":
                0,

            "min_support_score":
                0.0,

            "num_steps":
                0,

            "frac_supported":
                0.0,

            "frac_contradicted":
                0.0,

            "frac_unclear":
                1.0,

            "mean_support":
                0.0,

            "mean_contradiction":
                0.0,

            "conflict":
                0.0,

            "support_spread":
                0.0,
        }


    support = np.asarray(
        [
            r["support"]
            for r in step_reports
        ],
        dtype=float,
    )


    contradiction = np.asarray(
        [
            r["contradiction"]
            for r in step_reports
        ],
        dtype=float,
    )


    labels = [
        r["label"]
        for r in step_reports
    ]


    max_support = float(
        support.max()
    )


    min_support = float(
        support.min()
    )


    max_contradiction = float(
        contradiction.max()
    )


    return {
        "max_contradiction_score":
            max_contradiction,

        "any_contradicted":
            int(
                any(
                    label
                    ==
                    "contradicted"

                    for label
                    in labels
                )
            ),

        "min_support_score":
            min_support,

        "num_steps":
            int(
                m
            ),

        "frac_supported":
            float(
                np.mean(
                    [
                        label
                        ==
                        "supported"

                        for label
                        in labels
                    ]
                )
            ),

        "frac_contradicted":
            float(
                np.mean(
                    [
                        label
                        ==
                        "contradicted"

                        for label
                        in labels
                    ]
                )
            ),

        "frac_unclear":
            float(
                np.mean(
                    [
                        label
                        ==
                        "unclear"

                        for label
                        in labels
                    ]
                )
            ),

        "mean_support":
            float(
                support.mean()
            ),

        "mean_contradiction":
            float(
                contradiction.mean()
            ),

        "conflict":
            float(
                max_support
                *
                max_contradiction
            ),

        "support_spread":
            float(
                max_support
                -
                min_support
            ),
    }



def make_final_claim(
    question,
    answer,
):

    question = (
        str(question)
        .strip()
    )


    answer = (
        str(answer)
        .strip()
    )


    if not answer:
        return ""


    return (
        f'The answer to the question '
        f'"{question}" is "{answer}".'
    )


def verify_final_answer(
    row,
    sentences,
    bm25,
):

    question = (
        row[
            "question"
        ]
    )



    answer = (
        str(
            row[
                "majority_answer"
            ]
        )
        .strip()
    )


    claim = make_final_claim(
        question,
        answer,
    )


    if not claim:

        return {
            "final_answer":
                "",

            "final_claim":
                "",

            "retrieved_evidence":
                [],

            "support":
                0.0,

            "contradiction":
                0.0,

            "contradicted":
                0,

            "label":
                "unclear",

            "best_support_evidence":
                "",

            "best_contradiction_evidence":
                "",
        }



    retrieved = [
        str(x).strip()
        for x
        in row.get(
            "question_evidence",
            []
        )
        if str(x).strip()
    ]


    if not retrieved:

        retrieved = retrieve_top_k(
            sentences,
            bm25,
            question,
            RETRIEVE_K,
        )


    pair_scores = nli_score_pairs(
        retrieved,
        claim,
    )


    agg = aggregate_nli(
        pair_scores
    )


    support = float(
        agg[
            "support"
        ]
    )


    contradiction = float(
        agg[
            "contradiction"
        ]
    )


    label = step_label(
        support,
        contradiction,
    )


    return {
        "final_answer":
            answer,

        "final_claim":
            claim,

        "retrieved_evidence":
            retrieved,

        "support":
            support,

        "contradiction":
            contradiction,

        # Independent binary final contradiction feature.
        "contradicted":
            int(
                contradiction
                >=
                CONTRADICTION_THRESHOLD
            ),

        "label":
            label,

        "best_support_evidence":
            agg[
                "best_support_evidence"
            ],

        "best_contradiction_evidence":
            agg[
                "best_contradiction_evidence"
            ],
    }



def verify_question(
    row,
):

    question = (
        row[
            "question"
        ]
    )


    example = (
        question_to_example[
            question
        ]
    )


    sentences, bm25 = (
        build_bm25_for_example(
            example
        )
    )


    selected_steps = [
        str(step).strip()
        for step
        in row.get(
            "selected_steps",
            []
        )
        if str(step).strip()
    ]


    step_reports = []


    for step_index, step_text in enumerate(
        selected_steps,
        start=1,
    ):

        report = (
            verify_reasoning_step(
                step_index,
                step_text,
                sentences,
                bm25,
            )
        )


        step_reports.append(
            report
        )




    r_features = (
        reasoning_features(
            step_reports
        )
    )




    final_report = (
        verify_final_answer(
            row,
            sentences,
            bm25,
        )
    )



    features = {



        **r_features,




        "self_consistency":
            float(
                row[
                    "self_consistency"
                ]
            ),

        "answer_norm_entropy":
            float(
                row[
                    "answer_norm_entropy"
                ]
            ),

        "answer_logprob":
            float(
                row[
                    "answer_logprob"
                ]
            ),




        "fa_support_retr":
            float(
                final_report[
                    "support"
                ]
            ),

        "fa_contradiction_retr":
            float(
                final_report[
                    "contradiction"
                ]
            ),

        "fa_contradicted_retr":
            int(
                final_report[
                    "contradicted"
                ]
            ),
    }


    return {
        "question_index":
            int(
                row[
                    "question_index"
                ]
            ),

        "split":
            row[
                "split"
            ],

        "question":
            question,

        "gold_answer":
            row[
                "gold_answer"
            ],


        "majority_answer":
            row[
                "majority_answer"
            ],

        "loop_final_answer":
            row.get(
                "loop_final_answer",
                "",
            ),

        "selected_steps":
            selected_steps,

        "step_reports":
            step_reports,

        "final_answer_verification":
            final_report,

        "features":
            features,

        "verification_protocol": {
            "type":
                "posthoc_passive",

            "nli_model":
                NLI_MODEL_ID,

            "retrieve_k":
                RETRIEVE_K,

            "step_retrieval_query":
                "reasoning_step",

            "final_retrieval_query":
                "question",

            "support_threshold":
                SUPPORT_THRESHOLD,

            "contradiction_threshold":
                CONTRADICTION_THRESHOLD,

            "final_prediction_field":
                "majority_answer",

            "loop_final_used":
                False,

            "gold_evidence_used":
                False,

            "verifier_changed_trajectory":
                False,
        },
    }



if os.path.exists(
    PROGRESS_JSON
):

    with open(
        PROGRESS_JSON,
        "r",
        encoding="utf-8",
    ) as f:

        progress = json.load(
            f
        )

else:

    progress = {}


rows_to_run = (
    generated
    if LIMIT is None
    else generated[:LIMIT]
)


print()
print(
    "=" * 80
)

print(
    "POST-HOC HOTPOT VERIFICATION"
)

print(
    "=" * 80
)

print(
    "Target questions:",
    len(
        rows_to_run
    )
)

print(
    "Already completed:",
    len(
        progress
    )
)


for row in tqdm(
    rows_to_run
):

    qi = int(
        row[
            "question_index"
        ]
    )


    key = str(
        qi
    )


    if key in progress:

        continue


    try:

        result = (
            verify_question(
                row
            )
        )


    except Exception:



        with open(
            PROGRESS_JSON,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                progress,
                f,
                indent=2,
                ensure_ascii=False,
            )


        print()
        print(
            "FAILED:"
        )
        print(
            "question_index:",
            qi
        )
        print(
            "question:",
            row[
                "question"
            ]
        )

        raise


    progress[
        key
    ] = result


    if (
        len(progress)
        %
        SAVE_EVERY
        ==
        0
    ):

        with open(
            PROGRESS_JSON,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                progress,
                f,
                indent=2,
                ensure_ascii=False,
            )


        print(
            f"{len(progress)}/"
            f"{len(rows_to_run)} saved"
        )

with open(
    PROGRESS_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        progress,
        f,
        indent=2,
        ensure_ascii=False,
    )




if LIMIT is None:

    assert len(
        progress
    ) == 1000


    verified_rows = [
        progress[
            str(
                int(
                    row[
                        "question_index"
                    ]
                )
            )
        ]
        for row in generated
    ]


    with open(
        OUTPUT_JSON,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            verified_rows,
            f,
            indent=2,
            ensure_ascii=False,
        )



    feature_rows = []


    for r in verified_rows:

        feature_rows.append(
            {
                "question_index":
                    r[
                        "question_index"
                    ],

                "split":
                    r[
                        "split"
                    ],

                "question":
                    r[
                        "question"
                    ],

                "gold_answer":
                    r[
                        "gold_answer"
                    ],

                "majority_answer":
                    r[
                        "majority_answer"
                    ],

                **r[
                    "features"
                ],
            }
        )


    df = pd.DataFrame(
        feature_rows
    )



    REASONING_FEATURES = [
        "max_contradiction_score",
        "any_contradicted",
        "min_support_score",
        "num_steps",
        "frac_supported",
        "frac_contradicted",
        "frac_unclear",
        "mean_support",
        "mean_contradiction",
        "conflict",
        "support_spread",
    ]


    CONFIDENCE_FEATURES = [
        "self_consistency",
        "answer_norm_entropy",
        "answer_logprob",
    ]


    FINAL_VERIFICATION_FEATURES = [
        "fa_support_retr",
        "fa_contradiction_retr",
        "fa_contradicted_retr",
    ]


    FEATURE_COLUMNS = (
        REASONING_FEATURES
        +
        CONFIDENCE_FEATURES
        +
        FINAL_VERIFICATION_FEATURES
    )


    assert (
        len(
            FEATURE_COLUMNS
        )
        ==
        17
    )




    print()
    print(
        "=" * 80
    )

    print(
        "17-FEATURE SANITY CHECK"
    )

    print(
        "=" * 80
    )


    print(
        "Rows:",
        len(
            df
        )
    )


    print(
        "dev:",
        sum(
            df[
                "split"
            ]
            ==
            "dev"
        )
    )


    print(
        "test:",
        sum(
            df[
                "split"
            ]
            ==
            "test"
        )
    )


    assert len(
        df
    ) == 1000


    assert (
        (
            df[
                "split"
            ]
            ==
            "dev"
        )
        .sum()
        ==
        800
    )


    assert (
        (
            df[
                "split"
            ]
            ==
            "test"
        )
        .sum()
        ==
        200
    )



    numeric_features = (
        df[
            FEATURE_COLUMNS
        ]
        .astype(
            float
        )
    )


    bad_columns = []


    for column in FEATURE_COLUMNS:

        values = (
            numeric_features[
                column
            ]
            .to_numpy()
        )


        if not np.all(
            np.isfinite(
                values
            )
        ):

            bad_columns.append(
                column
            )


    if bad_columns:

        raise RuntimeError(
            "Non-finite values found in "
            f"features: {bad_columns}\n"
            "Do NOT impute them yet."
        )


    print(
        "All 17 features finite: YES"
    )



    df.to_csv(
        OUTPUT_CSV,
        index=False,
    )


    print()
    print(
        "Saved verified JSON:"
    )
    print(
        OUTPUT_JSON
    )


    print()
    print(
        "Saved 17-feature CSV:"
    )
    print(
        OUTPUT_CSV
    )



    all_step_reports = [
        step
        for row
        in verified_rows
        for step
        in row[
            "step_reports"
        ]
    ]


    label_counts = Counter(
        step[
            "label"
        ]
        for step
        in all_step_reports
    )


    print()
    print(
        "=" * 80
    )

    print(
        "POST-HOC VERIFICATION SUMMARY"
    )

    print(
        "=" * 80
    )


    print(
        "Total retained reasoning steps:",
        len(
            all_step_reports
        )
    )


    print(
        "Step labels:",
        dict(
            label_counts
        )
    )


    print(
        "Questions with >=1 contradicted step:",
        int(
            df[
                "any_contradicted"
            ]
            .sum()
        )
    )


    print(
        "Questions with zero reasoning steps:",
        int(
            (
                df[
                    "num_steps"
                ]
                ==
                0
            )
            .sum()
        )
    )


    print(
        "Final answers flagged contradicted:",
        int(
            df[
                "fa_contradicted_retr"
            ]
            .sum()
        )
    )


    print(
        "Mean step support:",
        round(
            float(
                df[
                    "mean_support"
                ]
                .mean()
            ),
            4,
        )
    )


    print(
        "Mean step contradiction:",
        round(
            float(
                df[
                    "mean_contradiction"
                ]
                .mean()
            ),
            4,
        )
    )


    print(
        "Mean final-answer support:",
        round(
            float(
                df[
                    "fa_support_retr"
                ]
                .mean()
            ),
            4,
        )
    )


    print(
        "Mean final-answer contradiction:",
        round(
            float(
                df[
                    "fa_contradiction_retr"
                ]
                .mean()
            ),
            4,
        )
    )


    print()
    print(
        "17 feature columns:"
    )

    for i, feature in enumerate(
        FEATURE_COLUMNS,
        start=1,
    ):

        print(
            f"{i:2d}. {feature}"
        )


    print()
    print(
        "DONE."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded frozen Hotpot generations: 1000
dev: 800
test: 200

Loading HotpotQA distractor validation split...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

NLI device: cuda
NLI labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
ENT index: 0
NEU index: 1
CON index: 2

POST-HOC HOTPOT VERIFICATION
Target questions: 1000
Already completed: 0


  0%|          | 0/1000 [00:00<?, ?it/s]

10/1000 saved
20/1000 saved
30/1000 saved
40/1000 saved
50/1000 saved
60/1000 saved
70/1000 saved
80/1000 saved
90/1000 saved
100/1000 saved
110/1000 saved
120/1000 saved
130/1000 saved
140/1000 saved
150/1000 saved
160/1000 saved
170/1000 saved
180/1000 saved
190/1000 saved
200/1000 saved
210/1000 saved
220/1000 saved
230/1000 saved
240/1000 saved
250/1000 saved
260/1000 saved
270/1000 saved
280/1000 saved
290/1000 saved
300/1000 saved
310/1000 saved
320/1000 saved
330/1000 saved
340/1000 saved
350/1000 saved
360/1000 saved
370/1000 saved
380/1000 saved
390/1000 saved
400/1000 saved
410/1000 saved
420/1000 saved
430/1000 saved
440/1000 saved
450/1000 saved
460/1000 saved
470/1000 saved
480/1000 saved
490/1000 saved
500/1000 saved
510/1000 saved
520/1000 saved
530/1000 saved
540/1000 saved
550/1000 saved
560/1000 saved
570/1000 saved
580/1000 saved
590/1000 saved
600/1000 saved
610/1000 saved
620/1000 saved
630/1000 saved
640/1000 saved
650/1000 saved
660/1000 saved
670/1000 saved
680/

In [ ]:


!pip -q install -U openai pandas transformers



import os
import re
import json
import time

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from openai import OpenAI

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)



DATA_DIR = "/content/drive/MyDrive/hedge_run"

INPUT_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_verified_1000.json"
)

CLAIM_CACHE = (
    f"{DATA_DIR}/"
    "hotpot_v2_final_claims_progress.json"
)

PATCH_PROGRESS = (
    f"{DATA_DIR}/"
    "hotpot_v2_final_verification_patch_progress.json"
)

OUTPUT_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_verified_FINAL_1000.json"
)

OUTPUT_CSV = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_features_17_FINAL.csv"
)



CLAIM_MODEL = "gpt-4o-mini"

NLI_MODEL_ID = (
    "MoritzLaurer/"
    "DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
)

NLI_BATCH_SIZE = 10
NLI_MAX_LENGTH = 512

SUPPORT_THRESHOLD = 0.50
CONTRADICTION_THRESHOLD = 0.50

SAVE_EVERY = 10

MAX_API_RETRIES = 4




OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)


if not OPENAI_API_KEY:

    try:

        from google.colab import userdata

        OPENAI_API_KEY = userdata.get(
            "OPENAI_API_KEY"
        )

    except Exception:

        OPENAI_API_KEY = None


if not OPENAI_API_KEY:

    raise RuntimeError(
        "OPENAI_API_KEY not found.\n"
        "Add it to Colab Secrets using the name OPENAI_API_KEY."
    )


client = OpenAI(
    api_key=OPENAI_API_KEY
)



if not os.path.exists(
    INPUT_JSON
):

    raise FileNotFoundError(
        INPUT_JSON
    )


with open(
    INPUT_JSON,
    "r",
    encoding="utf-8",
) as f:

    rows = json.load(
        f
    )


assert len(rows) == 1000


print(
    "Loaded:",
    len(rows)
)

print(
    "dev:",
    sum(
        r["split"] == "dev"
        for r in rows
    )
)

print(
    "test:",
    sum(
        r["split"] == "test"
        for r in rows
    )
)



CLAIM_INSTRUCTIONS = """
You are a deterministic claim-conversion component for a question-answering experiment.

Your task is ONLY to rewrite a question and a proposed answer into one short,
self-contained declarative factual claim.

CRITICAL RULES:

1. Do NOT decide whether the proposed answer is correct.
2. Do NOT correct the proposed answer.
3. Do NOT use outside knowledge.
4. Do NOT add any facts that are absent from the question or proposed answer.
5. Preserve the exact relation asked by the question.
6. Preserve all relevant entities from the question.
7. Insert the proposed answer into the correct answer slot.
8. Output exactly ONE declarative claim.
9. Do not output "The answer is ...".
10. Do not mention the words "question", "proposed answer", or "claim".
11. Do not explain your transformation.
12. Do not use bullet points or quotation marks.

For yes/no questions:
- If the answer is Yes, express the proposition positively.
- If the answer is No, express the proposition negatively.

For comparison questions:
- Express the comparison directly.

Examples:

Question:
Which magazine launched first, Real Simple or Life?
Proposed answer:
Life

Output:
Life was launched before Real Simple.

Question:
Are Willard Mack and Subhash Ghai both involved in film?
Proposed answer:
Yes

Output:
Willard Mack and Subhash Ghai are both involved in film.

Question:
Were Ulrich Walter and Léopold Eyharts both from Germany?
Proposed answer:
No

Output:
Ulrich Walter and Léopold Eyharts were not both from Germany.

Question:
Who was born last, Dave Peverett or Jang Hyun-seung?
Proposed answer:
Jang Hyun-seung

Output:
Jang Hyun-seung was born later than Dave Peverett.

Question:
Where did Cale Gundy's brother play football in college?
Proposed answer:
Oklahoma State University

Output:
Cale Gundy's brother played college football at Oklahoma State University.
""".strip()



def clean_claim(
    text,
):

    text = str(
        text
    ).strip()

    text = re.sub(
        r"^\s*(claim|output)\s*:\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = text.strip(
        ' \n\t"\'`'
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


def convert_to_claim(
    question,
    answer,
):

    question = str(
        question
    ).strip()

    answer = str(
        answer
    ).strip()


    if not answer:

        return ""


    user_input = f"""
Question:
{question}

Proposed answer:
{answer}

Rewrite this as one declarative factual statement.
""".strip()


    last_error = None


    for attempt in range(
        MAX_API_RETRIES
    ):

        try:

            response = (
                client.responses.create(
                    model=CLAIM_MODEL,

                    instructions=CLAIM_INSTRUCTIONS,

                    input=user_input,

                    temperature=0,

                    max_output_tokens=100,

                    # We do not need server-side response storage
                    # for this offline transformation.
                    store=False,
                )
            )


            claim = clean_claim(
                response.output_text
            )


            if claim:

                return claim


            last_error = (
                "empty response"
            )


        except Exception as exc:

            last_error = str(
                exc
            )


        time.sleep(
            2 ** attempt
        )


    raise RuntimeError(
        "Claim conversion failed.\n"
        f"Question: {question}\n"
        f"Answer: {answer}\n"
        f"Error: {last_error}"
    )



if os.path.exists(
    CLAIM_CACHE
):

    with open(
        CLAIM_CACHE,
        "r",
        encoding="utf-8",
    ) as f:

        claim_cache = json.load(
            f
        )

else:

    claim_cache = {}


print(
    "\nExisting cached claims:",
    len(
        claim_cache
    )
)



print()
print(
    "=" * 80
)

print(
    "GENERATING QUESTION-CONDITIONED FINAL CLAIMS"
)

print(
    "=" * 80
)


for row in tqdm(
    rows
):

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    if qi in claim_cache:

        continue


    question = (
        row[
            "question"
        ]
    )



    answer = (
        row[
            "majority_answer"
        ]
    )


    claim = convert_to_claim(
        question,
        answer,
    )


    claim_cache[
        qi
    ] = {
        "question_index":
            int(
                row[
                    "question_index"
                ]
            ),

        "question":
            question,

        "majority_answer":
            answer,

        "claim":
            claim,

        "model":
            CLAIM_MODEL,

        "gold_used":
            False,

        "evidence_used":
            False,
    }


    if (
        len(
            claim_cache
        )
        %
        SAVE_EVERY
        ==
        0
    ):

        with open(
            CLAIM_CACHE,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                claim_cache,
                f,
                indent=2,
                ensure_ascii=False,
            )


        print(
            f"{len(claim_cache)}/1000 claims saved"
        )


# Final claim cache save.

with open(
    CLAIM_CACHE,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        claim_cache,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert len(
    claim_cache
) == 1000


print()
print(
    "All 1000 claims generated."
)




print()
print(
    "=" * 80
)

print(
    "CLAIM SANITY CHECK — FIRST 20"
)

print(
    "=" * 80
)


for row in rows[:20]:

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    print()
    print(
        "QUESTION:"
    )
    print(
        row[
            "question"
        ]
    )


    print(
        "ANSWER:"
    )
    print(
        row[
            "majority_answer"
        ]
    )


    print(
        "CLAIM:"
    )
    print(
        claim_cache[
            qi
        ][
            "claim"
        ]
    )


    print(
        "-" * 80
    )



need_nli_reload = not all(
    name in globals()
    for name in [
        "nli_model",
        "nli_tokenizer",
        "ENT_IDX",
        "NEU_IDX",
        "CON_IDX",
    ]
)


if need_nli_reload:

    print(
        "\nLoading NLI verifier..."
    )


    nli_tokenizer = (
        AutoTokenizer.from_pretrained(
            NLI_MODEL_ID
        )
    )


    if torch.cuda.is_available():

        NLI_DEVICE = torch.device(
            "cuda"
        )

        nli_dtype = (
            torch.float16
        )

    else:

        NLI_DEVICE = torch.device(
            "cpu"
        )

        nli_dtype = (
            torch.float32
        )


    try:

        nli_model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                NLI_MODEL_ID,
                dtype=nli_dtype,
            )
        )

    except TypeError:

        nli_model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                NLI_MODEL_ID,
                torch_dtype=nli_dtype,
            )
        )


    nli_model = (
        nli_model.to(
            NLI_DEVICE
        )
    )

    nli_model.eval()


    id2label = {
        int(k):
            str(v).lower()

        for k, v
        in nli_model.config.id2label.items()
    }


    ENT_IDX = [
        i
        for i, label
        in id2label.items()
        if "entail" in label
    ][0]


    NEU_IDX = [
        i
        for i, label
        in id2label.items()
        if "neutral" in label
    ][0]


    CON_IDX = [
        i
        for i, label
        in id2label.items()
        if "contrad" in label
    ][0]


else:

    print(
        "\nReusing NLI verifier already loaded in memory."
    )


NLI_DEVICE = next(
    nli_model.parameters()
).device


print(
    "NLI device:",
    NLI_DEVICE
)




@torch.no_grad()
def nli_score_pairs_patch(
    premises,
    hypothesis,
):

    if not premises:

        return []


    results = []


    for start in range(
        0,
        len(premises),
        NLI_BATCH_SIZE,
    ):

        batch = premises[
            start:
            start + NLI_BATCH_SIZE
        ]


        hypotheses = [
            hypothesis
        ] * len(
            batch
        )


        encoded = nli_tokenizer(
            batch,
            hypotheses,
            padding=True,
            truncation=True,
            max_length=NLI_MAX_LENGTH,
            return_tensors="pt",
        )


        encoded = {
            k:
                v.to(
                    NLI_DEVICE
                )

            for k, v
            in encoded.items()
        }


        logits = (
            nli_model(
                **encoded
            )
            .logits
            .float()
        )


        probs = (
            torch.softmax(
                logits,
                dim=-1,
            )
            .cpu()
            .numpy()
        )


        for evidence, p in zip(
            batch,
            probs,
        ):

            results.append(
                {
                    "evidence":
                        evidence,

                    "entailment":
                        float(
                            p[
                                ENT_IDX
                            ]
                        ),

                    "neutral":
                        float(
                            p[
                                NEU_IDX
                            ]
                        ),

                    "contradiction":
                        float(
                            p[
                                CON_IDX
                            ]
                        ),
                }
            )


    return results




def aggregate_nli_patch(
    scores,
):

    if not scores:

        return {
            "support":
                0.0,

            "contradiction":
                0.0,

            "neutral":
                1.0,

            "best_support_evidence":
                "",

            "best_contradiction_evidence":
                "",
        }


    support_idx = int(
        np.argmax(
            [
                x[
                    "entailment"
                ]
                for x
                in scores
            ]
        )
    )


    contradiction_idx = int(
        np.argmax(
            [
                x[
                    "contradiction"
                ]
                for x
                in scores
            ]
        )
    )


    neutral_idx = int(
        np.argmax(
            [
                x[
                    "neutral"
                ]
                for x
                in scores
            ]
        )
    )


    return {
        "support":
            float(
                scores[
                    support_idx
                ][
                    "entailment"
                ]
            ),

        "contradiction":
            float(
                scores[
                    contradiction_idx
                ][
                    "contradiction"
                ]
            ),

        "neutral":
            float(
                scores[
                    neutral_idx
                ][
                    "neutral"
                ]
            ),

        "best_support_evidence":
            scores[
                support_idx
            ][
                "evidence"
            ],

        "best_contradiction_evidence":
            scores[
                contradiction_idx
            ][
                "evidence"
            ],
    }




def patch_final_verification(
    row,
):

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    answer = str(
        row[
            "majority_answer"
        ]
    ).strip()


    claim = (
        claim_cache[
            qi
        ][
            "claim"
        ]
    )


    # EXACT SAME question-level evidence that was frozen during generation.

    evidence = [
        str(x).strip()

        for x
        in row[
            "final_answer_verification"
        ][
            "retrieved_evidence"
        ]

        if str(x).strip()
    ]


    if not evidence:

        raise RuntimeError(
            "Missing final-answer evidence for "
            f"question_index={qi}"
        )


    scores = (
        nli_score_pairs_patch(
            evidence,
            claim,
        )
    )


    agg = (
        aggregate_nli_patch(
            scores
        )
    )


    support = float(
        agg[
            "support"
        ]
    )


    contradiction = float(
        agg[
            "contradiction"
        ]
    )


    if (
        support
        >=
        SUPPORT_THRESHOLD
    ):

        label = (
            "supported"
        )

    elif (
        contradiction
        >=
        CONTRADICTION_THRESHOLD
    ):

        label = (
            "contradicted"
        )

    else:

        label = (
            "unclear"
        )


    return {
        "final_answer":
            answer,

        "final_claim":
            claim,

        "retrieved_evidence":
            evidence,

        "support":
            support,

        "contradiction":
            contradiction,

        "neutral":
            float(
                agg[
                    "neutral"
                ]
            ),

        "contradicted":
            int(
                contradiction
                >=
                CONTRADICTION_THRESHOLD
            ),

        "label":
            label,

        "best_support_evidence":
            agg[
                "best_support_evidence"
            ],

        "best_contradiction_evidence":
            agg[
                "best_contradiction_evidence"
            ],

        "claim_converter":
            CLAIM_MODEL,

        "gold_used_for_claim":
            False,

        "evidence_used_for_claim":
            False,
    }


if os.path.exists(
    PATCH_PROGRESS
):

    with open(
        PATCH_PROGRESS,
        "r",
        encoding="utf-8",
    ) as f:

        patch_progress = json.load(
            f
        )

else:

    patch_progress = {}


print()
print(
    "=" * 80
)

print(
    "RECOMPUTING FINAL-ANSWER VERIFICATION"
)

print(
    "=" * 80
)

print(
    "Already completed:",
    len(
        patch_progress
    )
)


for row in tqdm(
    rows
):

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    if qi in patch_progress:

        continue


    result = (
        patch_final_verification(
            row
        )
    )


    patch_progress[
        qi
    ] = result


    if (
        len(
            patch_progress
        )
        %
        SAVE_EVERY
        ==
        0
    ):

        with open(
            PATCH_PROGRESS,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                patch_progress,
                f,
                indent=2,
                ensure_ascii=False,
            )


        print(
            f"{len(patch_progress)}/1000 "
            "final verifications saved"
        )


with open(
    PATCH_PROGRESS,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        patch_progress,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert len(
    patch_progress
) == 1000



corrected_rows = []


for old_row in rows:

    qi = str(
        int(
            old_row[
                "question_index"
            ]
        )
    )


    row = json.loads(
        json.dumps(
            old_row
        )
    )


    new_final = (
        patch_progress[
            qi
        ]
    )




    row[
        "final_answer_verification"
    ] = new_final


    row[
        "features"
    ][
        "fa_support_retr"
    ] = float(
        new_final[
            "support"
        ]
    )


    row[
        "features"
    ][
        "fa_contradiction_retr"
    ] = float(
        new_final[
            "contradiction"
        ]
    )


    row[
        "features"
    ][
        "fa_contradicted_retr"
    ] = int(
        new_final[
            "contradicted"
        ]
    )


    row[
        "verification_protocol"
    ][
        "final_claim_conversion"
    ] = (
        "question_conditioned_llm"
    )


    row[
        "verification_protocol"
    ][
        "final_claim_model"
    ] = (
        CLAIM_MODEL
    )


    corrected_rows.append(
        row
    )


assert len(
    corrected_rows
) == 1000


UNCHANGED_FEATURES = [

    # reasoning 11
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",

    # confidence 3
    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]


for old_row, new_row in zip(
    rows,
    corrected_rows,
):

    for feature in (
        UNCHANGED_FEATURES
    ):

        old_value = float(
            old_row[
                "features"
            ][
                feature
            ]
        )


        new_value = float(
            new_row[
                "features"
            ][
                feature
            ]
        )


        if not np.isclose(
            old_value,
            new_value,
            equal_nan=True,
        ):

            raise RuntimeError(
                "Unexpected feature change: "
                f"{feature}"
            )


print(
    "\nConfirmed: all original 14 "
    "reasoning/confidence features are unchanged."
)



with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        corrected_rows,
        f,
        indent=2,
        ensure_ascii=False,
    )



feature_rows = []


for row in corrected_rows:

    feature_rows.append(
        {
            "question_index":
                int(
                    row[
                        "question_index"
                    ]
                ),

            "split":
                row[
                    "split"
                ],

            "question":
                row[
                    "question"
                ],

            "gold_answer":
                row[
                    "gold_answer"
                ],

            "majority_answer":
                row[
                    "majority_answer"
                ],

            "final_claim":
                row[
                    "final_answer_verification"
                ][
                    "final_claim"
                ],

            **row[
                "features"
            ],
        }
    )


df = pd.DataFrame(
    feature_rows
)


REASONING_FEATURES = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]


CONFIDENCE_FEATURES = [
    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]


FINAL_FEATURES = [
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]


FEATURE_COLUMNS = (
    REASONING_FEATURES
    +
    CONFIDENCE_FEATURES
    +
    FINAL_FEATURES
)


assert len(
    FEATURE_COLUMNS
) == 17


assert len(
    df
) == 1000


assert (
    df[
        "split"
    ]
    .eq(
        "dev"
    )
    .sum()
    ==
    800
)


assert (
    df[
        "split"
    ]
    .eq(
        "test"
    )
    .sum()
    ==
    200
)


X = (
    df[
        FEATURE_COLUMNS
    ]
    .astype(
        float
    )
    .to_numpy()
)


if not np.all(
    np.isfinite(
        X
    )
):

    raise RuntimeError(
        "Non-finite values remain "
        "in the 17 features."
    )


df.to_csv(
    OUTPUT_CSV,
    index=False,
)


print()
print(
    "=" * 80
)

print(
    "CORRECTED FINAL-ANSWER VERIFICATION SUMMARY"
)

print(
    "=" * 80
)


print(
    "Rows:",
    len(
        df
    )
)


print(
    "Final answers supported:",
    int(
        (
            df[
                "fa_support_retr"
            ]
            >=
            SUPPORT_THRESHOLD
        )
        .sum()
    )
)


print(
    "Final answers contradiction >= 0.5:",
    int(
        df[
            "fa_contradicted_retr"
        ]
        .sum()
    )
)


print(
    "Mean final-answer support:",
    round(
        float(
            df[
                "fa_support_retr"
            ]
            .mean()
        ),
        4,
    )
)


print(
    "Mean final-answer contradiction:",
    round(
        float(
            df[
                "fa_contradiction_retr"
            ]
            .mean()
        ),
        4,
    )
)


print()
print(
    "Saved corrected JSON:"
)

print(
    OUTPUT_JSON
)


print()
print(
    "Saved corrected 17-feature CSV:"
)

print(
    OUTPUT_CSV
)


print()
print(
    "DONE."
)

Loaded: 1000
dev: 800
test: 200

Existing cached claims: 0

GENERATING QUESTION-CONDITIONED FINAL CLAIMS


  0%|          | 0/1000 [00:00<?, ?it/s]

10/1000 claims saved
20/1000 claims saved
30/1000 claims saved
40/1000 claims saved
50/1000 claims saved
60/1000 claims saved
70/1000 claims saved
80/1000 claims saved
90/1000 claims saved
100/1000 claims saved
110/1000 claims saved
120/1000 claims saved
130/1000 claims saved
140/1000 claims saved
150/1000 claims saved
160/1000 claims saved
170/1000 claims saved
180/1000 claims saved
190/1000 claims saved
200/1000 claims saved
210/1000 claims saved
220/1000 claims saved
230/1000 claims saved
240/1000 claims saved
250/1000 claims saved
260/1000 claims saved
270/1000 claims saved
280/1000 claims saved
290/1000 claims saved
300/1000 claims saved
310/1000 claims saved
320/1000 claims saved
330/1000 claims saved
340/1000 claims saved
350/1000 claims saved
360/1000 claims saved
370/1000 claims saved
380/1000 claims saved
390/1000 claims saved
400/1000 claims saved
410/1000 claims saved
420/1000 claims saved
430/1000 claims saved
440/1000 claims saved
450/1000 claims saved
460/1000 claims sav

config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

NLI device: cuda:0

RECOMPUTING FINAL-ANSWER VERIFICATION
Already completed: 0


  0%|          | 0/1000 [00:00<?, ?it/s]

10/1000 final verifications saved
20/1000 final verifications saved
30/1000 final verifications saved
40/1000 final verifications saved
50/1000 final verifications saved
60/1000 final verifications saved
70/1000 final verifications saved
80/1000 final verifications saved
90/1000 final verifications saved
100/1000 final verifications saved
110/1000 final verifications saved
120/1000 final verifications saved
130/1000 final verifications saved
140/1000 final verifications saved
150/1000 final verifications saved
160/1000 final verifications saved
170/1000 final verifications saved
180/1000 final verifications saved
190/1000 final verifications saved
200/1000 final verifications saved
210/1000 final verifications saved
220/1000 final verifications saved
230/1000 final verifications saved
240/1000 final verifications saved
250/1000 final verifications saved
260/1000 final verifications saved
270/1000 final verifications saved
280/1000 final verifications saved
290/1000 final verifications 

In [ ]:
import os
import re
import json
import copy
import random

import numpy as np
import torch

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
)

from sentence_transformers import SentenceTransformer



DATA_DIR = "/content/drive/MyDrive/hedge_run"

INPUT_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_1000.json"
)

OUTPUT_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_1000_YN_PATCHED.json"
)

PATCH_DETAILS_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_yn_patch_details.json"
)


MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

EMBED_ID = "BAAI/bge-small-en-v1.5"


FINAL_SAMPLES = 10

FINAL_TEMPERATURE = 0.7

FINAL_TOP_P = 0.95

FINAL_MAX_NEW_TOKENS = 80

ANSWER_MERGE_SIM = 0.80

EMPTY_FINAL_RETRY_ATTEMPTS = 2

SEED = 42


random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


if not os.path.exists(
    INPUT_JSON
):

    raise FileNotFoundError(
        INPUT_JSON
    )


with open(
    INPUT_JSON,
    "r",
    encoding="utf-8",
) as f:

    rows = json.load(f)


assert len(rows) == 1000


print(
    "Loaded:",
    len(rows)
)


print(
    "dev:",
    sum(
        r["split"] == "dev"
        for r in rows
    )
)


print(
    "test:",
    sum(
        r["split"] == "test"
        for r in rows
    )
)



AUX_STARTS = (
    "is ",
    "are ",
    "was ",
    "were ",
    "do ",
    "does ",
    "did ",
    "can ",
    "could ",
    "would ",
    "will ",
    "has ",
    "have ",
    "had ",
)


def old_is_yes_no_question(
    question,
):

    q = (
        str(question)
        .strip()
        .lower()
    )


    return q.startswith(
        AUX_STARTS
    )


def contains_or(
    question,
):

    return bool(
        re.search(
            r"\bor\b",
            str(question).lower(),
        )
    )



GENUINE_YES_NO_WITH_OR = {
    (
        "Was Steve Sekely and Willian King Baggot "
        "known by multiple names or nicknames?"
    )
}


def is_yes_no_question(
    question,
):

    q_original = (
        str(question)
        .strip()
    )


    q = (
        q_original
        .lower()
    )



    if not q.startswith(
        AUX_STARTS
    ):

        return False



    if (
        q_original
        in GENUINE_YES_NO_WITH_OR
    ):

        return True




    if re.search(
        r"\bor\b",
        q,
    ):

        return False


    return True



candidate_rows = [
    row

    for row in rows

    if (
        old_is_yes_no_question(
            row[
                "question"
            ]
        )

        and

        contains_or(
            row[
                "question"
            ]
        )
    )
]


patch_rows = [
    row

    for row in candidate_rows

    if (
        str(
            row[
                "question"
            ]
        ).strip()

        not in

        GENUINE_YES_NO_WITH_OR
    )
]


print()
print(
    "=" * 80
)

print(
    "PATCH SET"
)

print(
    "=" * 80
)


print(
    "Auxiliary + 'or' candidates:",
    len(
        candidate_rows
    )
)


print(
    "Genuine yes/no exclusions:",
    (
        len(candidate_rows)
        -
        len(patch_rows)
    )
)


print(
    "Questions requiring patch:",
    len(
        patch_rows
    )
)


for i, row in enumerate(
    patch_rows,
    start=1,
):

    print()

    print(
        f"{i}. "
        f"{row['question']}"
    )

    print(
        "   old majority:",
        row[
            "majority_answer"
        ]
    )



if len(
    candidate_rows
) != 13:

    raise RuntimeError(
        "Expected exactly 13 auxiliary-initial "
        "questions containing 'or', "
        f"but found {len(candidate_rows)}."
    )


if len(
    patch_rows
) != 12:

    raise RuntimeError(
        "Expected exactly 12 questions requiring "
        f"the patch, but found {len(patch_rows)}."
    )


excluded = [
    row

    for row in candidate_rows

    if (
        str(
            row[
                "question"
            ]
        ).strip()

        in

        GENUINE_YES_NO_WITH_OR
    )
]


if len(
    excluded
) != 1:

    raise RuntimeError(
        "Expected exactly one genuine yes/no "
        "question containing 'or'."
    )


print()
print(
    "Confirmed patch size = 12."
)


print()
print(
    "Excluded genuine yes/no question:"
)


for row in excluded:

    print(
        "-",
        row[
            "question"
        ]
    )


PATCH_IDS = {
    int(
        row[
            "question_index"
        ]
    )

    for row in patch_rows
}




print()
print(
    "Patch question indices:"
)

print(
    sorted(
        PATCH_IDS
    )
)


print()
print(
    "=" * 80
)

print(
    "LOADING QWEN"
)

print(
    "=" * 80
)


if not torch.cuda.is_available():

    raise RuntimeError(
        "GPU runtime required."
    )


compute_dtype = (
    torch.bfloat16

    if torch.cuda.is_bf16_supported()

    else torch.float16
)


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=compute_dtype,

    bnb_4bit_use_double_quant=True,
)


tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
    )
)


if tokenizer.pad_token is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_ID,

        quantization_config=bnb_config,

        device_map="auto",

        trust_remote_code=True,
    )
)


model.eval()


MODEL_DEVICE = (
    next(
        model.parameters()
    )
    .device
)


print(
    "Model device:",
    MODEL_DEVICE
)




print(
    "\nLoading semantic answer embedder..."
)


embedder = SentenceTransformer(
    EMBED_ID,
    device="cpu",
)



SYSTEM_PROMPT = """
You answer multi-hop questions by reasoning ONE unit at a time.

On each turn, output EXACTLY ONE of the following and nothing else:

<step>
one declarative factual statement
</step>

or

<final>
short final answer
</final>

Rules for <step>:
- It must be a declarative factual claim, never a question.
- One fact per step.
- Keep it short and self-contained.
- Build on the reasoning so far.
- Do not repeat previous steps.
- Emit <final> only when the previous steps are enough to answer the question.
""".strip()


def apply_chat_template(
    user_content,
):

    messages = [

        {
            "role":
                "system",

            "content":
                SYSTEM_PROMPT,
        },

        {
            "role":
                "user",

            "content":
                user_content.strip(),
        },
    ]


    return tokenizer.apply_chat_template(
        messages,

        tokenize=False,

        add_generation_prompt=True,
    )



def format_history(
    history_steps,
):

    if not history_steps:

        return "(none yet)"


    return "\n".join(

        f"<step>\n"
        f"{str(step).strip()}\n"
        f"</step>"

        for step in history_steps
    )



def format_evidence(
    evidence,
):

    evidence = [
        str(x).strip()

        for x in evidence

        if str(x).strip()
    ]


    if not evidence:

        return "(none)"


    return "\n".join(

        f"E{i}: {sentence}"

        for i, sentence in enumerate(
            evidence,
            start=1,
        )
    )



def build_final_prompt(
    question,
    evidence,
    history_steps,
):

    yes_no = (
        is_yes_no_question(
            question
        )
    )


    if yes_no:

        answer_rule = (
            "The final answer must be exactly Yes or No."
        )

    else:

        answer_rule = (
            "Answer with the specific entity, person, place, "
            "title, number, category, alternative, comparison "
            "direction, or phrase requested by the question. "
            "Do not answer Yes or No unless the question "
            "genuinely asks for a yes/no judgement."
        )


    user_content = f"""
Evidence:
{format_evidence(evidence)}

Question:
{question}

Reasoning so far:
{format_history(history_steps)}

Give the final answer only.

Use exactly:
<final>
short final answer
</final>

Rules:
- {answer_rule}
- If the question presents alternatives using "or", answer the requested alternative, category, or relation rather than Yes or No.
- If the question asks "before or after", answer the requested relation.
- If the question asks which of two entities satisfies a comparison, answer the entity.
- Do not explain.
- Do not leave the answer blank.
""".strip()


    return apply_chat_template(
        user_content
    )




def clean_text(
    text,
):

    text = str(
        text
    ).strip()


    text = re.sub(
        r"</?(step|final)>",
        "",
        text,
        flags=re.IGNORECASE,
    )


    text = re.sub(
        r"\s+",
        " ",
        text,
    )


    return text.strip()


def parse_unit(
    text,
):

    raw = str(
        text
    )



    match = re.search(
        r"<final>\s*(.*?)\s*</final>",

        raw,

        flags=(
            re.IGNORECASE
            |
            re.DOTALL
        ),
    )


    if match:

        return {
            "type":
                "final",

            "text":
                clean_text(
                    match.group(1)
                ),

            "raw":
                raw,
        }


    match = re.search(
        r"<final>\s*(.*)",

        raw,

        flags=(
            re.IGNORECASE
            |
            re.DOTALL
        ),
    )


    if match:

        content = re.split(
            r"</final>|<step>|</step>",

            match.group(1),

            flags=re.IGNORECASE,
        )[0]


        return {
            "type":
                "final",

            "text":
                clean_text(
                    content
                ),

            "raw":
                raw,
        }



    return {
        "type":
            "final",

        "text":
            clean_text(
                raw
            ),

        "raw":
            raw,
    }



def repair_empty_answer(
    unit,
):

    text = str(
        unit.get(
            "text",
            "",
        )
    ).strip()


    if text:

        return text


    return clean_text(
        unit.get(
            "raw",
            "",
        )
    )


class StopOnStrings(
    StoppingCriteria
):

    def __init__(
        self,
        tokenizer,
        start_length,
        stop_strings,
    ):

        super().__init__()

        self.tokenizer = tokenizer

        self.start_length = (
            start_length
        )

        self.stop_strings = (
            stop_strings
        )


    def __call__(
        self,
        input_ids,
        scores,
        **kwargs,
    ):

        # Stop only when EVERY generated sequence has reached a stopping string

        for row in input_ids:

            generated = (
                self.tokenizer.decode(
                    row[
                        self.start_length:
                    ],
                    skip_special_tokens=True,
                )
            )


            if not any(
                stop in generated

                for stop
                in self.stop_strings
            ):

                return False


        return True


@torch.no_grad()
def generate_units(
    prompt,
    n_samples,
):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    )


    inputs = {
        key:
            value.to(
                MODEL_DEVICE
            )

        for key, value
        in inputs.items()
    }


    start_length = (
        inputs[
            "input_ids"
        ]
        .shape[-1]
    )


    stopping = (
        StoppingCriteriaList(
            [
                StopOnStrings(
                    tokenizer,
                    start_length,
                    [
                        "</final>",
                    ],
                )
            ]
        )
    )


    outputs = model.generate(

        **inputs,

        do_sample=True,

        temperature=(
            FINAL_TEMPERATURE
        ),

        top_p=(
            FINAL_TOP_P
        ),

        max_new_tokens=(
            FINAL_MAX_NEW_TOKENS
        ),

        num_return_sequences=(
            n_samples
        ),

        stopping_criteria=(
            stopping
        ),

        pad_token_id=(
            tokenizer.eos_token_id
        ),
    )


    decoded = (
        tokenizer.batch_decode(
            outputs[
                :,
                start_length:
            ],

            skip_special_tokens=True,
        )
    )


    del inputs
    del outputs


    torch.cuda.empty_cache()


    return [
        parse_unit(
            text
        )

        for text in decoded
    ]




def generate_final_samples(
    question,
    evidence,
    history_steps,
):

    prompt = build_final_prompt(
        question,
        evidence,
        history_steps,
    )


    answers = []


    for attempt in range(
        EMPTY_FINAL_RETRY_ATTEMPTS
        +
        1
    ):

        remaining = (
            FINAL_SAMPLES
            -
            len(answers)
        )


        if remaining <= 0:

            break


        units = generate_units(
            prompt,
            remaining,
        )


        for unit in units:

            answer = (
                repair_empty_answer(
                    unit
                )
                .strip()
            )


            if answer:

                answers.append(
                    answer
                )


    return answers[
        :FINAL_SAMPLES
    ]



def normalize_answer(
    text,
):

    text = (
        str(text)
        .lower()
        .strip()
    )


    text = re.sub(
        r"\b(the|a|an)\b",
        " ",
        text,
    )


    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text,
    )


    text = re.sub(
        r"\s+",
        " ",
        text,
    )


    return text.strip()


def cluster_answers(
    answers,
):

    clean = [
        str(answer).strip()

        for answer in answers

        if str(answer).strip()
    ]


    if not clean:

        return []


    norms = [
        normalize_answer(
            answer
        )

        for answer in clean
    ]


    embeddings = (
        embedder.encode(
            clean,

            normalize_embeddings=True,

            convert_to_numpy=True,

            show_progress_bar=False,
        )
    )


    clusters = []


    for i, norm_i in enumerate(
        norms
    ):

        placed = False


        for cluster in clusters:

            norm_j = (
                cluster[
                    "norm"
                ]
            )




            same = (
                norm_i
                ==
                norm_j
            )



            if (
                not same
                and norm_i
                and norm_j
            ):

                same = (
                    norm_i in norm_j

                    or

                    norm_j in norm_i
                )




            if not same:

                similarity = float(
                    np.dot(
                        embeddings[
                            i
                        ],

                        embeddings[
                            cluster[
                                "idx0"
                            ]
                        ],
                    )
                )


                same = (
                    similarity
                    >=
                    ANSWER_MERGE_SIM
                )


            if same:

                cluster[
                    "idxs"
                ].append(
                    i
                )


                placed = True

                break


        if not placed:

            clusters.append(
                {
                    "norm":
                        norm_i,

                    "idx0":
                        i,

                    "idxs":
                        [
                            i
                        ],
                }
            )


    return clusters



def shannon_entropy(
    counts,
):

    counts = [
        int(count)

        for count in counts

        if count > 0
    ]


    if len(
        counts
    ) <= 1:

        return (
            0.0,
            0.0,
        )


    total = float(
        sum(
            counts
        )
    )


    probabilities = (
        np.asarray(
            counts,
            dtype=float,
        )
        /
        total
    )


    entropy = float(
        -np.sum(
            probabilities
            *
            np.log2(
                probabilities
                +
                1e-12
            )
        )
    )


    norm_entropy = float(
        entropy
        /
        np.log2(
            len(
                counts
            )
        )
    )


    return (
        entropy,
        norm_entropy,
    )



def self_consistency_stats(
    answers,
):

    answers = [
        str(answer).strip()

        for answer in answers

        if str(answer).strip()
    ]


    if not answers:

        return {
            "majority_answer":
                "",

            "self_consistency":
                0.0,

            "norm_entropy":
                1.0,

            "num_clusters":
                0,

            "n":
                0,
        }


    clusters = cluster_answers(
        answers
    )


    sizes = [
        len(
            cluster[
                "idxs"
            ]
        )

        for cluster in clusters
    ]


    biggest = max(
        clusters,

        key=lambda cluster:
            len(
                cluster[
                    "idxs"
                ]
            ),
    )


    majority_answer = (
        answers[
            biggest[
                "idxs"
            ][0]
        ]
    )


    _, norm_entropy = (
        shannon_entropy(
            sizes
        )
    )


    return {
        "majority_answer":
            majority_answer,

        "self_consistency":
            float(
                max(
                    sizes
                )
                /
                len(
                    answers
                )
            ),

        "norm_entropy":
            float(
                norm_entropy
            ),

        "num_clusters":
            int(
                len(
                    clusters
                )
            ),

        "n":
            int(
                len(
                    answers
                )
            ),
    }



@torch.no_grad()
def answer_logprob(
    question,
    evidence,
    history_steps,
    answer,
):

    answer = str(
        answer
    ).strip()


    if not answer:

        return float(
            "nan"
        )


    prompt = build_final_prompt(
        question,
        evidence,
        history_steps,
    )


    prefix_text = (
        prompt
        +
        "<final>\n"
    )




    prefix_ids = tokenizer(
        prefix_text,
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    answer_ids = tokenizer(
        answer,
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    if not answer_ids:

        return float(
            "nan"
        )


    all_ids = (
        prefix_ids
        +
        answer_ids
    )


    input_ids = torch.tensor(
        [
            all_ids
        ],

        dtype=torch.long,

        device=MODEL_DEVICE,
    )


    attention_mask = (
        torch.ones_like(
            input_ids
        )
    )


    outputs = model(
        input_ids=input_ids,

        attention_mask=attention_mask,
    )


    logits = (
        outputs
        .logits[
            0
        ]
        .float()
    )



    start = (
        len(
            prefix_ids
        )
        -
        1
    )


    end = (
        start
        +
        len(
            answer_ids
        )
    )


    answer_logits = (
        logits[
            start:end
        ]
    )


    targets = torch.tensor(
        answer_ids,

        dtype=torch.long,

        device=MODEL_DEVICE,
    )


    log_probs = (
        torch.log_softmax(
            answer_logits,
            dim=-1,
        )
    )


    token_log_probs = (
        log_probs[
            torch.arange(
                len(
                    answer_ids
                ),

                device=MODEL_DEVICE,
            ),

            targets,
        ]
    )


    mean_logprob = float(
        token_log_probs
        .mean()
        .item()
    )


    del input_ids
    del attention_mask
    del outputs
    del logits


    return mean_logprob



patched_rows = copy.deepcopy(
    rows
)


patch_details = {}


print()
print(
    "=" * 80
)

print(
    "REGENERATING FINAL ANSWERS FOR 12 QUESTIONS"
)

print(
    "=" * 80
)



for row in tqdm(
    patched_rows
):

    qi = int(
        row[
            "question_index"
        ]
    )


    if qi not in PATCH_IDS:

        continue




    question_seed = (
        SEED
        +
        qi
    )


    random.seed(
        question_seed
    )


    np.random.seed(
        question_seed
    )


    torch.manual_seed(
        question_seed
    )


    torch.cuda.manual_seed_all(
        question_seed
    )


    question = str(
        row[
            "question"
        ]
    )




    evidence = copy.deepcopy(
        row.get(
            "question_evidence",
            [],
        )
    )



    history = copy.deepcopy(
        row.get(
            "selected_steps",
            [],
        )
    )




    old_values = {

        "final_answers":
            copy.deepcopy(
                row.get(
                    "final_answers",
                    [],
                )
            ),

        "majority_answer":
            row.get(
                "majority_answer",
                "",
            ),

        "self_consistency":
            row.get(
                "self_consistency"
            ),

        "answer_norm_entropy":
            row.get(
                "answer_norm_entropy"
            ),

        "answer_logprob":
            row.get(
                "answer_logprob"
            ),
    }




    final_answers = (
        generate_final_samples(
            question,
            evidence,
            history,
        )
    )


    if len(
        final_answers
    ) != FINAL_SAMPLES:

        raise RuntimeError(
            f"question_index={qi}: "
            f"expected {FINAL_SAMPLES} final samples, "
            f"but got {len(final_answers)}."
        )



    stats = (
        self_consistency_stats(
            final_answers
        )
    )


    majority_answer = (
        stats[
            "majority_answer"
        ]
    )


    if not majority_answer:

        raise RuntimeError(
            f"Empty majority answer for question_index={qi}"
        )



    logprob = (
        answer_logprob(
            question,
            evidence,
            history,
            majority_answer,
        )
    )


    if not np.isfinite(
        logprob
    ):

        raise RuntimeError(
            f"Invalid answer_logprob "
            f"for question_index={qi}"
        )




    row[
        "final_answers"
    ] = final_answers


    row[
        "majority_answer"
    ] = majority_answer


    row[
        "self_consistency"
    ] = float(
        stats[
            "self_consistency"
        ]
    )


    row[
        "answer_norm_entropy"
    ] = float(
        stats[
            "norm_entropy"
        ]
    )


    row[
        "answer_logprob"
    ] = float(
        logprob
    )




    row[
        "final_answer_question_type_patch"
    ] = {

        "applied":
            True,

        "reason":
            "auxiliary_initial_alternative_question",

        "old_question_type":
            "yes_no",

        "corrected_question_type":
            "alternative_or_choice",

        "reasoning_regenerated":
            False,

        "evidence_changed":
            False,

        "loop_final_changed":
            False,

        "seed":
            int(
                question_seed
            ),
    }



    patch_details[
        str(
            qi
        )
    ] = {

        "question_index":
            qi,

        "question":
            question,

        "gold_answer_diagnostic_only":
            row.get(
                "gold_answer",
                "",
            ),

        "old":
            old_values,

        "new": {

            "final_answers":
                copy.deepcopy(
                    final_answers
                ),

            "majority_answer":
                majority_answer,

            "self_consistency":
                float(
                    stats[
                        "self_consistency"
                    ]
                ),

            "answer_norm_entropy":
                float(
                    stats[
                        "norm_entropy"
                    ]
                ),

            "answer_logprob":
                float(
                    logprob
                ),
        },
    }




actually_patched = [
    row

    for row in patched_rows

    if (
        int(
            row[
                "question_index"
            ]
        )
        in PATCH_IDS
    )
]


assert len(
    actually_patched
) == 12


assert len(
    patch_details
) == 12



for old_row, new_row in zip(
    rows,
    patched_rows,
):

    old_qi = int(
        old_row[
            "question_index"
        ]
    )


    new_qi = int(
        new_row[
            "question_index"
        ]
    )


    assert (
        old_qi
        ==
        new_qi
    )


    if old_qi not in PATCH_IDS:

        if (
            old_row
            !=
            new_row
        ):

            raise RuntimeError(
                "An unaffected record changed!\n"
                f"question_index={old_qi}"
            )


print()
print(
    "Confirmed: all 988 unaffected records "
    "are unchanged."
)



for old_row, new_row in zip(
    rows,
    patched_rows,
):

    qi = int(
        old_row[
            "question_index"
        ]
    )


    if qi not in PATCH_IDS:

        continue


    if (
        old_row.get(
            "selected_steps",
            []
        )
        !=
        new_row.get(
            "selected_steps",
            []
        )
    ):

        raise RuntimeError(
            "Reasoning changed unexpectedly for "
            f"question_index={qi}"
        )


    if (
        old_row.get(
            "question_evidence",
            []
        )
        !=
        new_row.get(
            "question_evidence",
            []
        )
    ):

        raise RuntimeError(
            "Evidence changed unexpectedly for "
            f"question_index={qi}"
        )


    if (
        old_row.get(
            "loop_final_answer",
            ""
        )
        !=
        new_row.get(
            "loop_final_answer",
            ""
        )
    ):

        raise RuntimeError(
            "loop_final_answer changed unexpectedly for "
            f"question_index={qi}"
        )


print(
    "Confirmed: reasoning/evidence/loop final "
    "remain frozen for all 12 patched questions."
)



def simple_normalize(
    answer,
):

    return (
        str(answer)
        .strip()
        .lower()
        .rstrip(".")
    )


still_yes_no = []


for row in patched_rows:

    qi = int(
        row[
            "question_index"
        ]
    )


    if qi not in PATCH_IDS:

        continue


    answer = simple_normalize(
        row[
            "majority_answer"
        ]
    )


    if answer in {
        "yes",
        "no",
    }:

        still_yes_no.append(
            {
                "question_index":
                    qi,

                "question":
                    row[
                        "question"
                    ],

                "majority_answer":
                    row[
                        "majority_answer"
                    ],
            }
        )


print()
print(
    "Patched majority answers still exactly Yes/No:",
    len(
        still_yes_no
    )
)


if still_yes_no:

    for item in still_yes_no:

        print()

        print(
            item[
                "question"
            ]
        )

        print(
            "->",
            item[
                "majority_answer"
            ]
        )




for old_row, new_row in zip(
    rows,
    patched_rows,
):

    question = str(
        old_row[
            "question"
        ]
    ).strip()


    if (
        question
        in
        GENUINE_YES_NO_WITH_OR
    ):

        assert (
            old_row
            ==
            new_row
        )


        print()
        print(
            "Confirmed excluded genuine yes/no "
            "question was untouched:"
        )

        print(
            question
        )

        print(
            "Majority:",
            old_row[
                "majority_answer"
            ]
        )



with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        patched_rows,
        f,
        indent=2,
        ensure_ascii=False,
    )


with open(
    PATCH_DETAILS_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        patch_details,
        f,
        indent=2,
        ensure_ascii=False,
    )



print()
print(
    "=" * 80
)

print(
    "PATCH RESULTS"
)

print(
    "=" * 80
)


ordered_patch_details = sorted(
    patch_details.values(),

    key=lambda x:
        x[
            "question_index"
        ],
)


for item in ordered_patch_details:

    print()

    print(
        "QUESTION:"
    )

    print(
        item[
            "question"
        ]
    )


    print(
        "GOLD [DIAGNOSTIC ONLY]:"
    )

    print(
        item[
            "gold_answer_diagnostic_only"
        ]
    )


    print(
        "OLD MAJORITY:"
    )

    print(
        item[
            "old"
        ][
            "majority_answer"
        ]
    )


    print(
        "NEW MAJORITY:"
    )

    print(
        item[
            "new"
        ][
            "majority_answer"
        ]
    )


    print(
        "NEW FINAL SAMPLES:"
    )

    print(
        item[
            "new"
        ][
            "final_answers"
        ]
    )


    print(
        "OLD SELF-CONSISTENCY:",
        item[
            "old"
        ][
            "self_consistency"
        ]
    )


    print(
        "NEW SELF-CONSISTENCY:",
        item[
            "new"
        ][
            "self_consistency"
        ]
    )


    print(
        "OLD NORMALISED ENTROPY:",
        item[
            "old"
        ][
            "answer_norm_entropy"
        ]
    )


    print(
        "NEW NORMALISED ENTROPY:",
        item[
            "new"
        ][
            "answer_norm_entropy"
        ]
    )


    print(
        "OLD ANSWER LOGPROB:",
        item[
            "old"
        ][
            "answer_logprob"
        ]
    )


    print(
        "NEW ANSWER LOGPROB:",
        item[
            "new"
        ][
            "answer_logprob"
        ]
    )


    print(
        "-" * 80
    )



print()
print(
    "=" * 80
)

print(
    "PATCH SUMMARY"
)

print(
    "=" * 80
)


print(
    "Total records:",
    len(
        patched_rows
    )
)


print(
    "Patched:",
    len(
        PATCH_IDS
    )
)


print(
    "Untouched:",
    (
        len(
            patched_rows
        )
        -
        len(
            PATCH_IDS
        )
    )
)


print(
    "Patched majority answers still Yes/No:",
    len(
        still_yes_no
    )
)


print()
print(
    "Saved patched generation:"
)

print(
    OUTPUT_JSON
)


print()
print(
    "Saved patch details:"
)

print(
    PATCH_DETAILS_JSON
)


print()
print(
    "DONE."
)

Loaded: 1000
dev: 800
test: 200

PATCH SET
Auxiliary + 'or' candidates: 13
Genuine yes/no exclusions: 1
Questions requiring patch: 12

1. Did Billy Corgan's band The Smashing Pumpkins or Greek Fire originate in a more southern location?
   old majority: No

2. Are Philip Cortez and Julian Castro democratic or republican?
   old majority: Yes

3. Was the famous opera comique that was based on a Prosper Merimee novella a comedy or a tragedy? 
   old majority: No

4. Was Atom Egoyans biggest commercial success on stage or on film?
   old majority: Film

5. Did Big Pig or Blur have more members?
   old majority: No

6. Did Vertical Horizon or LCD Soundsystem start their bands first?
   old majority: No

7. Was California State University, Dominguez Hills founded before or after Pacific Lutheran University?
   old majority: No

8. Is Ashland, New Hampshire or Plymouth Regional High School located near the Scribner-Fellows State Forest?
   old majority: Yes

9. Was Vanderbilt University or E

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


REGENERATING FINAL ANSWERS FOR 12 QUESTIONS


  0%|          | 0/1000 [00:00<?, ?it/s]


Confirmed: all 988 unaffected records are unchanged.
Confirmed: reasoning/evidence/loop final remain frozen for all 12 patched questions.

Patched majority answers still exactly Yes/No: 0

Confirmed excluded genuine yes/no question was untouched:
Was Steve Sekely and Willian King Baggot known by multiple names or nicknames?
Majority: Yes

PATCH RESULTS

QUESTION:
Did Billy Corgan's band The Smashing Pumpkins or Greek Fire originate in a more southern location?
GOLD [DIAGNOSTIC ONLY]:
Greek Fire
OLD MAJORITY:
No
NEW MAJORITY:
Greek Fire
NEW FINAL SAMPLES:
['Greek Fire', 'Greek Fire', 'Greek Fire', 'Greek Fire', 'Greek Fire', 'Greek Fire', 'Greek Fire', 'Greek Fire', 'Greek Fire', 'Greek Fire']
OLD SELF-CONSISTENCY: 0.9
NEW SELF-CONSISTENCY: 1.0
OLD NORMALISED ENTROPY: 0.4689955935863958
NEW NORMALISED ENTROPY: 0.0
OLD ANSWER LOGPROB: -0.2326415330171585
NEW ANSWER LOGPROB: -8.81477608345449e-05
--------------------------------------------------------------------------------

QUESTION:


In [ ]:
import os
import re
import json
import time
import copy
import random

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

from openai import OpenAI

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)


DATA_DIR = "/content/drive/MyDrive/hedge_run"



PATCHED_GENERATION_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_1000_YN_PATCHED.json"
)


REASONING_VERIFIED_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_verified_1000.json"
)



TEMPLATE_CACHE_PATH = (
    f"{DATA_DIR}/"
    "hotpot_v2_question_claim_templates_STRICT_V1.json"
)


FINAL_CLAIMS_PATH = (
    f"{DATA_DIR}/"
    "hotpot_v2_final_claims_STRICT_V1.json"
)




FINAL_NLI_PROGRESS = (
    f"{DATA_DIR}/"
    "hotpot_v2_final_nli_STRICT_V1_progress.json"
)




OUTPUT_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_1000.json"
)


OUTPUT_CSV = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_features17.csv"
)



RUN_NLI = False


CLAIM_MODEL = "gpt-4o-mini"


NLI_MODEL_ID = (
    "MoritzLaurer/"
    "DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
)


SUPPORT_THRESHOLD = 0.50

CONTRADICTION_THRESHOLD = 0.50


NLI_MAX_LENGTH = 512

NLI_BATCH_SIZE = 16


SAVE_EVERY = 10

MAX_API_RETRIES = 5

SEED = 42


random.seed(SEED)

np.random.seed(SEED)




for path in [
    PATCHED_GENERATION_JSON,
    REASONING_VERIFIED_JSON,
]:

    if not os.path.exists(path):

        raise FileNotFoundError(path)


with open(
    PATCHED_GENERATION_JSON,
    "r",
    encoding="utf-8",
) as f:

    generated = json.load(f)


with open(
    REASONING_VERIFIED_JSON,
    "r",
    encoding="utf-8",
) as f:

    verified_old = json.load(f)


assert len(generated) == 1000

assert len(verified_old) == 1000


print(
    "Patched generation:",
    len(generated)
)


print(
    "Reasoning verification rows:",
    len(verified_old)
)


print(
    "dev:",
    sum(
        r["split"] == "dev"
        for r in generated
    )
)


print(
    "test:",
    sum(
        r["split"] == "test"
        for r in generated
    )
)


assert (
    sum(
        r["split"] == "dev"
        for r in generated
    )
    ==
    800
)


assert (
    sum(
        r["split"] == "test"
        for r in generated
    )
    ==
    200
)




verified_by_qi = {
    int(r["question_index"]):
        r

    for r in verified_old
}


assert len(
    verified_by_qi
) == 1000




for row in generated:

    qi = int(
        row[
            "question_index"
        ]
    )


    old = (
        verified_by_qi[
            qi
        ]
    )


    if (
        row[
            "question"
        ]
        !=
        old[
            "question"
        ]
    ):

        raise RuntimeError(
            f"Question mismatch qi={qi}"
        )


    if (
        row.get(
            "selected_steps",
            []
        )
        !=
        old.get(
            "selected_steps",
            []
        )
    ):

        raise RuntimeError(
            f"Reasoning changed for qi={qi}"
        )


print(
    "Confirmed: all reasoning trajectories "
    "match the previously verified trajectories."
)




REASONING_FEATURES = [

    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]


CONFIDENCE_FEATURES = [

    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]


FINAL_FEATURES = [
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]


FEATURE_COLUMNS = (

    REASONING_FEATURES
    +
    CONFIDENCE_FEATURES
    +
    FINAL_FEATURES
)


assert len(
    FEATURE_COLUMNS
) == 17



for row in verified_old:

    features = (
        row.get(
            "features",
            {}
        )
    )


    for feature in (
        REASONING_FEATURES
    ):

        if feature not in features:

            raise RuntimeError(
                f"Missing reasoning feature "
                f"{feature} in qi="
                f"{row['question_index']}"
            )


print(
    "All 11 frozen reasoning features found."
)




AUX_STARTS = (

    "is ",
    "are ",
    "was ",
    "were ",

    "do ",
    "does ",
    "did ",

    "can ",
    "could ",
    "would ",
    "will ",

    "has ",
    "have ",
    "had ",
)


GENUINE_YES_NO_WITH_OR = {

    (
        "Was Steve Sekely and Willian King Baggot "
        "known by multiple names or nicknames?"
    )
}


def is_yes_no_question(
    question,
):

    original = (
        str(question)
        .strip()
    )


    q = (
        original
        .lower()
    )


    if not q.startswith(
        AUX_STARTS
    ):

        return False


    if (
        original
        in
        GENUINE_YES_NO_WITH_OR
    ):

        return True


    if re.search(
        r"\bor\b",
        q,
    ):

        return False


    return True



n_yes_no = sum(
    is_yes_no_question(
        row[
            "question"
        ]
    )

    for row in generated
)


print(
    "Questions treated as genuine yes/no:",
    n_yes_no
)



PATCH_IDS = {

    int(
        row[
            "question_index"
        ]
    )

    for row in generated

    if (
        row.get(
            "final_answer_question_type_patch",
            {}
        )
        .get(
            "applied",
            False,
        )
    )
}


print(
    "Detected patched alternative questions:",
    len(
        PATCH_IDS
    )
)


assert len(
    PATCH_IDS
) == 12



OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)


if not OPENAI_API_KEY:

    try:

        from google.colab import userdata

        OPENAI_API_KEY = userdata.get(
            "OPENAI_API_KEY"
        )

    except Exception:

        OPENAI_API_KEY = None


if not OPENAI_API_KEY:

    raise RuntimeError(
        "OPENAI_API_KEY not found. "
        "Add it to Colab Secrets."
    )


client = OpenAI(
    api_key=OPENAI_API_KEY
)



NON_YN_INSTRUCTIONS = r"""
You are a syntax-only question-to-claim-template converter used in a scientific
question-answering experiment.

You will receive ONLY an original question.

You must NOT answer the question.

Instead, rewrite the question as ONE short declarative factual claim template
containing the literal token:

[[ANSWER]]

exactly once.

[[ANSWER]] represents an unknown proposed answer that will be inserted later
by computer code.

CRITICAL RULES:

1. Do NOT answer the question.
2. Do NOT guess the answer.
3. Do NOT use outside knowledge.
4. Do NOT add facts that are not expressed in the question.
5. Preserve the EXACT relation requested by the question.
6. [[ANSWER]] must occupy the exact semantic answer slot requested.
7. Use [[ANSWER]] exactly once.
8. Do not replace [[ANSWER]] with an entity.
9. Do not say "the answer is".
10. Do not mention the word "question".
11. Produce a declarative claim, not a question.
12. Preserve comparison direction exactly.
13. Preserve all entities needed to express the requested relation.
14. For "who" questions, [[ANSWER]] must occupy the requested person/entity slot.
15. For "where" questions, [[ANSWER]] must occupy the requested location slot.
16. For "when" or "what year" questions, [[ANSWER]] must occupy the time/year slot.
17. For "by who" questions, [[ANSWER]] must occupy the agent/source slot after "by".
18. For "what show/book/song/etc." questions, [[ANSWER]] must occupy that object slot,
    not the person associated with it.
19. For an A-or-B comparison asking WHICH entity satisfies a relation, keep the
    comparison explicit and place [[ANSWER]] in the selected-entity slot.
20. For "before or after" questions, [[ANSWER]] occupies the comparison-relation slot.
21. Do not determine whether any possible answer would be factually correct.

Examples:

Question:
In what year was the university where Sergei Aleksandrovich Tokarev was a professor founded?

Template:
The university where Sergei Aleksandrovich Tokarev was a professor was founded in [[ANSWER]].


Question:
Between Greyia and Calibanus, which genus contains more species?

Template:
Between Greyia and Calibanus, [[ANSWER]] contains more species.


Question:
Who released the song "With or Without You" first, Jai McDowall or U2?

Template:
Between Jai McDowall and U2, [[ANSWER]] released the song "With or Without You" first.


Question:
Was Vanderbilt University or Emory University founded first?

Template:
Between Vanderbilt University and Emory University, [[ANSWER]] was founded first.


Question:
Was California State University, Dominguez Hills founded before or after Pacific Lutheran University?

Template:
California State University, Dominguez Hills was founded [[ANSWER]] Pacific Lutheran University.


Question:
Isabella Kelly was born at a ruined castle characterized as one of the most isolated fortifications in Britain by who?

Template:
The ruined castle where Isabella Kelly was born was characterized as one of the most isolated fortifications in Britain by [[ANSWER]].


Question:
For One Night Only was hosted by the man most well-known for hosting what show from 1962 until 1999?

Template:
The man who hosted For One Night Only was most well-known for hosting [[ANSWER]] from 1962 until 1999.
""".strip()



YN_INSTRUCTIONS = r"""
You are a syntax-only yes/no-question converter used in a scientific
question-answering experiment.

You will receive ONLY a yes/no question.

Do NOT answer it.

Return two short declarative factual claims:

- yes_claim: the proposition expressed if the answer to the question is Yes.
- no_claim: the proposition expressed if the answer to the question is No.

CRITICAL RULES:

1. Do NOT decide which claim is true.
2. Do NOT answer the question.
3. Do NOT use outside knowledge.
4. Use only information already expressed in the question.
5. Preserve exactly the relation asked.
6. The two claims must express opposite yes/no outcomes.
7. Preserve "both", comparisons, negation, dates, quantities, and entities.
8. Do not mention "the answer", "question", Yes, or No inside the claims.
9. Produce declarative claims, not questions.

Example:

Question:
Did John Updike and Tom Clancy both publish more than 15 bestselling novels?

yes_claim:
John Updike and Tom Clancy both published more than 15 bestselling novels.

no_claim:
John Updike and Tom Clancy did not both publish more than 15 bestselling novels.


Question:
Were the board games Clans and Drunter und Drüber both created by Leo Colovini?

yes_claim:
Clans and Drunter und Drüber were both created by Leo Colovini.

no_claim:
Clans and Drunter und Drüber were not both created by Leo Colovini.
""".strip()



NON_YN_SCHEMA = {

    "type":
        "object",

    "properties": {

        "template": {
            "type":
                "string"
        },
    },

    "required": [
        "template",
    ],

    "additionalProperties":
        False,
}


YN_SCHEMA = {

    "type":
        "object",

    "properties": {

        "yes_claim": {
            "type":
                "string"
        },

        "no_claim": {
            "type":
                "string"
        },
    },

    "required": [
        "yes_claim",
        "no_claim",
    ],

    "additionalProperties":
        False,
}



def clean_output_text(
    text,
):

    text = str(
        text
    ).strip()


    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


    return text



def validate_non_yn_template(
    template,
):

    template = (
        clean_output_text(
            template
        )
    )


    if (
        template.count(
            "[[ANSWER]]"
        )
        !=
        1
    ):

        raise ValueError(
            "Template must contain "
            "[[ANSWER]] exactly once."
        )


    if (
        len(template)
        <
        10
    ):

        raise ValueError(
            "Template too short."
        )


    low = (
        template
        .lower()
    )


    forbidden = [

        "the answer is",

        "answer to the question",

        "proposed answer",
    ]


    if any(
        phrase in low

        for phrase
        in forbidden
    ):

        raise ValueError(
            "Meta-answer wording found."
        )


    # It must be a declarative statement.
    if template.rstrip().endswith("?"):

        raise ValueError(
            "Template is still a question."
        )


    return template



def validate_yn_claims(
    yes_claim,
    no_claim,
):

    yes_claim = (
        clean_output_text(
            yes_claim
        )
    )


    no_claim = (
        clean_output_text(
            no_claim
        )
    )


    if (
        len(yes_claim)
        <
        5
        or
        len(no_claim)
        <
        5
    ):

        raise ValueError(
            "Yes/no claim too short."
        )


    if (
        yes_claim
        ==
        no_claim
    ):

        raise ValueError(
            "yes_claim and no_claim "
            "are identical."
        )


    if (
        yes_claim.endswith("?")
        or
        no_claim.endswith("?")
    ):

        raise ValueError(
            "Yes/no output is not declarative."
        )


    return (
        yes_claim,
        no_claim,
    )



def generate_non_yn_template(
    question,
):

    last_error = None


    for attempt in range(
        MAX_API_RETRIES
    ):

        try:

            response = (
                client.responses.create(

                    model=
                        CLAIM_MODEL,

                    instructions=
                        NON_YN_INSTRUCTIONS,

                    input=(
                        "QUESTION:\n"
                        +
                        str(
                            question
                        ).strip()
                    ),

                    temperature=
                        0,

                    max_output_tokens=
                        160,

                    store=
                        False,

                    text={
                        "format": {

                            "type":
                                "json_schema",

                            "name":
                                "question_claim_template",

                            "strict":
                                True,

                            "schema":
                                NON_YN_SCHEMA,
                        }
                    },
                )
            )


            obj = json.loads(
                response.output_text
            )


            template = (
                validate_non_yn_template(
                    obj[
                        "template"
                    ]
                )
            )


            return template


        except Exception as exc:

            last_error = exc


            if (
                attempt
                <
                MAX_API_RETRIES - 1
            ):

                time.sleep(
                    2 ** attempt
                )


    raise RuntimeError(
        "Failed to generate claim template.\n"
        f"Question: {question}\n"
        f"Last error: {last_error}"
    )


def generate_yn_claim_pair(
    question,
):

    last_error = None


    for attempt in range(
        MAX_API_RETRIES
    ):

        try:

            response = (
                client.responses.create(

                    model=
                        CLAIM_MODEL,

                    instructions=
                        YN_INSTRUCTIONS,

                    input=(
                        "QUESTION:\n"
                        +
                        str(
                            question
                        ).strip()
                    ),

                    temperature=
                        0,

                    max_output_tokens=
                        180,

                    store=
                        False,

                    text={
                        "format": {

                            "type":
                                "json_schema",

                            "name":
                                "yes_no_claim_pair",

                            "strict":
                                True,

                            "schema":
                                YN_SCHEMA,
                        }
                    },
                )
            )


            obj = json.loads(
                response.output_text
            )


            yes_claim, no_claim = (
                validate_yn_claims(

                    obj[
                        "yes_claim"
                    ],

                    obj[
                        "no_claim"
                    ],
                )
            )


            return (
                yes_claim,
                no_claim,
            )


        except Exception as exc:

            last_error = exc


            if (
                attempt
                <
                MAX_API_RETRIES - 1
            ):

                time.sleep(
                    2 ** attempt
                )


    raise RuntimeError(
        "Failed to generate yes/no claims.\n"
        f"Question: {question}\n"
        f"Last error: {last_error}"
    )



if os.path.exists(
    TEMPLATE_CACHE_PATH
):

    with open(
        TEMPLATE_CACHE_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        template_cache = json.load(
            f
        )

else:

    template_cache = {}


print()
print(
    "Existing strict templates:",
    len(
        template_cache
    )
)



print()
print(
    "=" * 80
)

print(
    "GENERATING STRICT QUESTION-ONLY CLAIM TEMPLATES"
)

print(
    "=" * 80
)


for row in tqdm(
    generated
):

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    question = (
        str(
            row[
                "question"
            ]
        )
        .strip()
    )


    kind = (
        "yes_no"

        if is_yes_no_question(
            question
        )

        else "slot"
    )



    if qi in template_cache:

        cached = (
            template_cache[
                qi
            ]
        )


        if (
            cached.get(
                "question"
            )
            !=
            question
        ):

            raise RuntimeError(
                f"Cached question mismatch qi={qi}"
            )


        if (
            cached.get(
                "kind"
            )
            !=
            kind
        ):

            raise RuntimeError(
                f"Cached question type mismatch qi={qi}"
            )


        continue




    if kind == "yes_no":

        yes_claim, no_claim = (
            generate_yn_claim_pair(
                question
            )
        )


        record = {
            "question_index":
                int(qi),
            "question":
                question,
            "kind":
                "yes_no",
            "yes_claim":
                yes_claim,
            "no_claim":
                no_claim,
            "converter_model":
                CLAIM_MODEL,
            "answer_seen_by_converter":
                False,
            "gold_seen_by_converter":
                False,
            "evidence_seen_by_converter":
                False,
        }


    else:

        template = (
            generate_non_yn_template(
                question
            )
        )


        record = {

            "question_index":
                int(qi),
            "question":
                question,
            "kind":
                "slot",
            "template":
                template,
            "converter_model":
                CLAIM_MODEL,
            "answer_seen_by_converter":
                False,
            "gold_seen_by_converter":
                False,
            "evidence_seen_by_converter":
                False,
        }


    template_cache[
        qi
    ] = record


    if (
        len(
            template_cache
        )
        %
        SAVE_EVERY
        ==
        0
    ):

        with open(
            TEMPLATE_CACHE_PATH,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                template_cache,
                f,
                indent=2,
                ensure_ascii=False,
            )


        print(
            f"{len(template_cache)}/1000 "
            "templates saved"
        )




with open(
    TEMPLATE_CACHE_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        template_cache,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert len(
    template_cache
) == 1000


print()
print(
    "All 1000 strict templates available."
)




def normalize_yes_no_answer(
    answer,
):

    a = (
        str(answer)
        .strip()
        .lower()
    )


    a = re.sub(
        r"[^a-z]",
        "",
        a,
    )


    if a in {
        "yes",
        "true",
    }:

        return "yes"


    if a in {
        "no",
        "false",
    }:

        return "no"


    return None



def construct_final_claim(
    row,
):

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    answer = (
        str(
            row[
                "majority_answer"
            ]
        )
        .strip()
    )


    if not answer:

        raise RuntimeError(
            f"Empty majority answer qi={qi}"
        )


    template_info = (
        template_cache[
            qi
        ]
    )


    kind = (
        template_info[
            "kind"
        ]
    )


    if kind == "slot":

        template = (
            template_info[
                "template"
            ]
        )



        claim = template.replace(
            "[[ANSWER]]",
            answer,
        )


        if (
            "[[ANSWER]]"
            in claim
        ):

            raise RuntimeError(
                f"Placeholder remains qi={qi}"
            )

        if (
            answer
            not in claim
        ):

            raise RuntimeError(
                f"Answer insertion failed qi={qi}"
            )


        branch = (
            "slot_substitution"
        )


    elif kind == "yes_no":

        yn = (
            normalize_yes_no_answer(
                answer
            )
        )


        if yn is None:

            raise RuntimeError(
                "Question was classified as yes/no "
                "but majority answer is not yes/no.\n"
                f"qi={qi}\n"
                f"Q={row['question']}\n"
                f"A={answer}"
            )


        if yn == "yes":

            claim = (
                template_info[
                    "yes_claim"
                ]
            )


        else:

            claim = (
                template_info[
                    "no_claim"
                ]
            )


        branch = (
            yn
        )


    else:

        raise RuntimeError(
            f"Unknown claim kind qi={qi}"
        )


    claim = (
        clean_output_text(
            claim
        )
    )


    if not claim:

        raise RuntimeError(
            f"Empty final claim qi={qi}"
        )


    return {

        "question_index":
            int(qi),

        "question":
            row[
                "question"
            ],

        "majority_answer":
            answer,

        "kind":
            kind,

        "branch":
            branch,

        "final_claim":
            claim,

        "converter_saw_answer":
            False,

        "converter_saw_gold":
            False,

        "converter_saw_evidence":
            False,
    }



Patched generation: 1000
Reasoning verification rows: 1000
dev: 800
test: 200
Confirmed: all reasoning trajectories match the previously verified trajectories.
All 11 frozen reasoning features found.
Questions treated as genuine yes/no: 66
Detected patched alternative questions: 12

Existing strict templates: 0

GENERATING STRICT QUESTION-ONLY CLAIM TEMPLATES


  0%|          | 0/1000 [00:00<?, ?it/s]

10/1000 templates saved
20/1000 templates saved
30/1000 templates saved
40/1000 templates saved
50/1000 templates saved
60/1000 templates saved
70/1000 templates saved
80/1000 templates saved
90/1000 templates saved
100/1000 templates saved
110/1000 templates saved
120/1000 templates saved
130/1000 templates saved
140/1000 templates saved
150/1000 templates saved
160/1000 templates saved
170/1000 templates saved
180/1000 templates saved
190/1000 templates saved
200/1000 templates saved
210/1000 templates saved
220/1000 templates saved
230/1000 templates saved
240/1000 templates saved
250/1000 templates saved
260/1000 templates saved
270/1000 templates saved
280/1000 templates saved
290/1000 templates saved
300/1000 templates saved
310/1000 templates saved
320/1000 templates saved
330/1000 templates saved
340/1000 templates saved
350/1000 templates saved
360/1000 templates saved
370/1000 templates saved
380/1000 templates saved
390/1000 templates saved
400/1000 templates saved
410/1000 

In [ ]:
def construct_final_claim(
    row,
):

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    answer = (
        str(
            row[
                "majority_answer"
            ]
        )
        .strip()
    )


    if not answer:

        raise RuntimeError(
            f"Empty majority answer qi={qi}"
        )


    template_info = (
        template_cache[
            qi
        ]
    )


    kind = (
        template_info[
            "kind"
        ]
    )


    if kind == "slot":

        template = (
            template_info[
                "template"
            ]
        )


        if template_info.get(
            "allow_full_sentence_answer",
            False,
        ):

            claim = answer

            branch = (
                "full_sentence_answer_verbatim"
            )


        else:

            claim = template.replace(
                "[[ANSWER]]",
                answer,
            )


            if (
                "[[ANSWER]]"
                in claim
            ):

                raise RuntimeError(
                    f"Placeholder remains qi={qi}"
                )


            if (
                answer
                not in claim
            ):

                raise RuntimeError(
                    f"Answer insertion failed qi={qi}"
                )


            branch = (
                "slot_substitution"
            )


    elif kind == "yes_no":

        yn = (
            normalize_yes_no_answer(
                answer
            )
        )


        if yn is None:

            raise RuntimeError(
                "Question classified as yes/no "
                "but majority answer is not yes/no.\n"
                f"qi={qi}\n"
                f"Q={row['question']}\n"
                f"A={answer}"
            )


        if yn == "yes":

            claim = (
                template_info[
                    "yes_claim"
                ]
            )

        else:

            claim = (
                template_info[
                    "no_claim"
                ]
            )


        branch = yn


    else:

        raise RuntimeError(
            f"Unknown claim kind qi={qi}"
        )


    claim = (
        clean_output_text(
            claim
        )
    )


    if not claim:

        raise RuntimeError(
            f"Empty final claim qi={qi}"
        )


    return {

        "question_index":
            int(qi),

        "question":
            row[
                "question"
            ],

        "majority_answer":
            answer,

        "kind":
            kind,

        "branch":
            branch,

        "final_claim":
            claim,

        "converter_saw_answer":
            False,

        "converter_saw_gold":
            False,

        "converter_saw_evidence":
            False,
    }

In [ ]:
import os
import json
import re

DATA_DIR = "/content/drive/MyDrive/hedge_run"

PATCHED_GENERATION_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_1000_YN_PATCHED.json"
)

TEMPLATE_CACHE_PATH = (
    f"{DATA_DIR}/"
    "hotpot_v2_question_claim_templates_STRICT_V1.json"
)

FINAL_CLAIMS_PATH = (
    f"{DATA_DIR}/"
    "hotpot_v2_final_claims_STRICT_V1.json"
)



with open(
    PATCHED_GENERATION_JSON,
    "r",
    encoding="utf-8",
) as f:
    generated = json.load(f)


with open(
    TEMPLATE_CACHE_PATH,
    "r",
    encoding="utf-8",
) as f:
    template_cache = json.load(f)


assert len(generated) == 1000
assert len(template_cache) == 1000


print(
    "Loaded generation:",
    len(generated)
)

print(
    "Loaded cached templates:",
    len(template_cache)
)



STRICT_TEMPLATE_PATCHES = {


    "2": (
        "The actor common to American Beauty and "
        "American Beauty is [[ANSWER]]."
    ),


    "17": (
        "The author of Armageddon in Retrospect "
        "was best known for the 1969 satire novel "
        "[[ANSWER]]."
    ),

    "281": (
        "African-American activist Allen Donaldson, "
        "who co-founded the Black Power movement of "
        "the 1960s and 1970s, adopted the name "
        "[[ANSWER]]."
    ),


    "495": (
        "Atom Egoyan's biggest commercial success "
        "was on [[ANSWER]]."
    ),
}


def validate_template(
    template,
):

    if (
        template.count(
            "[[ANSWER]]"
        )
        != 1
    ):
        raise ValueError(
            "Template must contain [[ANSWER]] exactly once."
        )

    if template.rstrip().endswith("?"):
        raise ValueError(
            "Template cannot be a question."
        )

    return template.strip()


for qi, new_template in (
    STRICT_TEMPLATE_PATCHES.items()
):

    if qi not in template_cache:
        raise RuntimeError(
            f"Missing qi={qi}"
        )

    if (
        template_cache[qi]["kind"]
        !=
        "slot"
    ):
        raise RuntimeError(
            f"qi={qi} is not a slot question"
        )

    new_template = (
        validate_template(
            new_template
        )
    )

    print()
    print(
        "QI:",
        qi
    )

    print(
        "OLD:",
        template_cache[
            qi
        ][
            "template"
        ]
    )

    print(
        "NEW:",
        new_template
    )

    template_cache[
        qi
    ][
        "template"
    ] = new_template

    template_cache[
        qi
    ][
        "manual_question_only_patch"
    ] = True


if "784" not in template_cache:
    raise RuntimeError(
        "QI 784 missing from template cache."
    )


template_cache[
    "784"
][
    "allow_full_sentence_answer"
] = True


print()
print(
    "QI 784 marked as "
    "full-sentence-answer case."
)



with open(
    TEMPLATE_CACHE_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        template_cache,
        f,
        indent=2,
        ensure_ascii=False,
    )


print()
print(
    "Patched template cache saved."
)



def normalize_yes_no_answer(
    answer,
):

    a = (
        str(answer)
        .strip()
        .lower()
    )

    a = re.sub(
        r"[^a-z]",
        "",
        a,
    )

    if a in {
        "yes",
        "true",
    }:
        return "yes"

    if a in {
        "no",
        "false",
    }:
        return "no"

    return None



def clean_output_text(
    text,
):

    return re.sub(
        r"\s+",
        " ",
        str(text),
    ).strip()


def construct_final_claim(
    row,
):

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    answer = str(
        row[
            "majority_answer"
        ]
    ).strip()


    if not answer:
        raise RuntimeError(
            f"Empty majority answer qi={qi}"
        )


    info = (
        template_cache[
            qi
        ]
    )


    kind = (
        info[
            "kind"
        ]
    )




    if kind == "slot":


        if info.get(
            "allow_full_sentence_answer",
            False,
        ):

            claim = answer

            branch = (
                "full_sentence_answer_verbatim"
            )

        else:

            template = (
                info[
                    "template"
                ]
            )

            if (
                template.count(
                    "[[ANSWER]]"
                )
                != 1
            ):
                raise RuntimeError(
                    f"Bad template qi={qi}"
                )

            claim = (
                template.replace(
                    "[[ANSWER]]",
                    answer,
                )
            )

            if (
                "[[ANSWER]]"
                in claim
            ):
                raise RuntimeError(
                    f"Placeholder remains qi={qi}"
                )

            branch = (
                "slot_substitution"
            )


    elif kind == "yes_no":

        yn = (
            normalize_yes_no_answer(
                answer
            )
        )

        if yn is None:
            raise RuntimeError(
                "Yes/no question has non-yes/no answer.\n"
                f"QI={qi}\n"
                f"Answer={answer}"
            )

        if yn == "yes":

            claim = (
                info[
                    "yes_claim"
                ]
            )

        else:

            claim = (
                info[
                    "no_claim"
                ]
            )

        branch = yn


    else:

        raise RuntimeError(
            f"Unknown template kind qi={qi}"
        )


    claim = (
        clean_output_text(
            claim
        )
    )


    if not claim:
        raise RuntimeError(
            f"Empty claim qi={qi}"
        )


    return {

        "question_index":
            int(qi),

        "question":
            row[
                "question"
            ],

        "majority_answer":
            answer,

        "kind":
            kind,

        "branch":
            branch,

        "final_claim":
            claim,

        "converter_saw_answer":
            False,

        "converter_saw_gold":
            False,

        "converter_saw_evidence":
            False,
    }



final_claims = {}


for row in generated:

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )

    final_claims[
        qi
    ] = (
        construct_final_claim(
            row
        )
    )


assert len(
    final_claims
) == 1000


with open(
    FINAL_CLAIMS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_claims,
        f,
        indent=2,
        ensure_ascii=False,
    )


print()
print(
    "Rebuilt and saved all 1000 final claims."
)



generated_by_qi = {
    str(
        int(
            row[
                "question_index"
            ]
        )
    ):
        row

    for row in generated
}


CHECK_IDS = [
    "2",
    "17",
    "281",
    "495",
    "784",
]


print()
print(
    "=" * 80
)

print(
    "FINAL PATCH CHECK"
)

print(
    "=" * 80
)


for qi in CHECK_IDS:

    row = (
        generated_by_qi[
            qi
        ]
    )

    result = (
        final_claims[
            qi
        ]
    )


    print()
    print(
        "QI:",
        qi
    )

    print(
        "QUESTION:"
    )
    print(
        row[
            "question"
        ]
    )

    print(
        "MAJORITY:"
    )
    print(
        row[
            "majority_answer"
        ]
    )

    print(
        "FINAL CLAIM:"
    )
    print(
        result[
            "final_claim"
        ]
    )

    print(
        "BRANCH:",
        result[
            "branch"
        ]
    )

    print(
        "-" * 80
    )


print()
print(
    "DONE."
)

Loaded generation: 1000
Loaded cached templates: 1000

QI: 2
OLD: American Beauty and American Beauty have [[ANSWER]] in common.
NEW: The actor common to American Beauty and American Beauty is [[ANSWER]].

QI: 17
OLD: Armageddon in Retrospect was written by the author who was best known for [[ANSWER]] satire novel from 1969.
NEW: The author of Armageddon in Retrospect was best known for the 1969 satire novel [[ANSWER]].

QI: 281
OLD: The name adopted by African-American activist Allen Donaldson co-founded the Black Power movement of the 1960s and 1970s as [[ANSWER]].
NEW: African-American activist Allen Donaldson, who co-founded the Black Power movement of the 1960s and 1970s, adopted the name [[ANSWER]].

QI: 495
OLD: Between stage and film, [[ANSWER]] was Atom Egoyan's biggest commercial success.
NEW: Atom Egoyan's biggest commercial success was on [[ANSWER]].

QI 784 marked as full-sentence-answer case.

Patched template cache saved.

Rebuilt and saved all 1000 final claims.

FINAL 

In [ ]:
import os
import json
import copy

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)



DATA_DIR = "/content/drive/MyDrive/hedge_run"


PATCHED_GENERATION_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_1000_YN_PATCHED.json"
)


REASONING_VERIFIED_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_verified_1000.json"
)


FINAL_CLAIMS_PATH = (
    f"{DATA_DIR}/"
    "hotpot_v2_final_claims_STRICT_V1.json"
)


NLI_PROGRESS_PATH = (
    f"{DATA_DIR}/"
    "hotpot_v2_final_nli_STRICT_V1_progress.json"
)


OUTPUT_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_1000.json"
)


OUTPUT_CSV = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_features17.csv"
)




NLI_MODEL_ID = (
    "MoritzLaurer/"
    "DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
)

SUPPORT_THRESHOLD = 0.50
CONTRADICTION_THRESHOLD = 0.50

NLI_MAX_LENGTH = 512
NLI_BATCH_SIZE = 16

SAVE_EVERY = 50


REASONING_FEATURES = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]


CONFIDENCE_FEATURES = [
    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]


FINAL_FEATURES = [
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]


FEATURE_COLUMNS = (
    REASONING_FEATURES
    +
    CONFIDENCE_FEATURES
    +
    FINAL_FEATURES
)


assert len(FEATURE_COLUMNS) == 17



with open(
    PATCHED_GENERATION_JSON,
    "r",
    encoding="utf-8",
) as f:
    generated = json.load(f)


with open(
    REASONING_VERIFIED_JSON,
    "r",
    encoding="utf-8",
) as f:
    reasoning_verified = json.load(f)


with open(
    FINAL_CLAIMS_PATH,
    "r",
    encoding="utf-8",
) as f:
    final_claims = json.load(f)


assert len(generated) == 1000
assert len(reasoning_verified) == 1000
assert len(final_claims) == 1000


print("Generation rows:", len(generated))
print("Reasoning rows:", len(reasoning_verified))
print("Final claims:", len(final_claims))


print(
    "dev:",
    sum(
        x["split"] == "dev"
        for x in generated
    )
)

print(
    "test:",
    sum(
        x["split"] == "test"
        for x in generated
    )
)



reasoning_by_qi = {
    int(row["question_index"]): row
    for row in reasoning_verified
}


generated_by_qi = {
    int(row["question_index"]): row
    for row in generated
}


assert len(reasoning_by_qi) == 1000
assert len(generated_by_qi) == 1000


for row in generated:

    qi = int(row["question_index"])

    old = reasoning_by_qi[qi]

    assert row["question"] == old["question"]

    assert (
        row.get("selected_steps", [])
        ==
        old.get("selected_steps", [])
    )


print(
    "Confirmed: all reasoning trajectories remain frozen."
)



EXPECTED_SUBSTRINGS = {

    "2":
        "actor common",

    "17":
        "1969 satire novel Slaughterhouse-Five",

    "281":
        "adopted the name Hakim Abdullah Jamal",

    "495":
        "biggest commercial success was on film",

    "784":
        "Vertical Horizon started their band first",
}


print()
print("=" * 80)
print("FINAL CLAIM INTEGRITY CHECK")
print("=" * 80)


for qi, expected in EXPECTED_SUBSTRINGS.items():

    claim = final_claims[qi]["final_claim"]

    print()
    print("QI:", qi)
    print(claim)

    if expected.lower() not in claim.lower():

        raise RuntimeError(
            f"Final claim for qi={qi} "
            "does not contain expected fixed form."
        )


# Specific duplication check.
if (
    "started their band first. started their band first"
    in
    final_claims["784"]["final_claim"].lower()
):

    raise RuntimeError(
        "QI 784 duplication has returned."
    )


print()
print(
    "All five manually inspected fixes are present."
)



print()
print("=" * 80)
print("LOADING NLI")
print("=" * 80)


nli_tokenizer = (
    AutoTokenizer.from_pretrained(
        NLI_MODEL_ID
    )
)


if torch.cuda.is_available():

    DEVICE = torch.device("cuda")

    DTYPE = torch.float16

else:

    DEVICE = torch.device("cpu")

    DTYPE = torch.float32


try:

    nli_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            NLI_MODEL_ID,
            dtype=DTYPE,
        )
    )

except TypeError:

    nli_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            NLI_MODEL_ID,
            torch_dtype=DTYPE,
        )
    )


nli_model = nli_model.to(
    DEVICE
)

nli_model.eval()


id2label = {
    int(k): str(v).lower()
    for k, v in
    nli_model.config.id2label.items()
}


ENT_IDX = next(
    i
    for i, label in id2label.items()
    if "entail" in label
)


NEU_IDX = next(
    i
    for i, label in id2label.items()
    if "neutral" in label
)


CON_IDX = next(
    i
    for i, label in id2label.items()
    if "contrad" in label
)


print(
    "NLI device:",
    DEVICE
)

print(
    "Labels:",
    id2label
)


def build_evidence_block(
    row,
):

    evidence = [
        str(x).strip()

        for x in row.get(
            "question_evidence",
            []
        )

        if str(x).strip()
    ]


    if not evidence:

        raise RuntimeError(
            "Missing question evidence for "
            f"qi={row['question_index']}"
        )


    return "\n".join(
        f"E{i}: {sentence}"

        for i, sentence in enumerate(
            evidence,
            start=1,
        )
    )



over_512 = []


for row in generated:

    qi = str(
        int(row["question_index"])
    )

    premise = build_evidence_block(
        row
    )

    hypothesis = (
        final_claims[qi][
            "final_claim"
        ]
    )


    token_ids = nli_tokenizer(
        premise,
        hypothesis,
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]


    if len(token_ids) > NLI_MAX_LENGTH:

        over_512.append(
            {
                "question_index":
                    int(qi),

                "length":
                    len(token_ids),
            }
        )


print()
print(
    "Pairs >512 tokens before truncation:",
    len(over_512)
)


if over_512:

    print(
        "Maximum pair length:",
        max(
            x["length"]
            for x in over_512
        )
    )



@torch.no_grad()
def score_nli_batch(
    premises,
    hypotheses,
):

    encoded = nli_tokenizer(
        premises,
        hypotheses,

        padding=True,

        truncation="only_first",

        max_length=NLI_MAX_LENGTH,

        return_tensors="pt",
    )


    encoded = {
        key:
            value.to(DEVICE)

        for key, value
        in encoded.items()
    }


    logits = (
        nli_model(
            **encoded
        )
        .logits
        .float()
    )


    probs = (
        torch.softmax(
            logits,
            dim=-1,
        )
        .cpu()
        .numpy()
    )


    return probs



if os.path.exists(
    NLI_PROGRESS_PATH
):

    with open(
        NLI_PROGRESS_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        nli_progress = json.load(f)

else:

    nli_progress = {}


print()
print(
    "Existing NLI results:",
    len(nli_progress)
)



pending = [
    row

    for row in generated

    if (
        str(
            int(row["question_index"])
        )
        not in
        nli_progress
    )
]


print(
    "Pending:",
    len(pending)
)



print()
print("=" * 80)
print("FINAL-ANSWER NLI")
print("=" * 80)


processed_since_save = 0


for start in tqdm(
    range(
        0,
        len(pending),
        NLI_BATCH_SIZE,
    )
):

    batch_rows = (
        pending[
            start:
            start + NLI_BATCH_SIZE
        ]
    )


    premises = [
        build_evidence_block(row)
        for row in batch_rows
    ]


    hypotheses = [
        final_claims[
            str(
                int(
                    row[
                        "question_index"
                    ]
                )
            )
        ][
            "final_claim"
        ]

        for row in batch_rows
    ]


    probs = score_nli_batch(
        premises,
        hypotheses,
    )


    for row, premise, claim, p in zip(
        batch_rows,
        premises,
        hypotheses,
        probs,
    ):

        qi = str(
            int(
                row[
                    "question_index"
                ]
            )
        )


        support = float(
            p[ENT_IDX]
        )


        neutral = float(
            p[NEU_IDX]
        )


        contradiction = float(
            p[CON_IDX]
        )


        if (
            support
            >=
            SUPPORT_THRESHOLD
        ):

            label = "supported"


        elif (
            contradiction
            >=
            CONTRADICTION_THRESHOLD
        ):

            label = "contradicted"


        else:

            label = "unclear"


        nli_progress[qi] = {

            "question_index":
                int(qi),

            "majority_answer":
                row[
                    "majority_answer"
                ],

            "final_claim":
                claim,

            "retrieved_evidence":
                copy.deepcopy(
                    row[
                        "question_evidence"
                    ]
                ),

            "support":
                support,

            "neutral":
                neutral,

            "contradiction":
                contradiction,

            "label":
                label,

            "contradicted":
                int(
                    contradiction
                    >=
                    CONTRADICTION_THRESHOLD
                ),

            "nli_model":
                NLI_MODEL_ID,

            "nli_evidence_mode":
                "combined_top10_bm25",

            "gold_used":
                False,
        }


        processed_since_save += 1


    if (
        processed_since_save
        >=
        SAVE_EVERY
    ):

        with open(
            NLI_PROGRESS_PATH,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                nli_progress,
                f,
                indent=2,
                ensure_ascii=False,
            )


        processed_since_save = 0


# Final progress save.

with open(
    NLI_PROGRESS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        nli_progress,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert len(
    nli_progress
) == 1000


print(
    "All final-answer NLI results complete."
)


final_rows = []


for gen_row in generated:

    qi_int = int(
        gen_row[
            "question_index"
        ]
    )


    qi = str(
        qi_int
    )


    reasoning_row = (
        reasoning_by_qi[
            qi_int
        ]
    )


    final_nli = (
        nli_progress[
            qi
        ]
    )



    reasoning_features = {
        feature:
            reasoning_row[
                "features"
            ][
                feature
            ]

        for feature
        in REASONING_FEATURES
    }



    confidence_features = {

        "self_consistency":
            float(
                gen_row[
                    "self_consistency"
                ]
            ),

        "answer_norm_entropy":
            float(
                gen_row[
                    "answer_norm_entropy"
                ]
            ),

        "answer_logprob":
            float(
                gen_row[
                    "answer_logprob"
                ]
            ),
    }


    final_features = {

        "fa_support_retr":
            float(
                final_nli[
                    "support"
                ]
            ),

        "fa_contradiction_retr":
            float(
                final_nli[
                    "contradiction"
                ]
            ),

        "fa_contradicted_retr":
            int(
                final_nli[
                    "contradicted"
                ]
            ),
    }


    features = {

        **reasoning_features,

        **confidence_features,

        **final_features,
    }


    if (
        set(features.keys())
        !=
        set(FEATURE_COLUMNS)
    ):

        raise RuntimeError(
            f"Wrong feature set qi={qi}"
        )


    final_row = {

        "question_index":
            qi_int,

        "split":
            gen_row[
                "split"
            ],

        "question":
            gen_row[
                "question"
            ],

        # Stored only for offline evaluation.
        "gold_answer":
            gen_row[
                "gold_answer"
            ],

        # Actual model prediction.
        "majority_answer":
            gen_row[
                "majority_answer"
            ],

        # Frozen reasoning.
        "selected_steps":
            copy.deepcopy(
                gen_row.get(
                    "selected_steps",
                    []
                )
            ),

        "step_reports":
            copy.deepcopy(
                reasoning_row.get(
                    "step_reports",
                    []
                )
            ),

        # Final uncertainty samples.
        "final_answers":
            copy.deepcopy(
                gen_row.get(
                    "final_answers",
                    []
                )
            ),

        "loop_final_answer":
            gen_row.get(
                "loop_final_answer",
                "",
            ),

        "question_evidence":
            copy.deepcopy(
                gen_row.get(
                    "question_evidence",
                    []
                )
            ),

        "self_consistency":
            float(
                gen_row[
                    "self_consistency"
                ]
            ),

        "answer_norm_entropy":
            float(
                gen_row[
                    "answer_norm_entropy"
                ]
            ),

        "answer_logprob":
            float(
                gen_row[
                    "answer_logprob"
                ]
            ),

        "final_claim":
            final_claims[
                qi
            ][
                "final_claim"
            ],

        "final_claim_info":
            copy.deepcopy(
                final_claims[
                    qi
                ]
            ),

        "final_answer_verification":
            copy.deepcopy(
                final_nli
            ),

        "features":
            features,

        "generation_protocol":
            copy.deepcopy(
                gen_row.get(
                    "generation_protocol",
                    {}
                )
            ),
    }


    if (
        "final_answer_question_type_patch"
        in
        gen_row
    ):

        final_row[
            "final_answer_question_type_patch"
        ] = copy.deepcopy(
            gen_row[
                "final_answer_question_type_patch"
            ]
        )


    final_rows.append(
        final_row
    )


assert len(
    final_rows
) == 1000


assert (
    sum(
        row["split"] == "dev"
        for row in final_rows
    )
    ==
    800
)


assert (
    sum(
        row["split"] == "test"
        for row in final_rows
    )
    ==
    200
)


# Confirm all 17 finite.

for row in final_rows:

    values = [
        float(
            row[
                "features"
            ][
                feature
            ]
        )

        for feature
        in FEATURE_COLUMNS
    ]


    if not np.all(
        np.isfinite(
            values
        )
    ):

        raise RuntimeError(
            "Non-finite feature in "
            f"qi={row['question_index']}"
        )


print(
    "All 17 features finite: YES"
)


for row in final_rows:

    qi = int(
        row[
            "question_index"
        ]
    )


    assert (
        row[
            "selected_steps"
        ]
        ==
        reasoning_by_qi[
            qi
        ][
            "selected_steps"
        ]
    )


print(
    "All reasoning trajectories still unchanged: YES"
)



with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_rows,
        f,
        indent=2,
        ensure_ascii=False,
    )



csv_rows = []


for row in final_rows:

    csv_rows.append(
        {

            "question_index":
                row[
                    "question_index"
                ],

            "split":
                row[
                    "split"
                ],

            "question":
                row[
                    "question"
                ],

            "gold_answer":
                row[
                    "gold_answer"
                ],

            "majority_answer":
                row[
                    "majority_answer"
                ],

            "final_claim":
                row[
                    "final_claim"
                ],

            **row[
                "features"
            ],
        }
    )


df = pd.DataFrame(
    csv_rows
)


df.to_csv(
    OUTPUT_CSV,
    index=False,
)



print()
print("=" * 80)
print("FINAL CLEAN HOTPOT FEATURE SUMMARY")
print("=" * 80)


print(
    "Rows:",
    len(df)
)


print(
    "dev:",
    int(
        df[
            "split"
        ]
        .eq("dev")
        .sum()
    )
)


print(
    "test:",
    int(
        df[
            "split"
        ]
        .eq("test")
        .sum()
    )
)


print(
    "Final support >= 0.5:",
    int(
        (
            df[
                "fa_support_retr"
            ]
            >=
            SUPPORT_THRESHOLD
        )
        .sum()
    )
)


print(
    "Final contradiction >= 0.5:",
    int(
        df[
            "fa_contradicted_retr"
        ]
        .sum()
    )
)


print(
    "Mean final support:",
    round(
        float(
            df[
                "fa_support_retr"
            ]
            .mean()
        ),
        4,
    )
)


print(
    "Mean final contradiction:",
    round(
        float(
            df[
                "fa_contradiction_retr"
            ]
            .mean()
        ),
        4,
    )
)


print(
    "Pairs >512 before truncation:",
    len(
        over_512
    )
)


print()
print(
    "17 FEATURES:"
)


for i, feature in enumerate(
    FEATURE_COLUMNS,
    start=1,
):

    print(
        f"{i:2d}. {feature}"
    )


print()
print(
    "AUTHORITATIVE FINAL JSON:"
)

print(
    OUTPUT_JSON
)


print()
print(
    "AUTHORITATIVE FINAL CSV:"
)

print(
    OUTPUT_CSV
)


print()
print(
    "DONE."
)

Generation rows: 1000
Reasoning rows: 1000
Final claims: 1000
dev: 800
test: 200
Confirmed: all reasoning trajectories remain frozen.

FINAL CLAIM INTEGRITY CHECK

QI: 2
The actor common to American Beauty and American Beauty is none.

QI: 17
The author of Armageddon in Retrospect was best known for the 1969 satire novel Slaughterhouse-Five.

QI: 281
African-American activist Allen Donaldson, who co-founded the Black Power movement of the 1960s and 1970s, adopted the name Hakim Abdullah Jamal.

QI: 495
Atom Egoyan's biggest commercial success was on film.

QI: 784
Vertical Horizon started their band first.

All five manually inspected fixes are present.

LOADING NLI


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors


NLI device: cuda
Labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}

Pairs >512 tokens before truncation: 30
Maximum pair length: 1033

Existing NLI results: 0
Pending: 1000

FINAL-ANSWER NLI


  0%|          | 0/63 [00:00<?, ?it/s]

All final-answer NLI results complete.
All 17 features finite: YES
All reasoning trajectories still unchanged: YES

FINAL CLEAN HOTPOT FEATURE SUMMARY
Rows: 1000
dev: 800
test: 200
Final support >= 0.5: 738
Final contradiction >= 0.5: 184
Mean final support: 0.7354
Mean final contradiction: 0.1889
Pairs >512 before truncation: 30

17 FEATURES:
 1. max_contradiction_score
 2. any_contradicted
 3. min_support_score
 4. num_steps
 5. frac_supported
 6. frac_contradicted
 7. frac_unclear
 8. mean_support
 9. mean_contradiction
10. conflict
11. support_spread
12. self_consistency
13. answer_norm_entropy
14. answer_logprob
15. fa_support_retr
16. fa_contradiction_retr
17. fa_contradicted_retr

AUTHORITATIVE FINAL JSON:
/content/drive/MyDrive/hedge_run/hotpot_passive_iterative_v2_FINAL_CLEAN_1000.json

AUTHORITATIVE FINAL CSV:
/content/drive/MyDrive/hedge_run/hotpot_passive_iterative_v2_FINAL_CLEAN_features17.csv

DONE.


In [ ]:


import os
import json
import time
import re

import pandas as pd

from tqdm.auto import tqdm
from openai import OpenAI



DATA_DIR = "/content/drive/MyDrive/hedge_run"


INPUT_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_1000.json"
)


JUDGE_PROGRESS = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_correctness_progress.json"
)


JUDGE_OUTPUT = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_correctness_1000.json"
)


LABELLED_JSON = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_LABELLED_1000.json"
)


LABELLED_CSV = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
)


JUDGE_MODEL = "gpt-4o-mini"

SAVE_EVERY = 10

MAX_RETRIES = 5



OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)


if not OPENAI_API_KEY:

    try:

        from google.colab import userdata

        OPENAI_API_KEY = userdata.get(
            "OPENAI_API_KEY"
        )

    except Exception:

        OPENAI_API_KEY = None


if not OPENAI_API_KEY:

    raise RuntimeError(
        "OPENAI_API_KEY not found."
    )


client = OpenAI(
    api_key=OPENAI_API_KEY
)



with open(
    INPUT_JSON,
    "r",
    encoding="utf-8",
) as f:

    rows = json.load(f)


assert len(rows) == 1000


print(
    "Loaded:",
    len(rows)
)


print(
    "dev:",
    sum(
        row["split"] == "dev"
        for row in rows
    )
)


print(
    "test:",
    sum(
        row["split"] == "test"
        for row in rows
    )
)



def normalize_answer(
    text,
):

    text = str(
        text
    ).lower()


    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text,
    )


    text = re.sub(
        r"\b(a|an|the)\b",
        " ",
        text,
    )


    text = re.sub(
        r"\s+",
        " ",
        text,
    )


    return text.strip()



JUDGE_INSTRUCTIONS = """
You are an offline correctness evaluator for a question-answering experiment.

You will receive:

1. a question,
2. the reference answer,
3. the model's predicted answer.

Determine whether the model prediction should be counted as correct.

Rules:

- Accept semantically equivalent paraphrases.
- Accept aliases, abbreviations, spelling variants, and harmless formatting differences.
- Accept an answer that contains the same required factual content in a slightly longer phrase.
- Do not require exact string matching.
- Do NOT accept a factually different answer.
- Do NOT accept an answer that answers a different relation.
- Do NOT accept an answer that is only related to the reference.
- Do NOT accept an answer that is broader or narrower when that changes what the question asks.
- For comparison questions, preserve comparison direction.
- For yes/no questions, the yes/no proposition must agree.
- Judge only whether the prediction answers the question correctly.
- Do not use the model's reasoning or confidence; they are not provided.

Return a binary correctness judgement.
""".strip()



JUDGE_SCHEMA = {

    "type":
        "object",

    "properties": {

        "correct": {
            "type":
                "boolean"
        },

        "reason": {
            "type":
                "string"
        },
    },

    "required": [
        "correct",
        "reason",
    ],

    "additionalProperties":
        False,
}



def judge_answer(
    question,
    gold,
    prediction,
):

    user_input = f"""
QUESTION:
{question}

REFERENCE ANSWER:
{gold}

MODEL PREDICTION:
{prediction}
""".strip()


    last_error = None


    for attempt in range(
        MAX_RETRIES
    ):

        try:

            response = client.responses.create(

                model=JUDGE_MODEL,

                instructions=JUDGE_INSTRUCTIONS,

                input=user_input,

                temperature=0,

                max_output_tokens=120,

                store=False,

                text={
                    "format": {

                        "type":
                            "json_schema",

                        "name":
                            "qa_correctness_judgment",

                        "strict":
                            True,

                        "schema":
                            JUDGE_SCHEMA,
                    }
                },
            )


            result = json.loads(
                response.output_text
            )


            return {
                "correct":
                    bool(
                        result[
                            "correct"
                        ]
                    ),

                "reason":
                    str(
                        result[
                            "reason"
                        ]
                    ).strip(),
            }


        except Exception as exc:

            last_error = exc


            if (
                attempt
                <
                MAX_RETRIES - 1
            ):

                time.sleep(
                    2 ** attempt
                )


    raise RuntimeError(
        "Correctness judge failed.\n"
        f"Question: {question}\n"
        f"Gold: {gold}\n"
        f"Prediction: {prediction}\n"
        f"Last error: {last_error}"
    )



if os.path.exists(
    JUDGE_PROGRESS
):

    with open(
        JUDGE_PROGRESS,
        "r",
        encoding="utf-8",
    ) as f:

        judged = json.load(f)

else:

    judged = {}


print(
    "\nExisting judgments:",
    len(judged)
)



print()
print(
    "=" * 80
)

print(
    "OFFLINE CORRECTNESS JUDGING"
)

print(
    "=" * 80
)


for row in tqdm(
    rows
):

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    if qi in judged:

        continue


    question = str(
        row[
            "question"
        ]
    ).strip()


    gold = str(
        row[
            "gold_answer"
        ]
    ).strip()


    prediction = str(
        row[
            "majority_answer"
        ]
    ).strip()


    result = judge_answer(
        question,
        gold,
        prediction,
    )


    judged[
        qi
    ] = {

        "question_index":
            int(qi),

        "question":
            question,

        "gold_answer":
            gold,

        "majority_answer":
            prediction,

        "judge_correct":
            bool(
                result[
                    "correct"
                ]
            ),

        "judge_reason":
            result[
                "reason"
            ],

        "normalized_exact_match":
            (
                normalize_answer(
                    prediction
                )
                ==
                normalize_answer(
                    gold
                )
            ),

        "judge_model":
            JUDGE_MODEL,
    }


    if (
        len(judged)
        %
        SAVE_EVERY
        ==
        0
    ):

        with open(
            JUDGE_PROGRESS,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                judged,
                f,
                indent=2,
                ensure_ascii=False,
            )


        print(
            f"{len(judged)}/1000 saved"
        )



with open(
    JUDGE_PROGRESS,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        judged,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert len(judged) == 1000


ordered_judgments = [

    judged[
        str(
            int(
                row[
                    "question_index"
                ]
            )
        )
    ]

    for row in rows
]


with open(
    JUDGE_OUTPUT,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        ordered_judgments,
        f,
        indent=2,
        ensure_ascii=False,
    )


labelled_rows = []


for row in rows:

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    new_row = dict(
        row
    )


    new_row[
        "judge_correct"
    ] = bool(
        judged[
            qi
        ][
            "judge_correct"
        ]
    )


    new_row[
        "judge_reason"
    ] = (
        judged[
            qi
        ][
            "judge_reason"
        ]
    )


    labelled_rows.append(
        new_row
    )


with open(
    LABELLED_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        labelled_rows,
        f,
        indent=2,
        ensure_ascii=False,
    )


REASONING_FEATURES = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]


CONFIDENCE_FEATURES = [
    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]


FINAL_FEATURES = [
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]


FEATURE_COLUMNS = (
    REASONING_FEATURES
    +
    CONFIDENCE_FEATURES
    +
    FINAL_FEATURES
)


csv_rows = []


for row in labelled_rows:

    csv_rows.append(
        {

            "question_index":
                row[
                    "question_index"
                ],

            "split":
                row[
                    "split"
                ],

            "question":
                row[
                    "question"
                ],

            "gold_answer":
                row[
                    "gold_answer"
                ],

            "majority_answer":
                row[
                    "majority_answer"
                ],

            "judge_correct":
                int(
                    row[
                        "judge_correct"
                    ]
                ),

            "is_wrong":
                int(
                    not row[
                        "judge_correct"
                    ]
                ),

            **row[
                "features"
            ],
        }
    )


df = pd.DataFrame(
    csv_rows
)


df.to_csv(
    LABELLED_CSV,
    index=False,
)


n_correct = int(
    df[
        "judge_correct"
    ].sum()
)


n_wrong = (
    len(df)
    -
    n_correct
)


dev = df[
    df[
        "split"
    ]
    ==
    "dev"
]


test = df[
    df[
        "split"
    ]
    ==
    "test"
]


exact_agreement = sum(

    bool(
        x[
            "normalized_exact_match"
        ]
    )
    ==
    bool(
        x[
            "judge_correct"
        ]
    )

    for x
    in ordered_judgments
)


print()
print(
    "=" * 80
)

print(
    "CORRECTNESS JUDGE SUMMARY"
)

print(
    "=" * 80
)


print(
    "Total:",
    len(df)
)


print(
    "Correct:",
    n_correct
)


print(
    "Wrong:",
    n_wrong
)


print(
    "Overall accuracy:",
    round(
        n_correct
        /
        len(df),
        4,
    )
)


print()
print(
    "DEV"
)

print(
    "n:",
    len(dev)
)

print(
    "correct:",
    int(
        dev[
            "judge_correct"
        ].sum()
    )
)

print(
    "wrong:",
    int(
        dev[
            "is_wrong"
        ].sum()
    )
)

print(
    "accuracy:",
    round(
        float(
            dev[
                "judge_correct"
            ].mean()
        ),
        4,
    )
)


print()
print(
    "TEST"
)

print(
    "n:",
    len(test)
)

print(
    "correct:",
    int(
        test[
            "judge_correct"
        ].sum()
    )
)

print(
    "wrong:",
    int(
        test[
            "is_wrong"
        ].sum()
    )
)

print(
    "accuracy:",
    round(
        float(
            test[
                "judge_correct"
            ].mean()
        ),
        4,
    )
)


print()
print(
    "Judge/exact-match agreement:",
    exact_agreement,
    "/",
    len(df)
)


print()
print(
    "Saved judgments:"
)

print(
    JUDGE_OUTPUT
)


print()
print(
    "Saved labelled JSON:"
)

print(
    LABELLED_JSON
)


print()
print(
    "Saved labelled feature CSV:"
)

print(
    LABELLED_CSV
)


print()
print(
    "DONE."
)

Loaded: 1000
dev: 800
test: 200

Existing judgments: 0

OFFLINE CORRECTNESS JUDGING


  0%|          | 0/1000 [00:00<?, ?it/s]

10/1000 saved
20/1000 saved
30/1000 saved
40/1000 saved
50/1000 saved
60/1000 saved
70/1000 saved
80/1000 saved
90/1000 saved
100/1000 saved
110/1000 saved
120/1000 saved
130/1000 saved
140/1000 saved
150/1000 saved
160/1000 saved
170/1000 saved
180/1000 saved
190/1000 saved
200/1000 saved
210/1000 saved
220/1000 saved
230/1000 saved
240/1000 saved
250/1000 saved
260/1000 saved
270/1000 saved
280/1000 saved
290/1000 saved
300/1000 saved
310/1000 saved
320/1000 saved
330/1000 saved
340/1000 saved
350/1000 saved
360/1000 saved
370/1000 saved
380/1000 saved
390/1000 saved
400/1000 saved
410/1000 saved
420/1000 saved
430/1000 saved
440/1000 saved
450/1000 saved
460/1000 saved
470/1000 saved
480/1000 saved
490/1000 saved
500/1000 saved
510/1000 saved
520/1000 saved
530/1000 saved
540/1000 saved
550/1000 saved
560/1000 saved
570/1000 saved
580/1000 saved
590/1000 saved
600/1000 saved
610/1000 saved
620/1000 saved
630/1000 saved
640/1000 saved
650/1000 saved
660/1000 saved
670/1000 saved
680/

Experiment 1

In [ ]:


import os
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score



DATA_DIR = "/content/drive/MyDrive/hedge_run"

INPUT_CSV = (
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
)

OUTPUT_JSON = (
    f"{DATA_DIR}/"
    "hotpot_experiment1_grouped_auroc.json"
)

OUTPUT_OOF_CSV = (
    f"{DATA_DIR}/"
    "hotpot_experiment1_oof_predictions.csv"
)


SEED = 42

N_SPLITS = 5

N_BOOTSTRAP = 5000

RISK_PERCENTILE = 60



REASONING_FEATURES = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]


CONFIDENCE_FEATURES = [
    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]


FINAL_VERIFICATION_FEATURES = [
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]


ANSWER_LEVEL_FEATURES = (
    CONFIDENCE_FEATURES
    +
    FINAL_VERIFICATION_FEATURES
)


FULL_FEATURES = (
    REASONING_FEATURES
    +
    CONFIDENCE_FEATURES
    +
    FINAL_VERIFICATION_FEATURES
)


FEATURE_GROUPS = {

    "Confidence":
        CONFIDENCE_FEATURES,

    "Final verification":
        FINAL_VERIFICATION_FEATURES,

    "Reasoning":
        REASONING_FEATURES,

    "Answer-level combined":
        ANSWER_LEVEL_FEATURES,

    "Full 17":
        FULL_FEATURES,
}


assert len(REASONING_FEATURES) == 11
assert len(CONFIDENCE_FEATURES) == 3
assert len(FINAL_VERIFICATION_FEATURES) == 3
assert len(ANSWER_LEVEL_FEATURES) == 6
assert len(FULL_FEATURES) == 17



df = pd.read_csv(
    INPUT_CSV
)


print(
    "Total rows:",
    len(df)
)


print(
    df["split"].value_counts()
)


dev = (
    df[
        df["split"] == "dev"
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(dev) == 800


# Target:
# 1 = wrong
# 0 = correct

y = (
    dev[
        "is_wrong"
    ]
    .astype(int)
    .to_numpy()
)


print()
print("=" * 80)
print("DEVELOPMENT LABEL DISTRIBUTION")
print("=" * 80)

print(
    "N:",
    len(y)
)

print(
    "Correct:",
    int((y == 0).sum())
)

print(
    "Wrong:",
    int((y == 1).sum())
)

print(
    "Error rate:",
    round(
        float(y.mean()),
        4,
    )
)


assert (y == 0).sum() == 505
assert (y == 1).sum() == 295



for feature in FULL_FEATURES:

    if feature not in dev.columns:

        raise RuntimeError(
            f"Missing feature: {feature}"
        )


X_all = (
    dev[
        FULL_FEATURES
    ]
    .astype(float)
    .to_numpy()
)


if not np.all(
    np.isfinite(X_all)
):

    raise RuntimeError(
        "Non-finite value found in features."
    )


print(
    "\nAll 17 development features finite: YES"
)



skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)


folds = list(
    skf.split(
        np.zeros(
            len(y)
        ),
        y,
    )
)


# Check fold distributions.

print()
print("=" * 80)
print("5-FOLD SPLIT")
print("=" * 80)


for fold_id, (
    train_idx,
    val_idx,
) in enumerate(
    folds,
    start=1,
):

    print(
        f"Fold {fold_id}:",
        f"train={len(train_idx)}",
        f"val={len(val_idx)}",
        f"val_wrong={int(y[val_idx].sum())}",
        f"val_correct={int((1-y[val_idx]).sum())}",
    )



def make_classifier():

    return Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),

            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    random_state=SEED,
                    max_iter=5000,
                    solver="lbfgs",
                ),
            ),
        ]
    )



def get_oof_predictions(
    X,
    y,
):

    oof_risk = np.full(
        len(y),
        np.nan,
        dtype=float,
    )


    fold_aurocs = []


    for fold_id, (
        train_idx,
        val_idx,
    ) in enumerate(
        folds,
        start=1,
    ):

        model = make_classifier()


        model.fit(
            X[train_idx],
            y[train_idx],
        )


        # Probability of class 1 = WRONG.
        risk = (
            model.predict_proba(
                X[val_idx]
            )[:, 1]
        )


        oof_risk[
            val_idx
        ] = risk


        fold_auc = roc_auc_score(
            y[val_idx],
            risk,
        )


        fold_aurocs.append(
            float(
                fold_auc
            )
        )


    if not np.all(
        np.isfinite(
            oof_risk
        )
    ):

        raise RuntimeError(
            "Missing OOF predictions."
        )


    overall_auc = float(
        roc_auc_score(
            y,
            oof_risk,
        )
    )


    return (
        oof_risk,
        fold_aurocs,
        overall_auc,
    )



def bootstrap_auc_ci(
    y,
    scores,
    n_bootstrap=N_BOOTSTRAP,
    seed=SEED,
):

    rng = np.random.default_rng(
        seed
    )


    n = len(y)

    aucs = []


    for _ in range(
        n_bootstrap
    ):

        idx = rng.integers(
            0,
            n,
            size=n,
        )


        y_b = y[idx]



        if np.unique(
            y_b
        ).size < 2:

            continue


        auc = roc_auc_score(
            y_b,
            scores[idx],
        )


        aucs.append(
            auc
        )


    lower = float(
        np.percentile(
            aucs,
            2.5,
        )
    )


    upper = float(
        np.percentile(
            aucs,
            97.5,
        )
    )


    return (
        lower,
        upper,
    )


results = {}

oof_predictions = {}


print()
print("=" * 80)
print("EXPERIMENT 1 — GROUPED OOF AUROC")
print("=" * 80)


for group_name, features in (
    FEATURE_GROUPS.items()
):

    X = (
        dev[
            features
        ]
        .astype(float)
        .to_numpy()
    )


    (
        oof_risk,
        fold_aurocs,
        overall_auc,
    ) = get_oof_predictions(
        X,
        y,
    )


    ci_low, ci_high = (
        bootstrap_auc_ci(
            y,
            oof_risk,
        )
    )


    results[
        group_name
    ] = {

        "n_features":
            len(features),

        "features":
            features,

        "oof_auroc":
            overall_auc,

        "ci95_low":
            ci_low,

        "ci95_high":
            ci_high,

        "fold_aurocs":
            fold_aurocs,

        "mean_fold_auroc":
            float(
                np.mean(
                    fold_aurocs
                )
            ),

        "std_fold_auroc":
            float(
                np.std(
                    fold_aurocs,
                    ddof=1,
                )
            ),
    }


    oof_predictions[
        group_name
    ] = oof_risk


    print()
    print(
        group_name
    )

    print(
        "features:",
        len(features)
    )

    print(
        "OOF AUROC:",
        round(
            overall_auc,
            4,
        )
    )

    print(
        "95% CI:",
        (
            round(ci_low, 4),
            round(ci_high, 4),
        )
    )

    print(
        "fold AUROCs:",
        [
            round(x, 4)
            for x in fold_aurocs
        ]
    )



full_oof_risk = (
    oof_predictions[
        "Full 17"
    ]
)


risk_threshold = float(
    np.percentile(
        full_oof_risk,
        RISK_PERCENTILE,
    )
)


dev_answer = (
    full_oof_risk
    <=
    risk_threshold
)


dev_abstain = (
    ~dev_answer
)


print()
print("=" * 80)
print("FULL-17 FROZEN RISK THRESHOLD")
print("=" * 80)


print(
    "Percentile:",
    RISK_PERCENTILE
)


print(
    "Risk threshold:",
    round(
        risk_threshold,
        6,
    )
)


print(
    "OOF development answered:",
    int(
        dev_answer.sum()
    )
)


print(
    "OOF development abstained:",
    int(
        dev_abstain.sum()
    )
)


print(
    "OOF development coverage:",
    round(
        float(
            dev_answer.mean()
        ),
        4,
    )
)



correct = (
    1 - y
)


answered_correct = (
    correct[
        dev_answer
    ]
)


selective_accuracy = float(
    answered_correct.mean()
)


answered_wrong = int(
    y[
        dev_answer
    ].sum()
)


abstained_correct = int(
    correct[
        dev_abstain
    ].sum()
)


print()
print(
    "OOF selective accuracy:",
    round(
        selective_accuracy,
        4,
    )
)


print(
    "Wrong answers returned:",
    answered_wrong
)


print(
    "Correct answers abstained:",
    abstained_correct
)



oof_df = dev[
    [
        "question_index",
        "question",
        "gold_answer",
        "majority_answer",
        "judge_correct",
        "is_wrong",
    ]
].copy()


for group_name, scores in (
    oof_predictions.items()
):

    col = (
        "risk_"
        +
        group_name
        .lower()
        .replace(
            " ",
            "_",
        )
        .replace(
            "-",
            "_",
        )
    )


    oof_df[
        col
    ] = scores


oof_df[
    "full17_answer"
] = (
    dev_answer.astype(int)
)


oof_df.to_csv(
    OUTPUT_OOF_CSV,
    index=False,
)



output = {

    "dataset":
        "HotpotQA",

    "development_n":
        len(dev),

    "correct":
        int(
            (y == 0).sum()
        ),

    "wrong":
        int(
            (y == 1).sum()
        ),

    "cv": {
        "n_splits":
            N_SPLITS,

        "shuffle":
            True,

        "random_state":
            SEED,
    },

    "classifier": {
        "type":
            "logistic_regression",

        "standardized":
            True,

        "class_weight":
            "balanced",
    },

    "target":
        "error",

    "feature_group_results":
        results,

    "full17_threshold": {
        "source":
            "OOF development error risk",

        "percentile":
            RISK_PERCENTILE,

        "threshold":
            risk_threshold,
    },
}


with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        output,
        f,
        indent=2,
    )



summary_rows = []


for group_name, result in (
    results.items()
):

    summary_rows.append(
        {
            "Feature group":
                group_name,

            "Features":
                result[
                    "n_features"
                ],

            "OOF AUROC":
                result[
                    "oof_auroc"
                ],

            "95% CI low":
                result[
                    "ci95_low"
                ],

            "95% CI high":
                result[
                    "ci95_high"
                ],

            "Mean fold AUROC":
                result[
                    "mean_fold_auroc"
                ],
        }
    )


summary = pd.DataFrame(
    summary_rows
)


print()
print("=" * 80)
print("FINAL EXPERIMENT 1 TABLE")
print("=" * 80)


print(
    summary.to_string(
        index=False,

        formatters={

            "OOF AUROC":
                lambda x:
                    f"{x:.4f}",

            "95% CI low":
                lambda x:
                    f"{x:.4f}",

            "95% CI high":
                lambda x:
                    f"{x:.4f}",

            "Mean fold AUROC":
                lambda x:
                    f"{x:.4f}",
        },
    )
)


print()
print(
    "Saved results:"
)

print(
    OUTPUT_JSON
)


print()
print(
    "Saved OOF predictions:"
)

print(
    OUTPUT_OOF_CSV
)


print()
print(
    "DONE."
)

Total rows: 1000
split
dev     800
test    200
Name: count, dtype: int64

DEVELOPMENT LABEL DISTRIBUTION
N: 800
Correct: 505
Wrong: 295
Error rate: 0.3688

All 17 development features finite: YES

5-FOLD SPLIT
Fold 1: train=640 val=160 val_wrong=59 val_correct=101
Fold 2: train=640 val=160 val_wrong=59 val_correct=101
Fold 3: train=640 val=160 val_wrong=59 val_correct=101
Fold 4: train=640 val=160 val_wrong=59 val_correct=101
Fold 5: train=640 val=160 val_wrong=59 val_correct=101

EXPERIMENT 1 — GROUPED OOF AUROC

Confidence
features: 3
OOF AUROC: 0.5796
95% CI: (0.538, 0.6227)
fold AUROCs: [0.6862, 0.5873, 0.6379, 0.5342, 0.6743]

Final verification
features: 3
OOF AUROC: 0.7366
95% CI: (0.6991, 0.7716)
fold AUROCs: [0.8089, 0.7261, 0.7035, 0.7567, 0.7615]

Reasoning
features: 11
OOF AUROC: 0.6103
95% CI: (0.5706, 0.6515)
fold AUROCs: [0.5816, 0.5931, 0.5962, 0.6942, 0.6139]

Answer-level combined
features: 6
OOF AUROC: 0.748
95% CI: (0.7095, 0.7826)
fold AUROCs: [0.8277, 0.729, 0.743

RQ1: Does Full-17 improve over Answer-Level Combined?

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score


DATA_DIR = "/content/drive/MyDrive/hedge_run"

OOF_PATH = (
    f"{DATA_DIR}/"
    "hotpot_experiment1_oof_predictions.csv"
)


df = pd.read_csv(
    OOF_PATH
)


y = (
    df["is_wrong"]
    .astype(int)
    .to_numpy()
)


answer_scores = (
    df[
        "risk_answer_level_combined"
    ]
    .astype(float)
    .to_numpy()
)


full_scores = (
    df[
        "risk_full_17"
    ]
    .astype(float)
    .to_numpy()
)



auc_answer = roc_auc_score(
    y,
    answer_scores,
)

auc_full = roc_auc_score(
    y,
    full_scores,
)


observed_delta = (
    auc_full
    -
    auc_answer
)




N_BOOTSTRAP = 10000
SEED = 42

rng = np.random.default_rng(
    SEED
)


deltas = []


for _ in range(
    N_BOOTSTRAP
):

    idx = rng.integers(
        0,
        len(y),
        size=len(y),
    )


    y_b = y[idx]



    if np.unique(
        y_b
    ).size < 2:

        continue


    auc_answer_b = roc_auc_score(
        y_b,
        answer_scores[idx],
    )


    auc_full_b = roc_auc_score(
        y_b,
        full_scores[idx],
    )


    deltas.append(
        auc_full_b
        -
        auc_answer_b
    )


deltas = np.asarray(
    deltas
)


ci_low = float(
    np.percentile(
        deltas,
        2.5,
    )
)


ci_high = float(
    np.percentile(
        deltas,
        97.5,
    )
)


p_full_not_better = float(
    np.mean(
        deltas <= 0
    )
)



p_two_sided = min(
    1.0,
    2
    *
    min(
        np.mean(
            deltas <= 0
        ),
        np.mean(
            deltas >= 0
        ),
    ),
)


print(
    "=" * 80
)

print(
    "PAIRED AUROC DIFFERENCE — RQ1"
)

print(
    "=" * 80
)


print(
    "Answer-level AUROC:",
    round(
        auc_answer,
        4,
    )
)


print(
    "Full-17 AUROC:",
    round(
        auc_full,
        4,
    )
)


print(
    "Observed delta:",
    round(
        observed_delta,
        4,
    )
)


print(
    "95% paired bootstrap CI:",
    (
        round(
            ci_low,
            4,
        ),
        round(
            ci_high,
            4,
        ),
    )
)


print(
    "P(delta <= 0):",
    round(
        p_full_not_better,
        4,
    )
)


print(
    "Approx. two-sided p:",
    round(
        p_two_sided,
        4,
    )
)


print(
    "Bootstrap mean delta:",
    round(
        float(
            deltas.mean()
        ),
        4,
    )
)

PAIRED AUROC DIFFERENCE — RQ1
Answer-level AUROC: 0.748
Full-17 AUROC: 0.7511
Observed delta: 0.0031
95% paired bootstrap CI: (-0.0183, 0.0249)
P(delta <= 0): 0.3843
Approx. two-sided p: 0.7686
Bootstrap mean delta: 0.0033


In [ ]:
import json
import re
from collections import Counter


PATH = (
    "/content/drive/MyDrive/hedge_run/"
    "2wiki_passive_iterative_v2_1000.json"
)


with open(
    PATH,
    "r",
    encoding="utf-8",
) as f:

    rows = json.load(f)


assert len(rows) == 1000


AUX_STARTS = (
    "is ",
    "are ",
    "was ",
    "were ",
    "do ",
    "does ",
    "did ",
    "can ",
    "could ",
    "would ",
    "will ",
    "has ",
    "have ",
    "had ",
)


def norm(
    text,
):

    text = (
        str(text)
        .lower()
        .strip()
    )

    text = re.sub(
        r"[^\w\s]",
        "",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text


def old_yes_no_rule(
    question,
):

    q = (
        str(question)
        .lower()
        .strip()
    )

    return q.startswith(
        AUX_STARTS
    )


def gold_is_yes_no(
    gold,
):

    return (
        norm(gold)
        in {
            "yes",
            "no",
        }
    )


def prediction_is_yes_no(
    answer,
):

    return (
        norm(answer)
        in {
            "yes",
            "no",
        }
    )


def contains_or(
    question,
):

    return bool(
        re.search(
            r"\bor\b",
            str(question).lower(),
        )
    )



old_yn = [
    row

    for row in rows

    if old_yes_no_rule(
        row["question"]
    )
]



or_candidates = [
    row

    for row in rows

    if (
        old_yes_no_rule(
            row["question"]
        )
        and
        contains_or(
            row["question"]
        )
    )
]



potentially_misclassified = [
    row

    for row in old_yn

    if not gold_is_yes_no(
        row["gold_answer"]
    )
]


forced_yes_no = [
    row

    for row in potentially_misclassified

    if prediction_is_yes_no(
        row["majority_answer"]
    )
]



print(
    "=" * 80
)

print(
    "2WIKI YES/NO HEURISTIC DIAGNOSTIC"
)

print(
    "=" * 80
)


print(
    "Total:",
    len(rows)
)


print(
    "dev:",
    sum(
        row["split"] == "dev"
        for row in rows
    )
)


print(
    "test:",
    sum(
        row["split"] == "test"
        for row in rows
    )
)


print(
    "\nQuestions classified yes/no by old rule:",
    len(old_yn)
)


print(
    "True yes/no according to gold:",
    sum(
        gold_is_yes_no(
            row["gold_answer"]
        )
        for row in rows
    )
)


print(
    "Auxiliary-initial + 'or' candidates:",
    len(or_candidates)
)


print(
    "Potentially misclassified by old rule:",
    len(
        potentially_misclassified
    )
)


print(
    "Potentially misclassified AND "
    "majority answer is Yes/No:",
    len(
        forced_yes_no
    )
)


print(
    "\nQuestion-start patterns:"
)

print(
    Counter(
        row[
            "question"
        ]
        .strip()
        .split()[0]
        .lower()

        for row
        in potentially_misclassified
    )
)



print()
print(
    "=" * 80
)

print(
    "AUXILIARY + OR CANDIDATES"
)

print(
    "=" * 80
)


for i, row in enumerate(
    or_candidates,
    start=1,
):

    print()
    print(
        f"{i}. QUESTION:"
    )

    print(
        row["question"]
    )


    print(
        "GOLD:"
    )

    print(
        row["gold_answer"]
    )


    print(
        "MAJORITY:"
    )

    print(
        row["majority_answer"]
    )


    print(
        "LOOP FINAL:"
    )

    print(
        row.get(
            "loop_final_answer",
            ""
        )
    )


    print(
        "FINAL SAMPLES:"
    )

    print(
        row.get(
            "final_answers",
            []
        )
    )


    print(
        "GOLD IS YES/NO:",
        gold_is_yes_no(
            row["gold_answer"]
        )
    )


    print(
        "MAJORITY IS YES/NO:",
        prediction_is_yes_no(
            row["majority_answer"]
        )
    )


    print(
        "-" * 80
    )



non_or_mismatches = [
    row

    for row in potentially_misclassified

    if not contains_or(
        row["question"]
    )
]


print()
print(
    "=" * 80
)

print(
    "NON-'OR' POTENTIAL MISCLASSIFICATIONS"
)

print(
    "=" * 80
)


print(
    "Count:",
    len(non_or_mismatches)
)


for row in non_or_mismatches:

    print()
    print(
        "QUESTION:"
    )

    print(
        row["question"]
    )


    print(
        "GOLD:"
    )

    print(
        row["gold_answer"]
    )


    print(
        "MAJORITY:"
    )

    print(
        row["majority_answer"]
    )


    print(
        "FINAL SAMPLES:"
    )

    print(
        row.get(
            "final_answers",
            []
        )
    )


    print(
        "-" * 80
    )


print()
print(
    "DONE."
)

2WIKI YES/NO HEURISTIC DIAGNOSTIC
Total: 1000
dev: 800
test: 200

Questions classified yes/no by old rule: 106
True yes/no according to gold: 95
Auxiliary-initial + 'or' candidates: 11
Potentially misclassified by old rule: 11
Potentially misclassified AND majority answer is Yes/No: 11

Question-start patterns:
Counter({'was': 11})

AUXILIARY + OR CANDIDATES

1. QUESTION:
Was Jorge Ledezma or Yuliya Baraley born first?
GOLD:
Jorge Ledezma
MAJORITY:
Yes
LOOP FINAL:
Jorge Ledezma was born first
FINAL SAMPLES:
['Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes']
GOLD IS YES/NO: False
MAJORITY IS YES/NO: True
--------------------------------------------------------------------------------

2. QUESTION:
Was Şemsettin Baş or Gwenc'Hlan Le Scouëzec born first?
GOLD:
Gwenc'Hlan Le Scouëzec
MAJORITY:
No
LOOP FINAL:
Gwenc'Hlan Le Scouëzec was born first
FINAL SAMPLES:
['No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No']
GOLD IS YES/NO: False
MAJORITY IS YES/NO: True
---

In [ ]:
import os
import re
import json
import copy
import random

import numpy as np
import torch

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
)

from sentence_transformers import SentenceTransformer



DATA_DIR = "/content/drive/MyDrive/hedge_run"


INPUT_JSON = (
    f"{DATA_DIR}/"
    "2wiki_passive_iterative_v2_1000.json"
)


OUTPUT_JSON = (
    f"{DATA_DIR}/"
    "2wiki_passive_iterative_v2_1000_YN_PATCHED.json"
)


PATCH_DETAILS_JSON = (
    f"{DATA_DIR}/"
    "2wiki_passive_iterative_v2_yn_patch_details.json"
)


MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

EMBED_ID = "BAAI/bge-small-en-v1.5"


FINAL_SAMPLES = 10

FINAL_TEMPERATURE = 0.7

FINAL_TOP_P = 0.95

FINAL_MAX_NEW_TOKENS = 80

ANSWER_MERGE_SIM = 0.80

EMPTY_FINAL_RETRY_ATTEMPTS = 2

SEED = 42


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)



with open(
    INPUT_JSON,
    "r",
    encoding="utf-8",
) as f:

    rows = json.load(f)


assert len(rows) == 1000


print("Loaded:", len(rows))

print(
    "dev:",
    sum(r["split"] == "dev" for r in rows)
)

print(
    "test:",
    sum(r["split"] == "test" for r in rows)
)



AUX_STARTS = (
    "is ",
    "are ",
    "was ",
    "were ",
    "do ",
    "does ",
    "did ",
    "can ",
    "could ",
    "would ",
    "will ",
    "has ",
    "have ",
    "had ",
)


def old_is_yes_no_question(question):

    q = str(question).strip().lower()

    return q.startswith(
        AUX_STARTS
    )


def is_yes_no_question(question):

    q = str(question).strip().lower()

    if not q.startswith(
        AUX_STARTS
    ):
        return False

    # Alternative / choice question.
    if re.search(
        r"\bor\b",
        q,
    ):
        return False

    return True



patch_rows = [

    row

    for row in rows

    if (
        old_is_yes_no_question(
            row["question"]
        )

        and

        not is_yes_no_question(
            row["question"]
        )
    )
]


print()
print("=" * 80)
print("2WIKI PATCH SET")
print("=" * 80)


print(
    "Questions requiring patch:",
    len(patch_rows)
)


for i, row in enumerate(
    patch_rows,
    start=1,
):

    print()

    print(
        f"{i}. {row['question']}"
    )

    print(
        "   old majority:",
        row["majority_answer"]
    )


# We manually inspected the diagnostic and expect exactly 11.

if len(patch_rows) != 11:

    raise RuntimeError(
        "Expected exactly 11 alternative questions, "
        f"but found {len(patch_rows)}."
    )


PATCH_IDS = {

    int(
        row["question_index"]
    )

    for row in patch_rows
}


print()
print("Confirmed patch size = 11.")

print(
    "Patch IDs:",
    sorted(PATCH_IDS)
)



if not torch.cuda.is_available():

    raise RuntimeError(
        "GPU runtime required."
    )


print()
print("=" * 80)
print("LOADING QWEN")
print("=" * 80)


compute_dtype = (

    torch.bfloat16

    if torch.cuda.is_bf16_supported()

    else torch.float16
)


bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=compute_dtype,

    bnb_4bit_use_double_quant=True,
)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
)


model.eval()


MODEL_DEVICE = next(
    model.parameters()
).device


print(
    "Qwen device:",
    MODEL_DEVICE
)


print(
    "\nLoading answer-clustering embedder..."
)


embedder = SentenceTransformer(
    EMBED_ID,
    device="cpu",
)



SYSTEM_PROMPT = """
You answer multi-hop questions by reasoning ONE unit at a time.

On each turn, output EXACTLY ONE of the following and nothing else:

<step>
one declarative factual statement
</step>

or

<final>
short final answer
</final>

Rules for <step>:
- It must be a declarative factual claim, never a question.
- One fact per step.
- Keep it short and self-contained.
- Build on the reasoning so far.
- Do not repeat previous steps.
- Emit <final> only when the previous steps are enough to answer the question.
""".strip()


def apply_chat_template(user_content):

    messages = [

        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },

        {
            "role": "user",
            "content": user_content.strip(),
        },
    ]


    return tokenizer.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True,
    )



def format_history(history_steps):

    if not history_steps:
        return "(none yet)"


    return "\n".join(

        f"<step>\n"
        f"{str(step).strip()}\n"
        f"</step>"

        for step in history_steps
    )



def format_evidence(evidence):

    evidence = [

        str(x).strip()

        for x in evidence

        if str(x).strip()
    ]


    if not evidence:
        return "(none)"


    return "\n".join(

        f"E{i}: {sentence}"

        for i, sentence in enumerate(
            evidence,
            start=1,
        )
    )


def build_final_prompt(
    question,
    evidence,
    history_steps,
):

    if is_yes_no_question(
        question
    ):

        answer_rule = (
            "The final answer must be exactly Yes or No."
        )

    else:

        answer_rule = (
            "Answer with the specific entity, person, place, "
            "title, number, category, alternative, comparison "
            "direction, or phrase requested by the question. "
            "Do not answer Yes or No unless the question "
            "genuinely asks for a yes/no judgement."
        )


    user_content = f"""
Evidence:
{format_evidence(evidence)}

Question:
{question}

Reasoning so far:
{format_history(history_steps)}

Give the final answer only.

Use exactly:
<final>
short final answer
</final>

Rules:
- {answer_rule}
- If the question presents alternatives using "or", answer the requested alternative rather than Yes or No.
- If the question asks which of two people was born first, answer that person's name.
- Do not explain.
- Do not leave the answer blank.
""".strip()


    return apply_chat_template(
        user_content
    )


def clean_text(text):

    text = str(text).strip()

    text = re.sub(
        r"</?(step|final)>",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


def parse_unit(text):

    raw = str(text)



    match = re.search(
        r"<final>\s*(.*?)\s*</final>",
        raw,
        flags=(
            re.IGNORECASE
            |
            re.DOTALL
        ),
    )


    if match:

        return {
            "type": "final",
            "text": clean_text(
                match.group(1)
            ),
            "raw": raw,
        }



    match = re.search(
        r"<final>\s*(.*)",
        raw,
        flags=(
            re.IGNORECASE
            |
            re.DOTALL
        ),
    )


    if match:

        content = re.split(
            r"</final>|<step>|</step>",
            match.group(1),
            flags=re.IGNORECASE,
        )[0]


        return {
            "type": "final",
            "text": clean_text(
                content
            ),
            "raw": raw,
        }



    return {
        "type": "final",
        "text": clean_text(raw),
        "raw": raw,
    }


def repair_empty_answer(unit):

    answer = str(
        unit.get(
            "text",
            "",
        )
    ).strip()


    if answer:
        return answer


    return clean_text(
        unit.get(
            "raw",
            "",
        )
    )


class StopOnStrings(
    StoppingCriteria
):

    def __init__(
        self,
        tokenizer,
        start_length,
        stop_strings,
    ):

        super().__init__()

        self.tokenizer = tokenizer

        self.start_length = start_length

        self.stop_strings = stop_strings


    def __call__(
        self,
        input_ids,
        scores,
        **kwargs,
    ):

        for row in input_ids:

            generated = (
                self.tokenizer.decode(
                    row[
                        self.start_length:
                    ],
                    skip_special_tokens=True,
                )
            )


            if not any(
                stop in generated
                for stop in self.stop_strings
            ):

                return False


        return True


@torch.no_grad()
def generate_units(
    prompt,
    n_samples,
):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    )


    inputs = {

        key: value.to(
            MODEL_DEVICE
        )

        for key, value
        in inputs.items()
    }


    start_length = (
        inputs[
            "input_ids"
        ]
        .shape[-1]
    )


    stopping = StoppingCriteriaList(
        [
            StopOnStrings(
                tokenizer,
                start_length,
                ["</final>"],
            )
        ]
    )


    outputs = model.generate(

        **inputs,

        do_sample=True,

        temperature=FINAL_TEMPERATURE,

        top_p=FINAL_TOP_P,

        max_new_tokens=FINAL_MAX_NEW_TOKENS,

        num_return_sequences=n_samples,

        stopping_criteria=stopping,

        pad_token_id=tokenizer.eos_token_id,
    )


    decoded = tokenizer.batch_decode(

        outputs[
            :,
            start_length:
        ],

        skip_special_tokens=True,
    )


    del inputs
    del outputs

    torch.cuda.empty_cache()


    return [
        parse_unit(x)
        for x in decoded
    ]



def generate_final_samples(
    question,
    evidence,
    history_steps,
):

    prompt = build_final_prompt(
        question,
        evidence,
        history_steps,
    )


    answers = []


    for _ in range(
        EMPTY_FINAL_RETRY_ATTEMPTS + 1
    ):

        remaining = (
            FINAL_SAMPLES
            -
            len(answers)
        )


        if remaining <= 0:
            break


        units = generate_units(
            prompt,
            remaining,
        )


        for unit in units:

            answer = (
                repair_empty_answer(
                    unit
                )
                .strip()
            )


            if answer:

                answers.append(
                    answer
                )


    return answers[
        :FINAL_SAMPLES
    ]



def normalize_answer(text):

    text = (
        str(text)
        .lower()
        .strip()
    )


    text = re.sub(
        r"\b(the|a|an)\b",
        " ",
        text,
    )


    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text,
    )


    text = re.sub(
        r"\s+",
        " ",
        text,
    )


    return text.strip()



def cluster_answers(answers):

    clean = [

        str(answer).strip()

        for answer in answers

        if str(answer).strip()
    ]


    if not clean:
        return []


    norms = [
        normalize_answer(answer)
        for answer in clean
    ]


    embeddings = embedder.encode(

        clean,

        normalize_embeddings=True,

        convert_to_numpy=True,

        show_progress_bar=False,
    )


    clusters = []


    for i, norm_i in enumerate(
        norms
    ):

        placed = False


        for cluster in clusters:

            norm_j = cluster["norm"]


            same = (
                norm_i == norm_j
            )


            if (
                not same
                and norm_i
                and norm_j
            ):

                same = (
                    norm_i in norm_j
                    or
                    norm_j in norm_i
                )


            if not same:

                similarity = float(
                    np.dot(
                        embeddings[i],
                        embeddings[
                            cluster["idx0"]
                        ],
                    )
                )


                same = (
                    similarity
                    >=
                    ANSWER_MERGE_SIM
                )


            if same:

                cluster["idxs"].append(
                    i
                )

                placed = True

                break


        if not placed:

            clusters.append(
                {
                    "norm": norm_i,
                    "idx0": i,
                    "idxs": [i],
                }
            )


    return clusters


def shannon_entropy(counts):

    counts = [
        int(x)
        for x in counts
        if x > 0
    ]


    if len(counts) <= 1:

        return (
            0.0,
            0.0,
        )


    probs = (
        np.asarray(
            counts,
            dtype=float,
        )
        /
        sum(counts)
    )


    entropy = float(
        -np.sum(
            probs
            *
            np.log2(
                probs + 1e-12
            )
        )
    )


    norm_entropy = float(
        entropy
        /
        np.log2(
            len(counts)
        )
    )


    return (
        entropy,
        norm_entropy,
    )


def self_consistency_stats(
    answers,
):

    answers = [

        str(answer).strip()

        for answer in answers

        if str(answer).strip()
    ]


    if not answers:

        return {
            "majority_answer": "",
            "self_consistency": 0.0,
            "norm_entropy": 1.0,
        }


    clusters = cluster_answers(
        answers
    )


    sizes = [
        len(cluster["idxs"])
        for cluster in clusters
    ]


    biggest = max(
        clusters,
        key=lambda c:
            len(c["idxs"]),
    )


    majority_answer = (
        answers[
            biggest["idxs"][0]
        ]
    )


    _, norm_entropy = (
        shannon_entropy(
            sizes
        )
    )


    return {

        "majority_answer":
            majority_answer,

        "self_consistency":
            float(
                max(sizes)
                /
                len(answers)
            ),

        "norm_entropy":
            float(
                norm_entropy
            ),
    }



@torch.no_grad()
def answer_logprob(
    question,
    evidence,
    history_steps,
    answer,
):

    answer = str(
        answer
    ).strip()


    if not answer:

        return float("nan")


    prompt = build_final_prompt(
        question,
        evidence,
        history_steps,
    )


    prefix_text = (
        prompt
        +
        "<final>\n"
    )


    prefix_ids = tokenizer(
        prefix_text,
        add_special_tokens=False,
    )["input_ids"]


    answer_ids = tokenizer(
        answer,
        add_special_tokens=False,
    )["input_ids"]


    if not answer_ids:

        return float("nan")


    all_ids = (
        prefix_ids
        +
        answer_ids
    )


    input_ids = torch.tensor(
        [all_ids],
        dtype=torch.long,
        device=MODEL_DEVICE,
    )


    attention_mask = torch.ones_like(
        input_ids
    )


    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )


    logits = (
        outputs
        .logits[0]
        .float()
    )


    start = (
        len(prefix_ids)
        -
        1
    )


    end = (
        start
        +
        len(answer_ids)
    )


    answer_logits = logits[
        start:end
    ]


    targets = torch.tensor(
        answer_ids,
        dtype=torch.long,
        device=MODEL_DEVICE,
    )


    log_probs = torch.log_softmax(
        answer_logits,
        dim=-1,
    )


    token_log_probs = log_probs[
        torch.arange(
            len(answer_ids),
            device=MODEL_DEVICE,
        ),
        targets,
    ]


    result = float(
        token_log_probs
        .mean()
        .item()
    )


    return result



patched_rows = copy.deepcopy(
    rows
)


patch_details = {}


print()
print("=" * 80)
print("REGENERATING 11 FINAL-ANSWER SETS")
print("=" * 80)



for row in tqdm(
    patched_rows
):

    qi = int(
        row["question_index"]
    )


    if qi not in PATCH_IDS:
        continue



    question_seed = (
        SEED + qi
    )


    random.seed(
        question_seed
    )

    np.random.seed(
        question_seed
    )

    torch.manual_seed(
        question_seed
    )

    torch.cuda.manual_seed_all(
        question_seed
    )


    question = str(
        row["question"]
    )



    evidence = copy.deepcopy(
        row.get(
            "question_evidence",
            [],
        )
    )


    history = copy.deepcopy(
        row.get(
            "selected_steps",
            [],
        )
    )


    old_values = {

        "final_answers":
            copy.deepcopy(
                row.get(
                    "final_answers",
                    [],
                )
            ),

        "majority_answer":
            row.get(
                "majority_answer",
                "",
            ),

        "self_consistency":
            row.get(
                "self_consistency"
            ),

        "answer_norm_entropy":
            row.get(
                "answer_norm_entropy"
            ),

        "answer_logprob":
            row.get(
                "answer_logprob"
            ),
    }



    final_answers = (
        generate_final_samples(
            question,
            evidence,
            history,
        )
    )


    if len(final_answers) != 10:

        raise RuntimeError(
            f"qi={qi}: expected 10 final samples, "
            f"got {len(final_answers)}"
        )


    stats = self_consistency_stats(
        final_answers
    )


    majority_answer = (
        stats[
            "majority_answer"
        ]
    )


    logprob = answer_logprob(

        question,

        evidence,

        history,

        majority_answer,
    )


    if not np.isfinite(
        logprob
    ):

        raise RuntimeError(
            f"Invalid answer logprob qi={qi}"
        )


    row[
        "final_answers"
    ] = final_answers


    row[
        "majority_answer"
    ] = majority_answer


    row[
        "self_consistency"
    ] = float(
        stats[
            "self_consistency"
        ]
    )


    row[
        "answer_norm_entropy"
    ] = float(
        stats[
            "norm_entropy"
        ]
    )


    row[
        "answer_logprob"
    ] = float(
        logprob
    )


    row[
        "final_answer_question_type_patch"
    ] = {

        "applied":
            True,

        "reason":
            "auxiliary_initial_alternative_question",

        "old_question_type":
            "yes_no",

        "corrected_question_type":
            "alternative_or_choice",

        "reasoning_regenerated":
            False,

        "evidence_changed":
            False,

        "loop_final_changed":
            False,

        "seed":
            int(question_seed),
    }


    patch_details[
        str(qi)
    ] = {

        "question_index":
            qi,

        "question":
            question,

        # Diagnostic only.
        "gold_answer_diagnostic_only":
            row.get(
                "gold_answer",
                "",
            ),

        "old":
            old_values,

        "new": {

            "final_answers":
                copy.deepcopy(
                    final_answers
                ),

            "majority_answer":
                majority_answer,

            "self_consistency":
                float(
                    stats[
                        "self_consistency"
                    ]
                ),

            "answer_norm_entropy":
                float(
                    stats[
                        "norm_entropy"
                    ]
                ),

            "answer_logprob":
                float(
                    logprob
                ),
        },
    }



assert len(
    patch_details
) == 11



for old_row, new_row in zip(
    rows,
    patched_rows,
):

    qi = int(
        old_row["question_index"]
    )


    assert (
        qi
        ==
        int(
            new_row["question_index"]
        )
    )


    if qi not in PATCH_IDS:

        if old_row != new_row:

            raise RuntimeError(
                "Unaffected record changed: "
                f"qi={qi}"
            )


print()
print(
    "Confirmed: all 989 unaffected records are unchanged."
)


for old_row, new_row in zip(
    rows,
    patched_rows,
):

    qi = int(
        old_row["question_index"]
    )


    if qi not in PATCH_IDS:
        continue


    assert (
        old_row.get(
            "selected_steps",
            [],
        )
        ==
        new_row.get(
            "selected_steps",
            [],
        )
    )


    assert (
        old_row.get(
            "question_evidence",
            [],
        )
        ==
        new_row.get(
            "question_evidence",
            [],
        )
    )


    assert (
        old_row.get(
            "loop_final_answer",
            "",
        )
        ==
        new_row.get(
            "loop_final_answer",
            "",
        )
    )


print(
    "Confirmed: reasoning/evidence/loop final remain "
    "frozen for all 11 patched questions."
)




def simple_norm(text):

    return (
        str(text)
        .strip()
        .lower()
        .rstrip(".")
    )


still_yes_no = []


for row in patched_rows:

    qi = int(
        row["question_index"]
    )


    if qi not in PATCH_IDS:
        continue


    if (
        simple_norm(
            row[
                "majority_answer"
            ]
        )
        in {
            "yes",
            "no",
        }
    ):

        still_yes_no.append(
            qi
        )


print()
print(
    "Patched majority answers still Yes/No:",
    len(still_yes_no)
)


if still_yes_no:

    print(
        "IDs:",
        still_yes_no
    )



with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        patched_rows,
        f,
        indent=2,
        ensure_ascii=False,
    )


with open(
    PATCH_DETAILS_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        patch_details,
        f,
        indent=2,
        ensure_ascii=False,
    )



print()
print("=" * 80)
print("2WIKI PATCH RESULTS")
print("=" * 80)


for item in sorted(
    patch_details.values(),
    key=lambda x:
        x[
            "question_index"
        ],
):

    print()

    print(
        "QUESTION:"
    )

    print(
        item[
            "question"
        ]
    )


    print(
        "GOLD [DIAGNOSTIC ONLY]:"
    )

    print(
        item[
            "gold_answer_diagnostic_only"
        ]
    )


    print(
        "OLD MAJORITY:"
    )

    print(
        item[
            "old"
        ][
            "majority_answer"
        ]
    )


    print(
        "NEW MAJORITY:"
    )

    print(
        item[
            "new"
        ][
            "majority_answer"
        ]
    )


    print(
        "NEW FINAL SAMPLES:"
    )

    print(
        item[
            "new"
        ][
            "final_answers"
        ]
    )


    print(
        "OLD SC:",
        item[
            "old"
        ][
            "self_consistency"
        ]
    )


    print(
        "NEW SC:",
        item[
            "new"
        ][
            "self_consistency"
        ]
    )


    print(
        "-" * 80
    )



print()
print("=" * 80)
print("2WIKI PATCH SUMMARY")
print("=" * 80)


print(
    "Total:",
    len(patched_rows)
)

print(
    "Patched:",
    len(PATCH_IDS)
)

print(
    "Untouched:",
    len(patched_rows)
    -
    len(PATCH_IDS)
)

print(
    "Patched majority answers still Yes/No:",
    len(still_yes_no)
)


print()
print(
    "Saved patched generation:"
)

print(
    OUTPUT_JSON
)


print()
print(
    "Saved patch details:"
)

print(
    PATCH_DETAILS_JSON
)


print()
print(
    "DONE."
)

Loaded: 1000
dev: 800
test: 200

2WIKI PATCH SET
Questions requiring patch: 11

1. Was Jorge Ledezma or Yuliya Baraley born first?
   old majority: Yes

2. Was Şemsettin Baş or Gwenc'Hlan Le Scouëzec born first?
   old majority: No

3. Was Joe Ollmann or Abdón Cifuentes born first?
   old majority: No

4. Was Frank Kühne or Subhash Bapurao Wankhede born first?
   old majority: Yes

5. Was Paul Antoine Brunel or George Markham Giffard born first?
   old majority: No

6. Was Angus Wagner or Juan Carlos Falcón born first?
   old majority: No

7. Was Rebecca West or Philippe Lefebure born first?
   old majority: Yes

8. Was Demetrious Cox or István Görgényi born first?
   old majority: Yes

9. Was Wayne Alan Harold or Aldo Buzzi born first?
   old majority: Yes

10. Was Denisa Křížová or Danny Halligan born first?
   old majority: No

11. Was Napoleão Laureano or Giovanni Vastola born first?
   old majority: No

Confirmed patch size = 11.
Patch IDs: [31, 1322, 4447, 4638, 5255, 5735, 6014,

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen device: cuda:0

Loading answer-clustering embedder...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


REGENERATING 11 FINAL-ANSWER SETS


  0%|          | 0/1000 [00:00<?, ?it/s]


Confirmed: all 989 unaffected records are unchanged.
Confirmed: reasoning/evidence/loop final remain frozen for all 11 patched questions.

Patched majority answers still Yes/No: 0

2WIKI PATCH RESULTS

QUESTION:
Was Şemsettin Baş or Gwenc'Hlan Le Scouëzec born first?
GOLD [DIAGNOSTIC ONLY]:
Gwenc'Hlan Le Scouëzec
OLD MAJORITY:
No
NEW MAJORITY:
Gwenc'Hlan Le Scouëzec
NEW FINAL SAMPLES:
["Gwenc'Hlan Le Scouëzec", "Gwenc'Hlan Le Scouëzec", "Gwenc'Hlan Le Scouëzec", "Gwenc'Hlan Le Scouëzec", "Gwenc'Hlan Le Scouëzec", "Gwenc'Hlan Le Scouëzec", "Gwenc'Hlan Le Scouëzec", "Gwenc'Hlan Le Scouëzec", "Gwenc'Hlan Le Scouëzec", "Gwenc'Hlan Le Scouëzec"]
OLD SC: 1.0
NEW SC: 1.0
--------------------------------------------------------------------------------

QUESTION:
Was Joe Ollmann or Abdón Cifuentes born first?
GOLD [DIAGNOSTIC ONLY]:
Abdón Cifuentes
OLD MAJORITY:
No
NEW MAJORITY:
Abdón Cifuentes
NEW FINAL SAMPLES:
['Abdón Cifuentes', 'Abdón Cifuentes', 'Abdón Cifuentes', 'Abdón Cifuentes', 'Abd

In [ ]:
import json
from pprint import pprint

D = "/content/drive/MyDrive/hedge_run"

PATCHED = f"{D}/2wiki_passive_iterative_v2_1000_YN_PATCHED.json"
FULL    = f"{D}/2wiki_full.json"

patched = json.load(open(PATCHED, encoding="utf-8"))
full    = json.load(open(FULL, encoding="utf-8"))

print("PATCHED TYPE:", type(patched))
print("PATCHED N:", len(patched))

print("\nPATCHED KEYS:")
print(patched[0].keys())

print("\nPATCHED FIRST RECORD:")
pprint(patched[0])

print("\n" + "="*80)

print("FULL TYPE:", type(full))
print("FULL N:", len(full) if hasattr(full, "__len__") else "N/A")

if isinstance(full, list):
    print("\nFULL KEYS:")
    print(full[0].keys())

    print("\nFULL FIRST RECORD:")
    pprint(full[0])

elif isinstance(full, dict):
    print("\nFULL TOP-LEVEL KEYS:")
    print(list(full.keys())[:30])

    first_key = next(iter(full))
    print("\nFIRST KEY:", first_key)
    print("FIRST VALUE TYPE:", type(full[first_key]))

    pprint(full[first_key])

PATCHED TYPE: <class 'list'>
PATCHED N: 1000

PATCHED KEYS:
dict_keys(['question_index', 'split', 'question', 'gold_answer', 'question_evidence', 'selected_steps', 'num_steps', 'stop_reason', 'loop_final_answer', 'generation_trace', 'final_answers', 'majority_answer', 'self_consistency', 'answer_norm_entropy', 'answer_logprob', 'num_answer_clusters', 'num_valid_final_samples', 'generation_protocol'])

PATCHED FIRST RECORD:
{'answer_logprob': -1.3033403774898034e-05,
 'answer_norm_entropy': 0.0,
 'final_answers': ['Max and Helen',
                   'Max and Helen',
                   'Max and Helen',
                   'Max and Helen',
                   'Max and Helen',
                   'Max and Helen',
                   'Max and Helen',
                   'Max and Helen',
                   'Max and Helen',
                   'Max and Helen'],
 'generation_protocol': {'candidate_selection': False,
                         'final_samples': 10,
                         'final_temper

In [ ]:
import os
import re
import json
import copy
import math

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

from rank_bm25 import BM25Okapi

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)


DATA_DIR = "/content/drive/MyDrive/hedge_run"


PATCHED_JSON = (
    f"{DATA_DIR}/"
    "2wiki_passive_iterative_v2_1000_YN_PATCHED.json"
)


FULL_JSON = (
    f"{DATA_DIR}/"
    "2wiki_full.json"
)


PROGRESS_JSON = (
    f"{DATA_DIR}/"
    "2wiki_passive_iterative_v2_step_verification_progress.json"
)


OUTPUT_JSON = (
    f"{DATA_DIR}/"
    "2wiki_passive_iterative_v2_verified_1000.json"
)


OUTPUT_CSV = (
    f"{DATA_DIR}/"
    "2wiki_passive_iterative_v2_features14_REASONING_CONF.csv"
)


NLI_MODEL_ID = (
    "MoritzLaurer/"
    "DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
)


STEP_RETRIEVE_K = 10

SUPPORT_THRESHOLD = 0.50

CONTRADICTION_THRESHOLD = 0.50

NLI_MAX_LENGTH = 512

NLI_BATCH_SIZE = 32

SAVE_EVERY = 10


REASONING_FEATURES = [

    "max_contradiction_score",

    "any_contradicted",

    "min_support_score",

    "num_steps",

    "frac_supported",

    "frac_contradicted",

    "frac_unclear",

    "mean_support",

    "mean_contradiction",

    "conflict",

    "support_spread",
]


CONFIDENCE_FEATURES = [

    "self_consistency",

    "answer_norm_entropy",

    "answer_logprob",
]


FEATURE_COLUMNS = (
    REASONING_FEATURES
    +
    CONFIDENCE_FEATURES
)


assert len(REASONING_FEATURES) == 11
assert len(CONFIDENCE_FEATURES) == 3
assert len(FEATURE_COLUMNS) == 14


with open(
    PATCHED_JSON,
    "r",
    encoding="utf-8",
) as f:

    rows = json.load(f)


with open(
    FULL_JSON,
    "r",
    encoding="utf-8",
) as f:

    full_rows = json.load(f)


assert len(rows) == 1000
assert len(full_rows) == 1000


print(
    "Patched generation rows:",
    len(rows)
)


print(
    "Full-context rows:",
    len(full_rows)
)


print(
    "dev:",
    sum(
        row["split"] == "dev"
        for row in rows
    )
)


print(
    "test:",
    sum(
        row["split"] == "test"
        for row in rows
    )
)



full_by_qi = {

    int(row["question_index"]):
        row

    for row in full_rows
}


assert len(full_by_qi) == 1000


for row in rows:

    qi = int(
        row["question_index"]
    )


    if qi not in full_by_qi:

        raise RuntimeError(
            f"Missing full context for qi={qi}"
        )


    full_row = (
        full_by_qi[
            qi
        ]
    )


    if (
        str(row["question"]).strip()
        !=
        str(full_row["question"]).strip()
    ):

        raise RuntimeError(
            f"Question mismatch qi={qi}"
        )


print(
    "Confirmed: all 1000 generated records "
    "match full-context records."
)



def flatten_context(
    full_row,
):

    context = (
        full_row[
            "context"
        ]
    )


    nested_sentences = (
        context[
            "sentences"
        ]
    )


    flat = []


    for paragraph in nested_sentences:

        for sentence in paragraph:

            sentence = str(
                sentence
            ).strip()


            if sentence:

                flat.append(
                    sentence
                )


    return flat



context_lengths = []


for row in rows:

    qi = int(
        row["question_index"]
    )


    sentences = flatten_context(
        full_by_qi[
            qi
        ]
    )


    if not sentences:

        raise RuntimeError(
            f"No context sentences qi={qi}"
        )


    context_lengths.append(
        len(sentences)
    )


print()
print(
    "Mean context sentences:",
    round(
        float(
            np.mean(
                context_lengths
            )
        ),
        2,
    )
)


print(
    "Min context sentences:",
    min(
        context_lengths
    )
)


print(
    "Max context sentences:",
    max(
        context_lengths
    )
)



def bm25_tokenize(
    text,
):

    return re.findall(
        r"[A-Za-z0-9À-ÖØ-öø-ÿĀ-ž']+",
        str(text).lower(),
    )



def retrieve_for_step(
    step,
    context_sentences,
    k=STEP_RETRIEVE_K,
):

    if not context_sentences:

        return []


    tokenized_corpus = [

        bm25_tokenize(
            sentence
        )

        for sentence
        in context_sentences
    ]


    bm25 = BM25Okapi(
        tokenized_corpus
    )


    query_tokens = bm25_tokenize(
        step
    )


    scores = np.asarray(
        bm25.get_scores(
            query_tokens
        ),
        dtype=float,
    )



    order = np.argsort(
        -scores,
        kind="stable",
    )


    k_eff = min(
        k,
        len(
            context_sentences
        ),
    )


    top_indices = (
        order[
            :k_eff
        ]
    )


    return [

        {
            "rank":
                rank,

            "sentence_index":
                int(idx),

            "sentence":
                context_sentences[
                    int(idx)
                ],

            "bm25_score":
                float(
                    scores[
                        int(idx)
                    ]
                ),
        }

        for rank, idx
        in enumerate(
            top_indices,
            start=1,
        )
    ]



print()
print("=" * 80)
print("LOADING NLI VERIFIER")
print("=" * 80)


nli_tokenizer = (
    AutoTokenizer.from_pretrained(
        NLI_MODEL_ID
    )
)


if torch.cuda.is_available():

    DEVICE = torch.device(
        "cuda"
    )

    DTYPE = torch.float16

else:

    DEVICE = torch.device(
        "cpu"
    )

    DTYPE = torch.float32


try:

    nli_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            NLI_MODEL_ID,
            dtype=DTYPE,
        )
    )

except TypeError:

    nli_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            NLI_MODEL_ID,
            torch_dtype=DTYPE,
        )
    )


nli_model = nli_model.to(
    DEVICE
)

nli_model.eval()


id2label = {

    int(k):
        str(v).lower()

    for k, v
    in nli_model.config.id2label.items()
}


ENT_IDX = next(

    i

    for i, label
    in id2label.items()

    if "entail" in label
)


NEU_IDX = next(

    i

    for i, label
    in id2label.items()

    if "neutral" in label
)


CON_IDX = next(

    i

    for i, label
    in id2label.items()

    if "contrad" in label
)


print(
    "NLI device:",
    DEVICE
)


print(
    "NLI labels:",
    id2label
)


@torch.no_grad()
def score_nli_pairs(
    premises,
    hypotheses,
):

    all_probs = []


    for start in range(
        0,
        len(premises),
        NLI_BATCH_SIZE,
    ):

        batch_p = (
            premises[
                start:
                start
                +
                NLI_BATCH_SIZE
            ]
        )


        batch_h = (
            hypotheses[
                start:
                start
                +
                NLI_BATCH_SIZE
            ]
        )


        encoded = nli_tokenizer(

            batch_p,

            batch_h,

            padding=True,

            truncation="only_first",

            max_length=NLI_MAX_LENGTH,

            return_tensors="pt",
        )


        encoded = {

            key:
                value.to(
                    DEVICE
                )

            for key, value
            in encoded.items()
        }


        logits = (
            nli_model(
                **encoded
            )
            .logits
            .float()
        )


        probs = (
            torch.softmax(
                logits,
                dim=-1,
            )
            .cpu()
            .numpy()
        )


        all_probs.append(
            probs
        )


    return np.concatenate(
        all_probs,
        axis=0,
    )


def verify_step(
    step,
    context_sentences,
):

    retrieved = (
        retrieve_for_step(
            step,
            context_sentences,
            k=STEP_RETRIEVE_K,
        )
    )


    if not retrieved:

        return {

            "step":
                step,

            "retrieved_evidence":
                [],

            "support_score":
                0.0,

            "contradiction_score":
                0.0,

            "neutral_score":
                1.0,

            "label":
                "unclear",
        }


    premises = [

        item[
            "sentence"
        ]

        for item
        in retrieved
    ]


    hypotheses = [

        step

        for _ in retrieved
    ]


    probs = score_nli_pairs(
        premises,
        hypotheses,
    )




    for item, p in zip(
        retrieved,
        probs,
    ):

        item[
            "entailment"
        ] = float(
            p[
                ENT_IDX
            ]
        )


        item[
            "neutral"
        ] = float(
            p[
                NEU_IDX
            ]
        )


        item[
            "contradiction"
        ] = float(
            p[
                CON_IDX
            ]
        )


    support = float(
        max(
            item[
                "entailment"
            ]

            for item
            in retrieved
        )
    )


    contradiction = float(
        max(
            item[
                "contradiction"
            ]

            for item
            in retrieved
        )
    )


    neutral = float(
        max(
            item[
                "neutral"
            ]

            for item
            in retrieved
        )
    )


    if (
        support
        >=
        SUPPORT_THRESHOLD
    ):

        label = (
            "supported"
        )


    elif (
        contradiction
        >=
        CONTRADICTION_THRESHOLD
    ):

        label = (
            "contradicted"
        )


    else:

        label = (
            "unclear"
        )


    return {

        "step":
            step,

        "retrieved_evidence":
            retrieved,

        "support_score":
            support,

        "contradiction_score":
            contradiction,

        "neutral_score":
            neutral,

        "label":
            label,
    }


def compute_reasoning_features(
    step_reports,
):

    n = len(
        step_reports
    )



    if n == 0:

        return {

            "max_contradiction_score":
                0.0,

            "any_contradicted":
                0,

            "min_support_score":
                0.0,

            "num_steps":
                0,

            "frac_supported":
                0.0,

            "frac_contradicted":
                0.0,

            "frac_unclear":
                1.0,

            "mean_support":
                0.0,

            "mean_contradiction":
                0.0,

            "conflict":
                0.0,

            "support_spread":
                0.0,
        }


    support_scores = np.asarray(

        [
            report[
                "support_score"
            ]

            for report
            in step_reports
        ],

        dtype=float,
    )


    contradiction_scores = np.asarray(

        [
            report[
                "contradiction_score"
            ]

            for report
            in step_reports
        ],

        dtype=float,
    )


    labels = [

        report[
            "label"
        ]

        for report
        in step_reports
    ]


    n_supported = sum(
        label == "supported"
        for label in labels
    )


    n_contradicted = sum(
        label == "contradicted"
        for label in labels
    )


    n_unclear = sum(
        label == "unclear"
        for label in labels
    )


    max_support = float(
        support_scores.max()
    )


    min_support = float(
        support_scores.min()
    )


    max_contradiction = float(
        contradiction_scores.max()
    )


    return {

        "max_contradiction_score":
            max_contradiction,

        "any_contradicted":
            int(
                n_contradicted
                >
                0
            ),

        "min_support_score":
            min_support,

        "num_steps":
            int(
                n
            ),

        "frac_supported":
            float(
                n_supported
                /
                n
            ),

        "frac_contradicted":
            float(
                n_contradicted
                /
                n
            ),

        "frac_unclear":
            float(
                n_unclear
                /
                n
            ),

        "mean_support":
            float(
                support_scores.mean()
            ),

        "mean_contradiction":
            float(
                contradiction_scores.mean()
            ),

        "conflict":
            float(
                max_support
                *
                max_contradiction
            ),

        "support_spread":
            float(
                max_support
                -
                min_support
            ),
    }



if os.path.exists(
    PROGRESS_JSON
):

    with open(
        PROGRESS_JSON,
        "r",
        encoding="utf-8",
    ) as f:

        progress = json.load(f)

else:

    progress = {}


print()
print(
    "Already verified:",
    len(
        progress
    )
)



print()
print("=" * 80)
print("2WIKI POST-HOC STEP VERIFICATION")
print("=" * 80)


for row in tqdm(
    rows
):

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    if qi in progress:

        continue


    full_row = (
        full_by_qi[
            int(qi)
        ]
    )


    context_sentences = (
        flatten_context(
            full_row
        )
    )


    selected_steps = [

        str(step).strip()

        for step
        in row.get(
            "selected_steps",
            []
        )

        if str(step).strip()
    ]


    step_reports = []


    for step_index, step in enumerate(
        selected_steps,
        start=1,
    ):

        report = verify_step(
            step,
            context_sentences,
        )


        report[
            "step_index"
        ] = int(
            step_index
        )


        step_reports.append(
            report
        )


    reasoning_features = (
        compute_reasoning_features(
            step_reports
        )
    )



    features = {

        **reasoning_features,

        "self_consistency":
            float(
                row[
                    "self_consistency"
                ]
            ),

        "answer_norm_entropy":
            float(
                row[
                    "answer_norm_entropy"
                ]
            ),

        "answer_logprob":
            float(
                row[
                    "answer_logprob"
                ]
            ),
    }


    progress[
        qi
    ] = {

        "question_index":
            int(
                qi
            ),

        "question":
            row[
                "question"
            ],

        "selected_steps":
            copy.deepcopy(
                selected_steps
            ),

        "step_reports":
            step_reports,

        "features":
            features,
    }


    if (
        len(
            progress
        )
        %
        SAVE_EVERY
        ==
        0
    ):

        with open(
            PROGRESS_JSON,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                progress,
                f,
                indent=2,
                ensure_ascii=False,
            )


        print(
            f"{len(progress)}/1000 saved"
        )


# Final progress save.

with open(
    PROGRESS_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        progress,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert len(
    progress
) == 1000



verified_rows = []


for row in rows:

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    verification = (
        progress[
            qi
        ]
    )


    new_row = copy.deepcopy(
        row
    )


    new_row[
        "step_reports"
    ] = copy.deepcopy(
        verification[
            "step_reports"
        ]
    )


    new_row[
        "features"
    ] = copy.deepcopy(
        verification[
            "features"
        ]
    )


    new_row[
        "verification_protocol"
    ] = {

        "stage":
            "post_hoc_reasoning_verification",

        "step_query":
            "reasoning_step",

        "retrieval":
            "BM25",

        "retrieval_source":
            "all_sentences_in_question_context",

        "retrieve_k":
            STEP_RETRIEVE_K,

        "nli_model":
            NLI_MODEL_ID,

        "nli_premise":
            "retrieved_evidence_sentence",

        "nli_hypothesis":
            "generated_reasoning_step",

        "support_aggregation":
            "max_entailment_over_top_k",

        "contradiction_aggregation":
            "max_contradiction_over_top_k",

        "support_threshold":
            SUPPORT_THRESHOLD,

        "contradiction_threshold":
            CONTRADICTION_THRESHOLD,

        "verifier_used_during_generation":
            False,

        "gold_used":
            False,
    }


    verified_rows.append(
        new_row
    )


assert len(
    verified_rows
) == 1000



for old_row, new_row in zip(
    rows,
    verified_rows,
):

    assert (
        old_row[
            "question_index"
        ]
        ==
        new_row[
            "question_index"
        ]
    )


    assert (
        old_row[
            "selected_steps"
        ]
        ==
        new_row[
            "selected_steps"
        ]
    )


    assert (
        old_row[
            "majority_answer"
        ]
        ==
        new_row[
            "majority_answer"
        ]
    )


    assert (
        old_row[
            "final_answers"
        ]
        ==
        new_row[
            "final_answers"
        ]
    )


    assert (
        old_row[
            "question_evidence"
        ]
        ==
        new_row[
            "question_evidence"
        ]
    )


print()
print(
    "Confirmed: generation outputs remain unchanged."
)


for row in verified_rows:

    values = [

        float(
            row[
                "features"
            ][
                feature
            ]
        )

        for feature
        in FEATURE_COLUMNS
    ]


    if not np.all(
        np.isfinite(
            values
        )
    ):

        raise RuntimeError(
            "Non-finite feature found "
            f"qi={row['question_index']}"
        )


print(
    "All 14 current features finite: YES"
)



with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        verified_rows,
        f,
        indent=2,
        ensure_ascii=False,
    )




csv_rows = []


for row in verified_rows:

    csv_rows.append(
        {

            "question_index":
                row[
                    "question_index"
                ],

            "split":
                row[
                    "split"
                ],

            "question":
                row[
                    "question"
                ],

            "gold_answer":
                row[
                    "gold_answer"
                ],

            "majority_answer":
                row[
                    "majority_answer"
                ],

            **row[
                "features"
            ],
        }
    )


df = pd.DataFrame(
    csv_rows
)


df.to_csv(
    OUTPUT_CSV,
    index=False,
)



all_reports = [

    report

    for row in verified_rows

    for report
    in row[
        "step_reports"
    ]
]


n_steps = len(
    all_reports
)


n_supported = sum(

    report[
        "label"
    ]
    ==
    "supported"

    for report
    in all_reports
)


n_contradicted = sum(

    report[
        "label"
    ]
    ==
    "contradicted"

    for report
    in all_reports
)


n_unclear = sum(

    report[
        "label"
    ]
    ==
    "unclear"

    for report
    in all_reports
)


questions_with_contradiction = sum(

    int(
        row[
            "features"
        ][
            "any_contradicted"
        ]
    )

    for row
    in verified_rows
)


zero_step = sum(

    int(
        row[
            "features"
        ][
            "num_steps"
        ]
        ==
        0
    )

    for row
    in verified_rows
)


mean_support = (

    float(
        np.mean(
            [
                report[
                    "support_score"
                ]

                for report
                in all_reports
            ]
        )
    )

    if all_reports

    else 0.0
)


mean_contradiction = (

    float(
        np.mean(
            [
                report[
                    "contradiction_score"
                ]

                for report
                in all_reports
            ]
        )
    )

    if all_reports

    else 0.0
)


print()
print("=" * 80)
print("2WIKI STEP VERIFICATION SUMMARY")
print("=" * 80)


print(
    "Rows:",
    len(
        verified_rows
    )
)


print(
    "Total retained reasoning steps:",
    n_steps
)


print(
    "Supported steps:",
    n_supported
)


print(
    "Unclear steps:",
    n_unclear
)


print(
    "Contradicted steps:",
    n_contradicted
)


print(
    "Questions with >=1 contradicted step:",
    questions_with_contradiction
)


print(
    "Zero-step trajectories:",
    zero_step
)


print(
    "Mean step support:",
    round(
        mean_support,
        4,
    )
)


print(
    "Mean step contradiction:",
    round(
        mean_contradiction,
        4,
    )
)


print()
print(
    "Saved verified JSON:"
)

print(
    OUTPUT_JSON
)


print()
print(
    "Saved 14-feature CSV:"
)

print(
    OUTPUT_CSV
)


print()
print(
    "DONE."
)

Patched generation rows: 1000
Full-context rows: 1000
dev: 800
test: 200
Confirmed: all 1000 generated records match full-context records.

Mean context sentences: 32.9
Min context sentences: 10
Max context sentences: 115

LOADING NLI VERIFIER


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

NLI device: cuda
NLI labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}

Already verified: 0

2WIKI POST-HOC STEP VERIFICATION


  0%|          | 0/1000 [00:00<?, ?it/s]

10/1000 saved
20/1000 saved
30/1000 saved
40/1000 saved
50/1000 saved
60/1000 saved
70/1000 saved
80/1000 saved
90/1000 saved
100/1000 saved
110/1000 saved
120/1000 saved
130/1000 saved
140/1000 saved
150/1000 saved
160/1000 saved
170/1000 saved
180/1000 saved
190/1000 saved
200/1000 saved
210/1000 saved
220/1000 saved
230/1000 saved
240/1000 saved
250/1000 saved
260/1000 saved
270/1000 saved
280/1000 saved
290/1000 saved
300/1000 saved
310/1000 saved
320/1000 saved
330/1000 saved
340/1000 saved
350/1000 saved
360/1000 saved
370/1000 saved
380/1000 saved
390/1000 saved
400/1000 saved
410/1000 saved
420/1000 saved
430/1000 saved
440/1000 saved
450/1000 saved
460/1000 saved
470/1000 saved
480/1000 saved
490/1000 saved
500/1000 saved
510/1000 saved
520/1000 saved
530/1000 saved
540/1000 saved
550/1000 saved
560/1000 saved
570/1000 saved
580/1000 saved
590/1000 saved
600/1000 saved
610/1000 saved
620/1000 saved
630/1000 saved
640/1000 saved
650/1000 saved
660/1000 saved
670/1000 saved
680/

In [ ]:


import os
import re
import json
import time
import random

from tqdm.auto import tqdm
from openai import OpenAI



DATA_DIR = "/content/drive/MyDrive/hedge_run"


INPUT_JSON = (
    f"{DATA_DIR}/"
    "2wiki_passive_iterative_v2_verified_1000.json"
)


TEMPLATE_CACHE_PATH = (
    f"{DATA_DIR}/"
    "2wiki_v2_question_claim_templates_STRICT_V1.json"
)


FINAL_CLAIMS_PATH = (
    f"{DATA_DIR}/"
    "2wiki_v2_final_claims_STRICT_V1.json"
)


CLAIM_MODEL = "gpt-4o-mini"

MAX_API_RETRIES = 5

SAVE_EVERY = 10

SEED = 42



RUN_NLI = False


random.seed(SEED)



with open(
    INPUT_JSON,
    "r",
    encoding="utf-8",
) as f:

    rows = json.load(f)


assert len(rows) == 1000


print(
    "Rows:",
    len(rows)
)


print(
    "dev:",
    sum(
        r["split"] == "dev"
        for r in rows
    )
)


print(
    "test:",
    sum(
        r["split"] == "test"
        for r in rows
    )
)



AUX_STARTS = (
    "is ",
    "are ",
    "was ",
    "were ",
    "do ",
    "does ",
    "did ",
    "can ",
    "could ",
    "would ",
    "will ",
    "has ",
    "have ",
    "had ",
)


def is_yes_no_question(
    question,
):

    q = (
        str(question)
        .strip()
        .lower()
    )


    if not q.startswith(
        AUX_STARTS
    ):

        return False


    if re.search(
        r"\bor\b",
        q,
    ):

        return False


    return True


n_yes_no = sum(
    is_yes_no_question(
        r["question"]
    )
    for r in rows
)


print(
    "Questions treated as genuine yes/no:",
    n_yes_no
)



OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)


if not OPENAI_API_KEY:

    try:

        from google.colab import userdata

        OPENAI_API_KEY = userdata.get(
            "OPENAI_API_KEY"
        )

    except Exception:

        OPENAI_API_KEY = None


if not OPENAI_API_KEY:

    raise RuntimeError(
        "OPENAI_API_KEY not found."
    )


client = OpenAI(
    api_key=OPENAI_API_KEY
)



NON_YN_INSTRUCTIONS = r"""
You are a syntax-only question-to-claim-template converter used in a scientific
question-answering experiment.

You receive ONLY the original question.

Do NOT answer the question.

Rewrite it as ONE short declarative factual claim template containing the
literal token:

[[ANSWER]]

exactly once.

[[ANSWER]] represents an unknown model prediction that will be inserted later
by computer code.

CRITICAL RULES:

1. Never answer or solve the question.
2. Never use outside knowledge.
3. Preserve the exact semantic relation requested.
4. [[ANSWER]] must occupy the exact answer slot.
5. Use [[ANSWER]] exactly once.
6. Do not write "the answer is".
7. Produce a declarative claim, not a question.
8. Preserve entities, dates, comparison direction, quantities, and negation.
9. Do not change the type of requested answer.
10. Do not decide whether any possible answer is factually correct.

For comparison questions:
- preserve both alternatives;
- put [[ANSWER]] in the selected-entity slot.

For "before or after":
- [[ANSWER]] occupies the relation slot.

For "who":
- [[ANSWER]] occupies the requested person/entity slot.

For "where":
- [[ANSWER]] occupies the location slot.

For "when"/"what year":
- [[ANSWER]] occupies the time slot.

For "what film/book/song/etc.":
- [[ANSWER]] must occupy that object's identity/title slot.

Examples:

Question:
Which film has the director who died earlier, Max And Helen or Held Einer Nacht?

Template:
Between Max And Helen and Held Einer Nacht, [[ANSWER]] has the director who died earlier.


Question:
Was Jorge Ledezma or Yuliya Baraley born first?

Template:
Between Jorge Ledezma and Yuliya Baraley, [[ANSWER]] was born first.


Question:
Which person was born earlier, Alice or Bob?

Template:
Between Alice and Bob, [[ANSWER]] was born earlier.


Question:
In what year was the university founded?

Template:
The university was founded in [[ANSWER]].


Question:
Who directed the film?

Template:
The film was directed by [[ANSWER]].
""".strip()



YN_INSTRUCTIONS = r"""
You are a syntax-only yes/no-question converter used in a scientific
question-answering experiment.

You receive ONLY a yes/no question.

Do NOT answer it.

Return:

yes_claim:
the declarative proposition expressed if the answer is Yes.

no_claim:
the declarative proposition expressed if the answer is No.

Rules:

1. Do not decide which one is factually true.
2. Do not use outside knowledge.
3. Preserve exactly the relation in the question.
4. Preserve all entities, dates, quantities, comparisons, and "both".
5. The two claims must represent opposite binary outcomes.
6. Do not mention the words "answer" or "question".
7. Produce declarative factual claims.
""".strip()


NON_YN_SCHEMA = {

    "type":
        "object",

    "properties": {

        "template": {
            "type":
                "string"
        },
    },

    "required": [
        "template",
    ],

    "additionalProperties":
        False,
}


YN_SCHEMA = {

    "type":
        "object",

    "properties": {

        "yes_claim": {
            "type":
                "string"
        },

        "no_claim": {
            "type":
                "string"
        },
    },

    "required": [
        "yes_claim",
        "no_claim",
    ],

    "additionalProperties":
        False,
}



def clean_text(
    text,
):

    return re.sub(
        r"\s+",
        " ",
        str(text),
    ).strip()



def validate_slot_template(
    template,
):

    template = clean_text(
        template
    )


    if (
        template.count(
            "[[ANSWER]]"
        )
        !=
        1
    ):

        raise ValueError(
            "Template must contain [[ANSWER]] exactly once."
        )


    if template.endswith("?"):

        raise ValueError(
            "Template remains a question."
        )


    low = template.lower()


    if (
        "the answer is"
        in low
        or
        "answer to the question"
        in low
    ):

        raise ValueError(
            "Meta-answer wording found."
        )


    return template



def validate_yes_no_pair(
    yes_claim,
    no_claim,
):

    yes_claim = clean_text(
        yes_claim
    )

    no_claim = clean_text(
        no_claim
    )


    if not yes_claim or not no_claim:

        raise ValueError(
            "Empty yes/no claim."
        )


    if yes_claim == no_claim:

        raise ValueError(
            "yes_claim == no_claim"
        )


    if (
        yes_claim.endswith("?")
        or
        no_claim.endswith("?")
    ):

        raise ValueError(
            "Yes/no claim remains a question."
        )


    return (
        yes_claim,
        no_claim,
    )


def generate_slot_template(
    question,
):

    last_error = None


    for attempt in range(
        MAX_API_RETRIES
    ):

        try:

            response = client.responses.create(

                model=
                    CLAIM_MODEL,

                instructions=
                    NON_YN_INSTRUCTIONS,

                input=(
                    "QUESTION:\n"
                    +
                    str(question).strip()
                ),

                temperature=
                    0,

                max_output_tokens=
                    160,

                store=
                    False,

                text={
                    "format": {

                        "type":
                            "json_schema",

                        "name":
                            "question_claim_template",

                        "strict":
                            True,

                        "schema":
                            NON_YN_SCHEMA,
                    }
                },
            )


            obj = json.loads(
                response.output_text
            )


            return validate_slot_template(
                obj["template"]
            )


        except Exception as exc:

            last_error = exc


            if (
                attempt
                <
                MAX_API_RETRIES - 1
            ):

                time.sleep(
                    2 ** attempt
                )


    raise RuntimeError(
        "Template generation failed.\n"
        f"Question: {question}\n"
        f"Last error: {last_error}"
    )


def generate_yes_no_pair(
    question,
):

    last_error = None


    for attempt in range(
        MAX_API_RETRIES
    ):

        try:

            response = client.responses.create(

                model=
                    CLAIM_MODEL,

                instructions=
                    YN_INSTRUCTIONS,

                input=(
                    "QUESTION:\n"
                    +
                    str(question).strip()
                ),

                temperature=
                    0,

                max_output_tokens=
                    180,

                store=
                    False,

                text={
                    "format": {

                        "type":
                            "json_schema",

                        "name":
                            "yes_no_claim_pair",

                        "strict":
                            True,

                        "schema":
                            YN_SCHEMA,
                    }
                },
            )


            obj = json.loads(
                response.output_text
            )


            return validate_yes_no_pair(

                obj["yes_claim"],

                obj["no_claim"],
            )


        except Exception as exc:

            last_error = exc


            if (
                attempt
                <
                MAX_API_RETRIES - 1
            ):

                time.sleep(
                    2 ** attempt
                )


    raise RuntimeError(
        "Yes/no conversion failed.\n"
        f"Question: {question}\n"
        f"Last error: {last_error}"
    )



if os.path.exists(
    TEMPLATE_CACHE_PATH
):

    with open(
        TEMPLATE_CACHE_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        template_cache = json.load(f)

else:

    template_cache = {}


print()
print(
    "Existing strict templates:",
    len(template_cache)
)



print()
print("=" * 80)
print("2WIKI STRICT QUESTION-ONLY CLAIM TEMPLATES")
print("=" * 80)


for row in tqdm(
    rows
):

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    question = str(
        row[
            "question"
        ]
    ).strip()


    kind = (

        "yes_no"

        if is_yes_no_question(
            question
        )

        else "slot"
    )


    if qi in template_cache:

        cached = template_cache[
            qi
        ]


        if (
            cached[
                "question"
            ]
            !=
            question
        ):

            raise RuntimeError(
                f"Cached question mismatch qi={qi}"
            )


        if (
            cached[
                "kind"
            ]
            !=
            kind
        ):

            raise RuntimeError(
                f"Cached type mismatch qi={qi}"
            )


        continue




    if kind == "slot":

        template = (
            generate_slot_template(
                question
            )
        )


        record = {

            "question_index":
                int(qi),

            "question":
                question,

            "kind":
                "slot",

            "template":
                template,

            "converter_model":
                CLAIM_MODEL,

            "answer_seen_by_converter":
                False,

            "gold_seen_by_converter":
                False,

            "evidence_seen_by_converter":
                False,
        }


    else:

        yes_claim, no_claim = (
            generate_yes_no_pair(
                question
            )
        )


        record = {

            "question_index":
                int(qi),

            "question":
                question,

            "kind":
                "yes_no",

            "yes_claim":
                yes_claim,

            "no_claim":
                no_claim,

            "converter_model":
                CLAIM_MODEL,

            "answer_seen_by_converter":
                False,

            "gold_seen_by_converter":
                False,

            "evidence_seen_by_converter":
                False,
        }


    template_cache[
        qi
    ] = record


    if (
        len(template_cache)
        %
        SAVE_EVERY
        ==
        0
    ):

        with open(
            TEMPLATE_CACHE_PATH,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                template_cache,
                f,
                indent=2,
                ensure_ascii=False,
            )


        print(
            f"{len(template_cache)}/1000 saved"
        )


# Final save.

with open(
    TEMPLATE_CACHE_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        template_cache,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert len(
    template_cache
) == 1000


print(
    "\nAll 1000 templates available."
)



def normalize_yes_no_answer(
    answer,
):

    a = (
        str(answer)
        .strip()
        .lower()
    )


    a = re.sub(
        r"[^a-z]",
        "",
        a,
    )


    if a in {
        "yes",
        "true",
    }:

        return "yes"


    if a in {
        "no",
        "false",
    }:

        return "no"


    return None


def construct_final_claim(
    row,
):

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    answer = str(
        row[
            "majority_answer"
        ]
    ).strip()


    if not answer:

        raise RuntimeError(
            f"Empty majority answer qi={qi}"
        )


    info = (
        template_cache[
            qi
        ]
    )


    if (
        info[
            "kind"
        ]
        ==
        "slot"
    ):

        template = (
            info[
                "template"
            ]
        )


        claim = template.replace(
            "[[ANSWER]]",
            answer,
        )


        if (
            "[[ANSWER]]"
            in claim
        ):

            raise RuntimeError(
                f"Placeholder remains qi={qi}"
            )


        if answer not in claim:

            raise RuntimeError(
                f"Answer insertion failed qi={qi}"
            )


        branch = (
            "slot_substitution"
        )


    else:

        yn = normalize_yes_no_answer(
            answer
        )


        if yn is None:

            raise RuntimeError(
                "Question classified yes/no but "
                "prediction is not yes/no.\n"
                f"QI={qi}\n"
                f"Q={row['question']}\n"
                f"A={answer}"
            )


        claim = (

            info[
                "yes_claim"
            ]

            if yn == "yes"

            else info[
                "no_claim"
            ]
        )


        branch = yn


    claim = clean_text(
        claim
    )


    return {

        "question_index":
            int(qi),

        "question":
            row[
                "question"
            ],

        "majority_answer":
            answer,

        "kind":
            info[
                "kind"
            ],

        "branch":
            branch,

        "final_claim":
            claim,

        "converter_saw_answer":
            False,

        "converter_saw_gold":
            False,

        "converter_saw_evidence":
            False,
    }


final_claims = {}


for row in rows:

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    final_claims[
        qi
    ] = (
        construct_final_claim(
            row
        )
    )


assert len(
    final_claims
) == 1000


with open(
    FINAL_CLAIMS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_claims,
        f,
        indent=2,
        ensure_ascii=False,
    )


print()
print(
    "Saved strict claims:"
)

print(
    FINAL_CLAIMS_PATH
)


n_slot = sum(

    record[
        "kind"
    ]
    ==
    "slot"

    for record
    in final_claims.values()
)


n_yn = sum(

    record[
        "kind"
    ]
    ==
    "yes_no"

    for record
    in final_claims.values()
)


print()
print("=" * 80)
print("2WIKI CLAIM TYPE SUMMARY")
print("=" * 80)


print(
    "Slot claims:",
    n_slot
)


print(
    "Yes/no claims:",
    n_yn
)


check_ids = set()


# First 20 records.
for row in rows[
    :20
]:

    check_ids.add(
        int(
            row[
                "question_index"
            ]
        )
    )



for row in rows:

    if (
        row.get(
            "final_answer_question_type_patch",
            {}
        )
        .get(
            "applied",
            False,
        )
    ):

        check_ids.add(
            int(
                row[
                    "question_index"
            ]
        )
    )



rng = random.Random(
    42
)


all_ids = [

    int(
        row[
            "question_index"
        ]
    )

    for row in rows
]


check_ids.update(
    rng.sample(
        all_ids,
        30,
    )
)


rows_by_qi = {

    int(
        row[
            "question_index"
        ]
    ):
        row

    for row in rows
}



print()
print("=" * 80)
print("2WIKI STRICT CLAIM MANUAL CHECK")
print("=" * 80)


print(
    "Examples shown:",
    len(
        check_ids
    )
)


for qi in sorted(
    check_ids
):

    row = rows_by_qi[
        qi
    ]


    record = final_claims[
        str(qi)
    ]


    info = template_cache[
        str(qi)
    ]


    print()
    print(
        "QI:",
        qi
    )


    print(
        "QUESTION:"
    )
    print(
        row[
            "question"
        ]
    )


    print(
        "MAJORITY ANSWER:"
    )
    print(
        row[
            "majority_answer"
        ]
    )


    print(
        "TYPE:",
        record[
            "kind"
        ]
    )


    if (
        record[
            "kind"
        ]
        ==
        "slot"
    ):

        print(
            "TEMPLATE:"
        )

        print(
            info[
                "template"
            ]
        )


    else:

        print(
            "YES CLAIM:"
        )
        print(
            info[
                "yes_claim"
            ]
        )


        print(
            "NO CLAIM:"
        )
        print(
            info[
                "no_claim"
            ]
        )


    print(
        "FINAL CLAIM:"
    )

    print(
        record[
            "final_claim"
        ]
    )


    print(
        "-" * 80
    )



print()
print("=" * 80)
print("2WIKI CLAIM TEMPLATE STAGE COMPLETE")
print("=" * 80)


print(
    "NLI has NOT been run yet."
)


print(
    "Inspect the manual-check examples above."
)


print(
    "The 1000 question-only templates are cached, "
    "so they will not need to be regenerated."
)

Rows: 1000
dev: 800
test: 200
Questions treated as genuine yes/no: 95

Existing strict templates: 0

2WIKI STRICT QUESTION-ONLY CLAIM TEMPLATES


  0%|          | 0/1000 [00:00<?, ?it/s]

10/1000 saved
20/1000 saved
30/1000 saved
40/1000 saved
50/1000 saved
60/1000 saved
70/1000 saved
80/1000 saved
90/1000 saved
100/1000 saved
110/1000 saved
120/1000 saved
130/1000 saved
140/1000 saved
150/1000 saved
160/1000 saved
170/1000 saved
180/1000 saved
190/1000 saved
200/1000 saved
210/1000 saved
220/1000 saved
230/1000 saved
240/1000 saved
250/1000 saved
260/1000 saved
270/1000 saved
280/1000 saved
290/1000 saved
300/1000 saved
310/1000 saved
320/1000 saved
330/1000 saved
340/1000 saved
350/1000 saved
360/1000 saved
370/1000 saved
380/1000 saved
390/1000 saved
400/1000 saved
410/1000 saved
420/1000 saved
430/1000 saved
440/1000 saved
450/1000 saved
460/1000 saved
470/1000 saved
480/1000 saved
490/1000 saved
500/1000 saved
510/1000 saved
520/1000 saved
530/1000 saved
540/1000 saved
550/1000 saved
560/1000 saved
570/1000 saved
580/1000 saved
590/1000 saved
600/1000 saved
610/1000 saved
620/1000 saved
630/1000 saved
640/1000 saved
650/1000 saved
660/1000 saved
670/1000 saved
680/

In [ ]:
import json
import re

D = "/content/drive/MyDrive/hedge_run"

TEMPLATE_PATH = (
    f"{D}/"
    "2wiki_v2_question_claim_templates_STRICT_V1.json"
)

CLAIMS_PATH = (
    f"{D}/"
    "2wiki_v2_final_claims_STRICT_V1.json"
)

VERIFIED_PATH = (
    f"{D}/"
    "2wiki_passive_iterative_v2_verified_1000.json"
)



with open(TEMPLATE_PATH, encoding="utf-8") as f:
    templates = json.load(f)

with open(VERIFIED_PATH, encoding="utf-8") as f:
    rows = json.load(f)

assert len(templates) == 1000
assert len(rows) == 1000



PATCHES = {

    "886": (
        "The country of origin of the performer of the song "
        "That's Good, That's Bad (Frankie Laine Song) is [[ANSWER]]."
    ),

    "1947": (
        "The date of death of Deuteria's husband was [[ANSWER]]."
    ),

    "4243": (
        "The date of death of Joachim I, Prince Of Anhalt-Dessau's "
        "father was [[ANSWER]]."
    ),

    "8523": (
        "The date of birth of the founder of Mcgraw Electric "
        "was [[ANSWER]]."
    ),

    "1105": (
        "The place where the director of the film Free Lips died "
        "is [[ANSWER]]."
    ),

    "8994": (
        "Nicholas Hughes's mother's workplace is [[ANSWER]]."
    ),
}


def validate_template(template):

    if template.count("[[ANSWER]]") != 1:
        raise ValueError(
            "Template must contain [[ANSWER]] exactly once."
        )

    if template.rstrip().endswith("?"):
        raise ValueError(
            "Template must be declarative."
        )

    return template.strip()


for qi, new_template in PATCHES.items():

    if qi not in templates:
        raise RuntimeError(f"Missing QI {qi}")

    if templates[qi]["kind"] != "slot":
        raise RuntimeError(
            f"QI {qi} is unexpectedly not a slot claim."
        )

    old = templates[qi]["template"]

    new_template = validate_template(
        new_template
    )

    print()
    print("QI:", qi)
    print("OLD:", old)
    print("NEW:", new_template)

    templates[qi]["template"] = new_template
    templates[qi]["manual_question_only_patch"] = True



with open(
    TEMPLATE_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        templates,
        f,
        indent=2,
        ensure_ascii=False,
    )



rows_by_qi = {
    str(int(r["question_index"])): r
    for r in rows
}


def norm_yes_no(answer):

    x = re.sub(
        r"[^a-z]",
        "",
        str(answer).lower(),
    )

    if x in {"yes", "true"}:
        return "yes"

    if x in {"no", "false"}:
        return "no"

    return None


claims = {}


for qi, row in rows_by_qi.items():

    info = templates[qi]

    answer = str(
        row["majority_answer"]
    ).strip()

    if info["kind"] == "slot":

        template = info["template"]

        assert template.count("[[ANSWER]]") == 1

        claim = template.replace(
            "[[ANSWER]]",
            answer,
        )

        branch = "slot_substitution"

    else:

        yn = norm_yes_no(
            answer
        )

        if yn is None:
            raise RuntimeError(
                f"Non-yes/no answer for yes/no QI {qi}: {answer}"
            )

        claim = (
            info["yes_claim"]
            if yn == "yes"
            else info["no_claim"]
        )

        branch = yn


    claim = re.sub(
        r"\s+",
        " ",
        claim,
    ).strip()


    claims[qi] = {

        "question_index":
            int(qi),

        "question":
            row["question"],

        "majority_answer":
            answer,

        "kind":
            info["kind"],

        "branch":
            branch,

        "final_claim":
            claim,

        "converter_saw_answer":
            False,

        "converter_saw_gold":
            False,

        "converter_saw_evidence":
            False,
    }


assert len(claims) == 1000


with open(
    CLAIMS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        claims,
        f,
        indent=2,
        ensure_ascii=False,
    )



print()
print("=" * 80)
print("PATCHED CLAIM CHECK")
print("=" * 80)


for qi in PATCHES:

    print()
    print("QI:", qi)
    print("QUESTION:")
    print(rows_by_qi[qi]["question"])

    print("MAJORITY:")
    print(rows_by_qi[qi]["majority_answer"])

    print("FINAL CLAIM:")
    print(claims[qi]["final_claim"])

    print("-" * 80)


print()
print("=" * 80)
print("GLOBAL EDGE-CASE AUDIT")
print("=" * 80)


NO_INFO_TERMS = [
    "unknown",
    "not specified",
    "information not provided",
    "not provided",
    "cannot determine",
    "cannot be determined",
    "not enough information",
]


no_info = []

long_answers = []

sentence_answers = []


for qi, row in rows_by_qi.items():

    answer = str(
        row["majority_answer"]
    ).strip()

    low = answer.lower()


    if any(
        phrase in low
        for phrase in NO_INFO_TERMS
    ):
        no_info.append(qi)


    if len(answer.split()) >= 8:
        long_answers.append(qi)


    if (
        answer.endswith(".")
        or
        answer.endswith("!")
        or
        answer.endswith("?")
    ):
        sentence_answers.append(qi)


print(
    "No-information-like predictions:",
    len(no_info)
)

print(
    "IDs:",
    no_info
)


print()
print(
    "Long predictions (>=8 words):",
    len(long_answers)
)

print(
    "IDs:",
    long_answers
)


print()
print(
    "Predictions ending in sentence punctuation:",
    len(sentence_answers)
)

print(
    "IDs:",
    sentence_answers
)


print()
print(
    "Details for potentially unusual predictions:"
)


interesting = sorted(
    set(
        no_info
        +
        long_answers
        +
        sentence_answers
    ),
    key=int,
)


for qi in interesting:

    print()
    print("QI:", qi)
    print("QUESTION:", rows_by_qi[qi]["question"])
    print("ANSWER:", rows_by_qi[qi]["majority_answer"])
    print("CLAIM:", claims[qi]["final_claim"])
    print("-" * 80)


print()
print("DONE.")


QI: 886
OLD: The performer of the song That's Good, That's Bad (Frankie Laine Song) is from [[ANSWER]].
NEW: The country of origin of the performer of the song That's Good, That's Bad (Frankie Laine Song) is [[ANSWER]].

QI: 1947
OLD: Deuteria's husband died on [[ANSWER]].
NEW: The date of death of Deuteria's husband was [[ANSWER]].

QI: 4243
OLD: Joachim I, Prince Of Anhalt-Dessau's father died in [[ANSWER]].
NEW: The date of death of Joachim I, Prince Of Anhalt-Dessau's father was [[ANSWER]].

QI: 8523
OLD: The founder of Mcgraw Electric was born on [[ANSWER]].
NEW: The date of birth of the founder of Mcgraw Electric was [[ANSWER]].

QI: 1105
OLD: The director of the film Free Lips died in [[ANSWER]].
NEW: The place where the director of the film Free Lips died is [[ANSWER]].

QI: 8994
OLD: Nicholas Hughes's mother works at [[ANSWER]].
NEW: Nicholas Hughes's mother's workplace is [[ANSWER]].

PATCHED CLAIM CHECK

QI: 886
QUESTION:
Which country the performer of song That'S Good, Tha

In [ ]:
import json
import re

D = "/content/drive/MyDrive/hedge_run"

TEMPLATE_PATH = (
    f"{D}/"
    "2wiki_v2_question_claim_templates_STRICT_V1.json"
)

CLAIMS_PATH = (
    f"{D}/"
    "2wiki_v2_final_claims_STRICT_V1.json"
)

VERIFIED_PATH = (
    f"{D}/"
    "2wiki_passive_iterative_v2_verified_1000.json"
)


with open(TEMPLATE_PATH, encoding="utf-8") as f:
    templates = json.load(f)

with open(VERIFIED_PATH, encoding="utf-8") as f:
    rows = json.load(f)

assert len(templates) == 1000
assert len(rows) == 1000


rows_by_qi = {
    str(int(r["question_index"])): r
    for r in rows
}


FULL_SENTENCE_IDS = {
    "777",
    "2217",
    "2700",
    "3041",
    "3781",
    "5810",
    "6295",
    "6488",
    "7211",
    "7453",
    "7654",
    "8028",
    "8425",
    "8894",
    "9876",
    "10205",
    "10238",
    "10560",
    "11344",
    "12073",
}


assert len(FULL_SENTENCE_IDS) == 20


for qi in FULL_SENTENCE_IDS:

    if qi not in templates:
        raise RuntimeError(
            f"Missing template QI={qi}"
        )

    if templates[qi]["kind"] != "slot":
        raise RuntimeError(
            f"Unexpected non-slot QI={qi}"
        )

    templates[qi][
        "allow_full_sentence_answer"
    ] = True

    templates[qi][
        "full_sentence_reason"
    ] = (
        "model_prediction_is_already_propositional"
    )


with open(
    TEMPLATE_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        templates,
        f,
        indent=2,
        ensure_ascii=False,
    )



def norm_yes_no(answer):

    x = re.sub(
        r"[^a-z]",
        "",
        str(answer).lower(),
    )

    if x in {"yes", "true"}:
        return "yes"

    if x in {"no", "false"}:
        return "no"

    return None




claims = {}


for qi, row in rows_by_qi.items():

    info = templates[qi]

    answer = str(
        row["majority_answer"]
    ).strip()


    if not answer:
        raise RuntimeError(
            f"Empty answer QI={qi}"
        )



    if info["kind"] == "slot":

        # Model output is already a proposition.
        if info.get(
            "allow_full_sentence_answer",
            False,
        ):

            claim = answer

            branch = (
                "full_sentence_answer_verbatim"
            )

        else:

            template = info["template"]

            if template.count(
                "[[ANSWER]]"
            ) != 1:

                raise RuntimeError(
                    f"Bad template QI={qi}"
                )

            claim = template.replace(
                "[[ANSWER]]",
                answer,
            )

            branch = (
                "slot_substitution"
            )



    else:

        yn = norm_yes_no(
            answer
        )

        if yn is None:
            raise RuntimeError(
                f"Bad yes/no answer QI={qi}: {answer}"
            )

        claim = (
            info["yes_claim"]
            if yn == "yes"
            else info["no_claim"]
        )

        branch = yn


    claim = re.sub(
        r"\s+",
        " ",
        str(claim),
    ).strip()


    claims[qi] = {

        "question_index":
            int(qi),

        "question":
            row["question"],

        "majority_answer":
            answer,

        "kind":
            info["kind"],

        "branch":
            branch,

        "final_claim":
            claim,

        "converter_saw_answer":
            False,

        "converter_saw_gold":
            False,

        "converter_saw_evidence":
            False,
    }


assert len(claims) == 1000



with open(
    CLAIMS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        claims,
        f,
        indent=2,
        ensure_ascii=False,
    )



print("=" * 80)
print("FULL-SENTENCE CLAIM CHECK")
print("=" * 80)


for qi in sorted(
    FULL_SENTENCE_IDS,
    key=int,
):

    row = rows_by_qi[qi]

    print()
    print("QI:", qi)

    print("QUESTION:")
    print(
        row["question"]
    )

    print("MODEL PREDICTION:")
    print(
        row["majority_answer"]
    )

    print("FINAL CLAIM:")
    print(
        claims[qi][
            "final_claim"
        ]
    )

    print(
        "BRANCH:",
        claims[qi][
            "branch"
        ]
    )

    print("-" * 80)


for qi in FULL_SENTENCE_IDS:

    assert (
        claims[qi]["final_claim"]
        ==
        rows_by_qi[qi]["majority_answer"].strip()
    )

    assert (
        claims[qi]["branch"]
        ==
        "full_sentence_answer_verbatim"
    )


print()
print("=" * 80)
print("FINAL STRICT CLAIM STATUS")
print("=" * 80)

print(
    "Total claims:",
    len(claims)
)

print(
    "Full-sentence verbatim:",
    sum(
        x["branch"]
        ==
        "full_sentence_answer_verbatim"

        for x in claims.values()
    )
)

print(
    "Slot substitution:",
    sum(
        x["branch"]
        ==
        "slot_substitution"

        for x in claims.values()
    )
)

print(
    "Yes/no:",
    sum(
        x["kind"]
        ==
        "yes_no"

        for x in claims.values()
    )
)

print()
print(
    "Saved:",
    CLAIMS_PATH
)

print()
print("DONE.")

FULL-SENTENCE CLAIM CHECK

QI: 777
QUESTION:
Where did the director of film Mary Jane'S Pa die?
MODEL PREDICTION:
the information provided does not specify where the director of Mary Jane's Pa died.
FINAL CLAIM:
the information provided does not specify where the director of Mary Jane's Pa died.
BRANCH: full_sentence_answer_verbatim
--------------------------------------------------------------------------------

QI: 2217
QUESTION:
Where did Jean Gimpel's father die?
MODEL PREDICTION:
the evidence does not provide information about where Jean Gimpel's father died.
FINAL CLAIM:
the evidence does not provide information about where Jean Gimpel's father died.
BRANCH: full_sentence_answer_verbatim
--------------------------------------------------------------------------------

QI: 2700
QUESTION:
Which film whose director is younger, Lucky Cowboy or O.K. Connery?
MODEL PREDICTION:
Josef Berne directed Lucky Cowboy.
FINAL CLAIM:
Josef Berne directed Lucky Cowboy.
BRANCH: full_sentence_answe

In [ ]:


import os
import json
import copy

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)



D = "/content/drive/MyDrive/hedge_run"


VERIFIED_JSON = (
    f"{D}/"
    "2wiki_passive_iterative_v2_verified_1000.json"
)


CLAIMS_JSON = (
    f"{D}/"
    "2wiki_v2_final_claims_STRICT_V1.json"
)


NLI_PROGRESS_JSON = (
    f"{D}/"
    "2wiki_v2_final_nli_STRICT_V1_progress.json"
)


OUTPUT_JSON = (
    f"{D}/"
    "2wiki_passive_iterative_v2_FINAL_CLEAN_1000.json"
)


OUTPUT_CSV = (
    f"{D}/"
    "2wiki_passive_iterative_v2_FINAL_CLEAN_features17.csv"
)



NLI_MODEL_ID = (
    "MoritzLaurer/"
    "DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
)


SUPPORT_THRESHOLD = 0.50

CONTRADICTION_THRESHOLD = 0.50

NLI_MAX_LENGTH = 512

NLI_BATCH_SIZE = 16

SAVE_EVERY = 50



REASONING_FEATURES = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]


CONFIDENCE_FEATURES = [
    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]


FINAL_FEATURES = [
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]


FEATURE_COLUMNS = (
    REASONING_FEATURES
    +
    CONFIDENCE_FEATURES
    +
    FINAL_FEATURES
)


assert len(FEATURE_COLUMNS) == 17


with open(
    VERIFIED_JSON,
    "r",
    encoding="utf-8",
) as f:

    rows = json.load(f)


with open(
    CLAIMS_JSON,
    "r",
    encoding="utf-8",
) as f:

    claims = json.load(f)


assert len(rows) == 1000
assert len(claims) == 1000


print(
    "Verified rows:",
    len(rows)
)


print(
    "Final claims:",
    len(claims)
)


print(
    "dev:",
    sum(
        r["split"] == "dev"
        for r in rows
    )
)


print(
    "test:",
    sum(
        r["split"] == "test"
        for r in rows
    )
)



n_full_sentence = sum(

    c["branch"]
    ==
    "full_sentence_answer_verbatim"

    for c in claims.values()
)


n_slot = sum(

    c["branch"]
    ==
    "slot_substitution"

    for c in claims.values()
)


n_yn = sum(

    c["kind"]
    ==
    "yes_no"

    for c in claims.values()
)


print()
print(
    "Full-sentence verbatim:",
    n_full_sentence
)

print(
    "Slot substitution:",
    n_slot
)

print(
    "Yes/no:",
    n_yn
)


assert n_full_sentence == 20
assert n_slot == 885
assert n_yn == 95


for row in rows:

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    if qi not in claims:

        raise RuntimeError(
            f"Missing claim QI={qi}"
        )


    if (
        str(
            claims[qi]["question"]
        ).strip()
        !=
        str(
            row["question"]
        ).strip()
    ):

        raise RuntimeError(
            f"Question mismatch QI={qi}"
        )


    if (
        str(
            claims[qi]["majority_answer"]
        ).strip()
        !=
        str(
            row["majority_answer"]
        ).strip()
    ):

        raise RuntimeError(
            f"Majority mismatch QI={qi}"
        )


print(
    "Claim/generation alignment: YES"
)



def build_evidence_block(
    row,
):

    evidence = [

        str(x).strip()

        for x in row.get(
            "question_evidence",
            []
        )

        if str(x).strip()
    ]


    if not evidence:

        raise RuntimeError(
            "Missing question evidence "
            f"QI={row['question_index']}"
        )


    return "\n".join(

        f"E{i}: {sentence}"

        for i, sentence
        in enumerate(
            evidence,
            start=1,
        )
    )


print()
print("=" * 80)
print("LOADING FINAL-ANSWER NLI")
print("=" * 80)


tokenizer = (
    AutoTokenizer.from_pretrained(
        NLI_MODEL_ID
    )
)


if torch.cuda.is_available():

    DEVICE = torch.device(
        "cuda"
    )

    DTYPE = torch.float16

else:

    DEVICE = torch.device(
        "cpu"
    )

    DTYPE = torch.float32


try:

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            NLI_MODEL_ID,
            dtype=DTYPE,
        )
    )

except TypeError:

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            NLI_MODEL_ID,
            torch_dtype=DTYPE,
        )
    )


model = model.to(
    DEVICE
)

model.eval()


id2label = {

    int(k):
        str(v).lower()

    for k, v
    in model.config.id2label.items()
}


ENT_IDX = next(

    i

    for i, label
    in id2label.items()

    if "entail" in label
)


NEU_IDX = next(

    i

    for i, label
    in id2label.items()

    if "neutral" in label
)


CON_IDX = next(

    i

    for i, label
    in id2label.items()

    if "contrad" in label
)


print(
    "Device:",
    DEVICE
)


print(
    "Labels:",
    id2label
)



over_512 = []


for row in rows:

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    premise = build_evidence_block(
        row
    )


    hypothesis = (
        claims[
            qi
        ][
            "final_claim"
        ]
    )


    ids = tokenizer(

        premise,

        hypothesis,

        add_special_tokens=True,

        truncation=False,
    )[
        "input_ids"
    ]


    if len(ids) > NLI_MAX_LENGTH:

        over_512.append(
            {
                "question_index":
                    int(qi),

                "length":
                    len(ids),
            }
        )


print()
print(
    "Pairs >512 tokens before truncation:",
    len(over_512)
)


if over_512:

    print(
        "Maximum pair length:",
        max(
            x["length"]
            for x in over_512
        )
    )


@torch.no_grad()
def score_batch(
    premises,
    hypotheses,
):

    encoded = tokenizer(

        premises,

        hypotheses,

        padding=True,

        truncation="only_first",

        max_length=NLI_MAX_LENGTH,

        return_tensors="pt",
    )


    encoded = {

        key:
            value.to(
                DEVICE
            )

        for key, value
        in encoded.items()
    }


    logits = (
        model(
            **encoded
        )
        .logits
        .float()
    )


    probs = (
        torch.softmax(
            logits,
            dim=-1,
        )
        .cpu()
        .numpy()
    )


    return probs



if os.path.exists(
    NLI_PROGRESS_JSON
):

    with open(
        NLI_PROGRESS_JSON,
        "r",
        encoding="utf-8",
    ) as f:

        progress = json.load(f)

else:

    progress = {}


print()
print(
    "Existing final-NLI rows:",
    len(progress)
)


pending = [

    row

    for row in rows

    if (
        str(
            int(
                row[
                    "question_index"
                ]
            )
        )
        not in progress
    )
]


print(
    "Pending:",
    len(pending)
)



print()
print("=" * 80)
print("2WIKI FINAL-ANSWER NLI")
print("=" * 80)


processed_since_save = 0


for start in tqdm(

    range(
        0,
        len(pending),
        NLI_BATCH_SIZE,
    )
):

    batch = pending[
        start:
        start + NLI_BATCH_SIZE
    ]


    premises = [

        build_evidence_block(
            row
        )

        for row
        in batch
    ]


    hypotheses = [

        claims[
            str(
                int(
                    row[
                        "question_index"
                    ]
                )
            )
        ][
            "final_claim"
        ]

        for row
        in batch
    ]


    probs = score_batch(
        premises,
        hypotheses,
    )


    for row, claim, p in zip(
        batch,
        hypotheses,
        probs,
    ):

        qi = str(
            int(
                row[
                    "question_index"
                ]
            )
        )


        support = float(
            p[
                ENT_IDX
            ]
        )


        neutral = float(
            p[
                NEU_IDX
            ]
        )


        contradiction = float(
            p[
                CON_IDX
            ]
        )


        if (
            support
            >=
            SUPPORT_THRESHOLD
        ):

            label = "supported"

        elif (
            contradiction
            >=
            CONTRADICTION_THRESHOLD
        ):

            label = "contradicted"

        else:

            label = "unclear"


        progress[
            qi
        ] = {

            "question_index":
                int(qi),

            "majority_answer":
                row[
                    "majority_answer"
                ],

            "final_claim":
                claim,

            "retrieved_evidence":
                copy.deepcopy(
                    row[
                        "question_evidence"
                    ]
                ),

            "support":
                support,

            "neutral":
                neutral,

            "contradiction":
                contradiction,

            "label":
                label,

            "contradicted":
                int(
                    contradiction
                    >=
                    CONTRADICTION_THRESHOLD
                ),

            "nli_model":
                NLI_MODEL_ID,

            "nli_evidence_mode":
                "combined_top10_bm25",

            "gold_used":
                False,
        }


        processed_since_save += 1


    if (
        processed_since_save
        >=
        SAVE_EVERY
    ):

        with open(
            NLI_PROGRESS_JSON,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                progress,
                f,
                indent=2,
                ensure_ascii=False,
            )


        processed_since_save = 0


# Final save.

with open(
    NLI_PROGRESS_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        progress,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert len(progress) == 1000


print(
    "Final-answer NLI complete."
)




final_rows = []


for row in rows:

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    final_nli = (
        progress[
            qi
        ]
    )




    reasoning_features = {

        feature:
            float(
                row[
                    "features"
                ][
                    feature
                ]
            )

        for feature
        in REASONING_FEATURES
    }




    confidence_features = {

        "self_consistency":
            float(
                row[
                    "self_consistency"
                ]
            ),

        "answer_norm_entropy":
            float(
                row[
                    "answer_norm_entropy"
                ]
            ),

        "answer_logprob":
            float(
                row[
                    "answer_logprob"
                ]
            ),
    }




    final_features = {

        "fa_support_retr":
            float(
                final_nli[
                    "support"
                ]
            ),

        "fa_contradiction_retr":
            float(
                final_nli[
                    "contradiction"
                ]
            ),

        "fa_contradicted_retr":
            int(
                final_nli[
                    "contradicted"
                ]
            ),
    }


    features = {

        **reasoning_features,

        **confidence_features,

        **final_features,
    }


    if (
        set(
            features.keys()
        )
        !=
        set(
            FEATURE_COLUMNS
        )
    ):

        raise RuntimeError(
            f"Feature mismatch QI={qi}"
        )


    new_row = copy.deepcopy(
        row
    )


    new_row[
        "final_claim"
    ] = claims[
        qi
    ][
        "final_claim"
    ]


    new_row[
        "final_claim_info"
    ] = copy.deepcopy(
        claims[
            qi
        ]
    )


    new_row[
        "final_answer_verification"
    ] = copy.deepcopy(
        final_nli
    )


    new_row[
        "features"
    ] = features


    final_rows.append(
        new_row
    )


assert len(
    final_rows
) == 1000


assert (
    sum(
        r["split"] == "dev"
        for r in final_rows
    )
    ==
    800
)


assert (
    sum(
        r["split"] == "test"
        for r in final_rows
    )
    ==
    200
)


for row in final_rows:

    values = [

        float(
            row[
                "features"
            ][
                feature
            ]
        )

        for feature
        in FEATURE_COLUMNS
    ]


    if not np.all(
        np.isfinite(
            values
        )
    ):

        raise RuntimeError(
            "Non-finite feature "
            f"QI={row['question_index']}"
        )


print(
    "All 17 features finite: YES"
)


with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_rows,
        f,
        indent=2,
        ensure_ascii=False,
    )



csv_rows = []


for row in final_rows:

    csv_rows.append(
        {

            "question_index":
                row[
                    "question_index"
                ],

            "split":
                row[
                    "split"
                ],

            "question":
                row[
                    "question"
                ],

            "gold_answer":
                row[
                    "gold_answer"
                ],

            "majority_answer":
                row[
                    "majority_answer"
                ],

            "final_claim":
                row[
                    "final_claim"
                ],

            **row[
                "features"
            ],
        }
    )


df = pd.DataFrame(
    csv_rows
)


df.to_csv(
    OUTPUT_CSV,
    index=False,
)



print()
print("=" * 80)
print("2WIKI FINAL CLEAN FEATURE SUMMARY")
print("=" * 80)


print(
    "Rows:",
    len(df)
)


print(
    "dev:",
    int(
        df[
            "split"
        ]
        .eq(
            "dev"
        )
        .sum()
    )
)


print(
    "test:",
    int(
        df[
            "split"
        ]
        .eq(
            "test"
        )
        .sum()
    )
)


print(
    "Final support >= 0.5:",
    int(
        (
            df[
                "fa_support_retr"
            ]
            >=
            SUPPORT_THRESHOLD
        )
        .sum()
    )
)


print(
    "Final contradiction >= 0.5:",
    int(
        df[
            "fa_contradicted_retr"
        ]
        .sum()
    )
)


print(
    "Mean final support:",
    round(
        float(
            df[
                "fa_support_retr"
            ]
            .mean()
        ),
        4,
    )
)


print(
    "Mean final contradiction:",
    round(
        float(
            df[
                "fa_contradiction_retr"
            ]
            .mean()
        ),
        4,
    )
)


print(
    "Pairs >512 before truncation:",
    len(
        over_512
    )
)


if over_512:

    print(
        "Maximum pair length:",
        max(
            x[
                "length"
            ]
            for x
            in over_512
        )
    )


print()
print(
    "17 FEATURES:"
)


for i, feature in enumerate(
    FEATURE_COLUMNS,
    start=1,
):

    print(
        f"{i:2d}. {feature}"
    )


print()
print(
    "AUTHORITATIVE FINAL JSON:"
)

print(
    OUTPUT_JSON
)


print()
print(
    "AUTHORITATIVE FINAL CSV:"
)

print(
    OUTPUT_CSV
)


print()
print(
    "DONE."
)

Verified rows: 1000
Final claims: 1000
dev: 800
test: 200

Full-sentence verbatim: 20
Slot substitution: 885
Yes/no: 95
Claim/generation alignment: YES

LOADING FINAL-ANSWER NLI


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (574 > 512). Running this sequence through the model will result in indexing errors


Device: cuda
Labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}

Pairs >512 tokens before truncation: 11
Maximum pair length: 592

Existing final-NLI rows: 0
Pending: 1000

2WIKI FINAL-ANSWER NLI


  0%|          | 0/63 [00:00<?, ?it/s]

Final-answer NLI complete.
All 17 features finite: YES

2WIKI FINAL CLEAN FEATURE SUMMARY
Rows: 1000
dev: 800
test: 200
Final support >= 0.5: 590
Final contradiction >= 0.5: 187
Mean final support: 0.5888
Mean final contradiction: 0.196
Pairs >512 before truncation: 11
Maximum pair length: 592

17 FEATURES:
 1. max_contradiction_score
 2. any_contradicted
 3. min_support_score
 4. num_steps
 5. frac_supported
 6. frac_contradicted
 7. frac_unclear
 8. mean_support
 9. mean_contradiction
10. conflict
11. support_spread
12. self_consistency
13. answer_norm_entropy
14. answer_logprob
15. fa_support_retr
16. fa_contradiction_retr
17. fa_contradicted_retr

AUTHORITATIVE FINAL JSON:
/content/drive/MyDrive/hedge_run/2wiki_passive_iterative_v2_FINAL_CLEAN_1000.json

AUTHORITATIVE FINAL CSV:
/content/drive/MyDrive/hedge_run/2wiki_passive_iterative_v2_FINAL_CLEAN_features17.csv

DONE.


In [ ]:
import os
import re
import json
import time

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from openai import OpenAI

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score



D = "/content/drive/MyDrive/hedge_run"


INPUT_JSON = (
    f"{D}/"
    "2wiki_passive_iterative_v2_FINAL_CLEAN_1000.json"
)


JUDGE_PROGRESS = (
    f"{D}/"
    "2wiki_passive_iterative_v2_correctness_progress.json"
)


JUDGE_OUTPUT = (
    f"{D}/"
    "2wiki_passive_iterative_v2_correctness_1000.json"
)


LABELLED_JSON = (
    f"{D}/"
    "2wiki_passive_iterative_v2_FINAL_CLEAN_LABELLED_1000.json"
)


LABELLED_CSV = (
    f"{D}/"
    "2wiki_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
)


EXP1_JSON = (
    f"{D}/"
    "2wiki_experiment1_grouped_auroc.json"
)


OOF_CSV = (
    f"{D}/"
    "2wiki_experiment1_oof_predictions.csv"
)


JUDGE_MODEL = "gpt-4o-mini"

MAX_RETRIES = 5

SAVE_EVERY = 10

SEED = 42

N_SPLITS = 5

N_BOOTSTRAP = 5000

PAIRED_BOOTSTRAP = 10000

RISK_PERCENTILE = 60


REASONING_FEATURES = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]


CONFIDENCE_FEATURES = [
    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]


FINAL_FEATURES = [
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]


ANSWER_LEVEL_FEATURES = (
    CONFIDENCE_FEATURES
    +
    FINAL_FEATURES
)


FULL_FEATURES = (
    REASONING_FEATURES
    +
    CONFIDENCE_FEATURES
    +
    FINAL_FEATURES
)


FEATURE_GROUPS = {

    "Confidence":
        CONFIDENCE_FEATURES,

    "Final verification":
        FINAL_FEATURES,

    "Reasoning":
        REASONING_FEATURES,

    "Answer-level combined":
        ANSWER_LEVEL_FEATURES,

    "Full 17":
        FULL_FEATURES,
}


assert len(FULL_FEATURES) == 17



with open(
    INPUT_JSON,
    "r",
    encoding="utf-8",
) as f:

    rows = json.load(f)


assert len(rows) == 1000


print(
    "Loaded:",
    len(rows)
)


print(
    "dev:",
    sum(
        r["split"] == "dev"
        for r in rows
    )
)


print(
    "test:",
    sum(
        r["split"] == "test"
        for r in rows
    )
)



OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)


if not OPENAI_API_KEY:

    try:

        from google.colab import userdata

        OPENAI_API_KEY = userdata.get(
            "OPENAI_API_KEY"
        )

    except Exception:

        OPENAI_API_KEY = None


if not OPENAI_API_KEY:

    raise RuntimeError(
        "OPENAI_API_KEY not found."
    )


client = OpenAI(
    api_key=OPENAI_API_KEY
)



def normalize_answer(text):

    text = str(
        text
    ).lower()


    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text,
    )


    text = re.sub(
        r"\b(a|an|the)\b",
        " ",
        text,
    )


    text = re.sub(
        r"\s+",
        " ",
        text,
    )


    return text.strip()



JUDGE_INSTRUCTIONS = """
You are an offline correctness evaluator for a question-answering experiment.

You receive:
1. a question,
2. the reference answer,
3. the model's predicted answer.

Determine whether the model prediction should be counted as correct.

Rules:

- Accept semantically equivalent paraphrases.
- Accept aliases, abbreviations, spelling variants, capitalization variants,
  and harmless formatting differences.
- Accept a slightly longer prediction if it preserves the same required fact.
- Do not require exact string matching.
- Reject factually different answers.
- Reject answers to a different relation.
- Reject answers that are only related to the reference.
- Reject broader or narrower answers when that changes what the question asks.
- For comparison questions, preserve comparison direction.
- For yes/no questions, the binary proposition must agree.
- Statements such as "unknown", "not provided", "cannot determine", or
  evidence-insufficiency explanations are incorrect unless they genuinely
  express the same answer as the reference answer.

Judge ONLY correctness of the final prediction.

Do not use reasoning, confidence, retrieval scores, or reliability features.

Return a binary correctness judgement.
""".strip()


JUDGE_SCHEMA = {

    "type":
        "object",

    "properties": {

        "correct": {
            "type":
                "boolean"
        },

        "reason": {
            "type":
                "string"
        },
    },

    "required": [
        "correct",
        "reason",
    ],

    "additionalProperties":
        False,
}


def judge_answer(
    question,
    gold,
    prediction,
):

    prompt = f"""
QUESTION:
{question}

REFERENCE ANSWER:
{gold}

MODEL PREDICTION:
{prediction}
""".strip()


    last_error = None


    for attempt in range(
        MAX_RETRIES
    ):

        try:

            response = client.responses.create(

                model=
                    JUDGE_MODEL,

                instructions=
                    JUDGE_INSTRUCTIONS,

                input=
                    prompt,

                temperature=
                    0,

                max_output_tokens=
                    120,

                store=
                    False,

                text={
                    "format": {

                        "type":
                            "json_schema",

                        "name":
                            "qa_correctness_judgment",

                        "strict":
                            True,

                        "schema":
                            JUDGE_SCHEMA,
                    }
                },
            )


            result = json.loads(
                response.output_text
            )


            return {

                "correct":
                    bool(
                        result[
                            "correct"
                        ]
                    ),

                "reason":
                    str(
                        result[
                            "reason"
                        ]
                    ).strip(),
            }


        except Exception as exc:

            last_error = exc


            if (
                attempt
                <
                MAX_RETRIES - 1
            ):

                time.sleep(
                    2 ** attempt
                )


    raise RuntimeError(
        "Judge failed.\n"
        f"Q={question}\n"
        f"GOLD={gold}\n"
        f"PRED={prediction}\n"
        f"ERROR={last_error}"
    )



if os.path.exists(
    JUDGE_PROGRESS
):

    with open(
        JUDGE_PROGRESS,
        "r",
        encoding="utf-8",
    ) as f:

        judged = json.load(f)

else:

    judged = {}


print()
print(
    "Existing judgments:",
    len(judged)
)



print()
print("=" * 80)
print("2WIKI OFFLINE CORRECTNESS JUDGE")
print("=" * 80)


for row in tqdm(
    rows
):

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    if qi in judged:
        continue


    question = str(
        row[
            "question"
        ]
    ).strip()


    gold = str(
        row[
            "gold_answer"
        ]
    ).strip()


    prediction = str(
        row[
            "majority_answer"
        ]
    ).strip()


    result = judge_answer(
        question,
        gold,
        prediction,
    )


    judged[
        qi
    ] = {

        "question_index":
            int(qi),

        "question":
            question,

        "gold_answer":
            gold,

        "majority_answer":
            prediction,

        "judge_correct":
            bool(
                result[
                    "correct"
                ]
            ),

        "judge_reason":
            result[
                "reason"
            ],

        "normalized_exact_match":
            (
                normalize_answer(
                    prediction
                )
                ==
                normalize_answer(
                    gold
                )
            ),

        "judge_model":
            JUDGE_MODEL,
    }


    if (
        len(judged)
        %
        SAVE_EVERY
        ==
        0
    ):

        with open(
            JUDGE_PROGRESS,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                judged,
                f,
                indent=2,
                ensure_ascii=False,
            )


        print(
            f"{len(judged)}/1000 saved"
        )


with open(
    JUDGE_PROGRESS,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        judged,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert len(judged) == 1000



ordered_judgments = [

    judged[
        str(
            int(
                row[
                    "question_index"
                ]
            )
        )
    ]

    for row in rows
]


with open(
    JUDGE_OUTPUT,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        ordered_judgments,
        f,
        indent=2,
        ensure_ascii=False,
    )



labelled_rows = []


for row in rows:

    qi = str(
        int(
            row[
                "question_index"
            ]
        )
    )


    new_row = dict(
        row
    )


    new_row[
        "judge_correct"
    ] = bool(
        judged[
            qi
        ][
            "judge_correct"
        ]
    )


    new_row[
        "judge_reason"
    ] = judged[
        qi
    ][
        "judge_reason"
    ]


    labelled_rows.append(
        new_row
    )


with open(
    LABELLED_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        labelled_rows,
        f,
        indent=2,
        ensure_ascii=False,
    )



csv_rows = []


for row in labelled_rows:

    csv_rows.append(
        {

            "question_index":
                row[
                    "question_index"
                ],

            "split":
                row[
                    "split"
                ],

            "question":
                row[
                    "question"
                ],

            "gold_answer":
                row[
                    "gold_answer"
                ],

            "majority_answer":
                row[
                    "majority_answer"
                ],

            "judge_correct":
                int(
                    row[
                        "judge_correct"
                    ]
                ),

            "is_wrong":
                int(
                    not row[
                        "judge_correct"
                    ]
                ),

            **row[
                "features"
            ],
        }
    )


df = pd.DataFrame(
    csv_rows
)


df.to_csv(
    LABELLED_CSV,
    index=False,
)



dev_df = (
    df[
        df[
            "split"
        ]
        ==
        "dev"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


test_df = (
    df[
        df[
            "split"
        ]
        ==
        "test"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert len(dev_df) == 800
assert len(test_df) == 200


n_correct = int(
    df[
        "judge_correct"
    ].sum()
)


print()
print("=" * 80)
print("2WIKI CORRECTNESS SUMMARY")
print("=" * 80)


print(
    "Total:",
    len(df)
)


print(
    "Correct:",
    n_correct
)


print(
    "Wrong:",
    len(df) - n_correct
)


print(
    "Overall accuracy:",
    round(
        float(
            df[
                "judge_correct"
            ].mean()
        ),
        4,
    )
)


print()
print(
    "DEV:"
)

print(
    "correct:",
    int(
        dev_df[
            "judge_correct"
        ].sum()
    )
)

print(
    "wrong:",
    int(
        dev_df[
            "is_wrong"
        ].sum()
    )
)

print(
    "accuracy:",
    round(
        float(
            dev_df[
                "judge_correct"
            ].mean()
        ),
        4,
    )
)


print()
print(
    "TEST:"
)

print(
    "correct:",
    int(
        test_df[
            "judge_correct"
        ].sum()
    )
)

print(
    "wrong:",
    int(
        test_df[
            "is_wrong"
        ].sum()
    )
)

print(
    "accuracy:",
    round(
        float(
            test_df[
                "judge_correct"
            ].mean()
        ),
        4,
    )
)


exact_agreement = sum(

    bool(
        x[
            "normalized_exact_match"
        ]
    )
    ==
    bool(
        x[
            "judge_correct"
        ]
    )

    for x
    in ordered_judgments
)


print(
    "\nJudge/exact-match agreement:",
    exact_agreement,
    "/1000"
)


dev = dev_df


y = (
    dev[
        "is_wrong"
    ]
    .astype(int)
    .to_numpy()
)


print()
print("=" * 80)
print("2WIKI EXPERIMENT 1 — LABEL DISTRIBUTION")
print("=" * 80)


print(
    "N:",
    len(y)
)


print(
    "Correct:",
    int(
        (y == 0).sum()
    )
)


print(
    "Wrong:",
    int(
        (y == 1).sum()
    )
)


print(
    "Error rate:",
    round(
        float(
            y.mean()
        ),
        4,
    )
)


for feature in FULL_FEATURES:

    if feature not in dev.columns:

        raise RuntimeError(
            f"Missing feature: {feature}"
        )


X_check = (
    dev[
        FULL_FEATURES
    ]
    .astype(float)
    .to_numpy()
)


assert np.all(
    np.isfinite(
        X_check
    )
)


print(
    "All 17 dev features finite: YES"
)


skf = StratifiedKFold(

    n_splits=
        N_SPLITS,

    shuffle=
        True,

    random_state=
        SEED,
)


folds = list(

    skf.split(
        np.zeros(
            len(y)
        ),
        y,
    )
)


def make_classifier():

    return Pipeline(
        [

            (
                "scaler",

                StandardScaler(),
            ),

            (
                "classifier",

                LogisticRegression(

                    class_weight=
                        "balanced",

                    random_state=
                        SEED,

                    max_iter=
                        5000,

                    solver=
                        "lbfgs",
                ),
            ),
        ]
    )




def get_oof(
    X,
    y,
):

    oof = np.full(
        len(y),
        np.nan,
    )


    fold_aurocs = []


    for train_idx, val_idx in folds:

        model = make_classifier()


        model.fit(
            X[
                train_idx
            ],
            y[
                train_idx
            ],
        )


        risk = (
            model
            .predict_proba(
                X[
                    val_idx
                ]
            )
            [:, 1]
        )


        oof[
            val_idx
        ] = risk


        fold_aurocs.append(

            float(
                roc_auc_score(
                    y[
                        val_idx
                    ],
                    risk,
                )
            )
        )


    assert np.all(
        np.isfinite(
            oof
        )
    )


    overall = float(
        roc_auc_score(
            y,
            oof,
        )
    )


    return (
        oof,
        fold_aurocs,
        overall,
    )



def bootstrap_auc_ci(
    y,
    scores,
    n_bootstrap=N_BOOTSTRAP,
):

    rng = (
        np.random.default_rng(
            SEED
        )
    )


    aucs = []


    for _ in range(
        n_bootstrap
    ):

        idx = rng.integers(
            0,
            len(y),
            size=len(y),
        )


        if (
            np.unique(
                y[
                    idx
                ]
            ).size
            <
            2
        ):

            continue


        aucs.append(

            roc_auc_score(
                y[
                    idx
                ],
                scores[
                    idx
                ],
            )
        )


    return (

        float(
            np.percentile(
                aucs,
                2.5,
            )
        ),

        float(
            np.percentile(
                aucs,
                97.5,
            )
        ),
    )



results = {}

oof_predictions = {}


print()
print("=" * 80)
print("2WIKI EXPERIMENT 1 — GROUPED OOF AUROC")
print("=" * 80)


for name, features in (
    FEATURE_GROUPS.items()
):

    X = (
        dev[
            features
        ]
        .astype(float)
        .to_numpy()
    )


    (
        risk,
        fold_auc,
        overall_auc,
    ) = get_oof(
        X,
        y,
    )


    ci_low, ci_high = (
        bootstrap_auc_ci(
            y,
            risk,
        )
    )


    oof_predictions[
        name
    ] = risk


    results[
        name
    ] = {

        "n_features":
            len(features),

        "features":
            features,

        "oof_auroc":
            overall_auc,

        "ci95_low":
            ci_low,

        "ci95_high":
            ci_high,

        "fold_aurocs":
            fold_auc,

        "mean_fold_auroc":
            float(
                np.mean(
                    fold_auc
                )
            ),

        "std_fold_auroc":
            float(
                np.std(
                    fold_auc,
                    ddof=1,
                )
            ),
    }


    print()
    print(
        name
    )

    print(
        "features:",
        len(features)
    )

    print(
        "OOF AUROC:",
        round(
            overall_auc,
            4,
        )
    )

    print(
        "95% CI:",
        (
            round(
                ci_low,
                4,
            ),
            round(
                ci_high,
                4,
            ),
        )
    )

    print(
        "fold AUROCs:",
        [
            round(
                x,
                4,
            )
            for x in fold_auc
        ]
    )



answer_scores = (
    oof_predictions[
        "Answer-level combined"
    ]
)


full_scores = (
    oof_predictions[
        "Full 17"
    ]
)


auc_answer = (
    roc_auc_score(
        y,
        answer_scores,
    )
)


auc_full = (
    roc_auc_score(
        y,
        full_scores,
    )
)


observed_delta = (
    auc_full
    -
    auc_answer
)


rng = np.random.default_rng(
    SEED
)


deltas = []


for _ in range(
    PAIRED_BOOTSTRAP
):

    idx = rng.integers(
        0,
        len(y),
        size=len(y),
    )


    if (
        np.unique(
            y[
                idx
            ]
        ).size
        <
        2
    ):

        continue


    delta = (

        roc_auc_score(
            y[
                idx
            ],
            full_scores[
                idx
            ],
        )

        -

        roc_auc_score(
            y[
                idx
            ],
            answer_scores[
                idx
            ],
        )
    )


    deltas.append(
        delta
    )


deltas = np.asarray(
    deltas
)


delta_low = float(
    np.percentile(
        deltas,
        2.5,
    )
)


delta_high = float(
    np.percentile(
        deltas,
        97.5,
    )
)


p_not_better = float(
    np.mean(
        deltas <= 0
    )
)


p_two_sided = min(

    1.0,

    2
    *
    min(

        np.mean(
            deltas <= 0
        ),

        np.mean(
            deltas >= 0
        ),
    ),
)


print()
print("=" * 80)
print("2WIKI PAIRED RQ1 COMPARISON")
print("=" * 80)


print(
    "Answer-level AUROC:",
    round(
        auc_answer,
        4,
    )
)


print(
    "Full-17 AUROC:",
    round(
        auc_full,
        4,
    )
)


print(
    "Observed delta:",
    round(
        observed_delta,
        4,
    )
)


print(
    "95% paired bootstrap CI:",
    (
        round(
            delta_low,
            4,
        ),
        round(
            delta_high,
            4,
        ),
    )
)


print(
    "P(delta <= 0):",
    round(
        p_not_better,
        4,
    )
)


print(
    "Approx. two-sided p:",
    round(
        p_two_sided,
        4,
    )
)


full_oof = (
    oof_predictions[
        "Full 17"
    ]
)


threshold = float(
    np.percentile(
        full_oof,
        RISK_PERCENTILE,
    )
)


answer_mask = (
    full_oof
    <=
    threshold
)


abstain_mask = (
    ~answer_mask
)


correct = (
    1 - y
)


selective_accuracy = float(
    correct[
        answer_mask
    ].mean()
)


print()
print("=" * 80)
print("2WIKI FULL-17 FROZEN RISK THRESHOLD")
print("=" * 80)


print(
    "Percentile:",
    RISK_PERCENTILE
)


print(
    "Risk threshold:",
    round(
        threshold,
        6,
    )
)


print(
    "OOF development answered:",
    int(
        answer_mask.sum()
    )
)


print(
    "OOF development abstained:",
    int(
        abstain_mask.sum()
    )
)


print(
    "OOF development coverage:",
    round(
        float(
            answer_mask.mean()
        ),
        4,
    )
)


print(
    "OOF selective accuracy:",
    round(
        selective_accuracy,
        4,
    )
)


print(
    "Wrong answers returned:",
    int(
        y[
            answer_mask
        ].sum()
    )
)


print(
    "Correct answers abstained:",
    int(
        correct[
            abstain_mask
        ].sum()
    )
)



oof_df = dev[
    [
        "question_index",
        "question",
        "gold_answer",
        "majority_answer",
        "judge_correct",
        "is_wrong",
    ]
].copy()


for name, values in (
    oof_predictions.items()
):

    column = (
        "risk_"
        +
        name
        .lower()
        .replace(
            " ",
            "_",
        )
        .replace(
            "-",
            "_",
        )
    )


    oof_df[
        column
    ] = values


oof_df[
    "full17_answer"
] = (
    answer_mask.astype(
        int
    )
)


oof_df.to_csv(
    OOF_CSV,
    index=False,
)



output = {

    "dataset":
        "2WikiMultiHopQA",

    "development_n":
        800,

    "correct":
        int(
            (y == 0).sum()
        ),

    "wrong":
        int(
            (y == 1).sum()
        ),

    "cv": {

        "n_splits":
            N_SPLITS,

        "shuffle":
            True,

        "random_state":
            SEED,
    },

    "classifier": {

        "type":
            "logistic_regression",

        "standardized":
            True,

        "class_weight":
            "balanced",
    },

    "target":
        "error",

    "feature_group_results":
        results,

    "paired_full_vs_answer_level": {

        "answer_level_auc":
            float(
                auc_answer
            ),

        "full17_auc":
            float(
                auc_full
            ),

        "delta":
            float(
                observed_delta
            ),

        "ci95_low":
            delta_low,

        "ci95_high":
            delta_high,

        "p_delta_le_zero":
            p_not_better,

        "approx_two_sided_p":
            p_two_sided,
    },

    "full17_threshold": {

        "source":
            "OOF development error risk",

        "percentile":
            RISK_PERCENTILE,

        "threshold":
            threshold,
    },
}


with open(
    EXP1_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        output,
        f,
        indent=2,
    )




summary = pd.DataFrame(
    [

        {

            "Feature group":
                name,

            "Features":
                result[
                    "n_features"
                ],

            "OOF AUROC":
                result[
                    "oof_auroc"
                ],

            "95% CI low":
                result[
                    "ci95_low"
                ],

            "95% CI high":
                result[
                    "ci95_high"
                ],
        }

        for name, result
        in results.items()
    ]
)


print()
print("=" * 80)
print("2WIKI FINAL EXPERIMENT 1 TABLE")
print("=" * 80)


print(
    summary.to_string(

        index=False,

        formatters={

            "OOF AUROC":
                lambda x:
                    f"{x:.4f}",

            "95% CI low":
                lambda x:
                    f"{x:.4f}",

            "95% CI high":
                lambda x:
                    f"{x:.4f}",
        },
    )
)


print()
print(
    "Saved labelled CSV:"
)

print(
    LABELLED_CSV
)


print()
print(
    "Saved OOF:"
)

print(
    OOF_CSV
)


print()
print(
    "Saved Experiment 1:"
)

print(
    EXP1_JSON
)


print()
print("DONE.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 110.1 MB/s eta 0:00:00
Loaded: 1000
dev: 800
test: 200

Existing judgments: 0

2WIKI OFFLINE CORRECTNESS JUDGE


  0%|          | 0/1000 [00:00<?, ?it/s]

10/1000 saved
20/1000 saved
30/1000 saved
40/1000 saved
50/1000 saved
60/1000 saved
70/1000 saved
80/1000 saved
90/1000 saved
100/1000 saved
110/1000 saved
120/1000 saved
130/1000 saved
140/1000 saved
150/1000 saved
160/1000 saved
170/1000 saved
180/1000 saved
190/1000 saved
200/1000 saved
210/1000 saved
220/1000 saved
230/1000 saved
240/1000 saved
250/1000 saved
260/1000 saved
270/1000 saved
280/1000 saved
290/1000 saved
300/1000 saved
310/1000 saved
320/1000 saved
330/1000 saved
340/1000 saved
350/1000 saved
360/1000 saved
370/1000 saved
380/1000 saved
390/1000 saved
400/1000 saved
410/1000 saved
420/1000 saved
430/1000 saved
440/1000 saved
450/1000 saved
460/1000 saved
470/1000 saved
480/1000 saved
490/1000 saved
500/1000 saved
510/1000 saved
520/1000 saved
530/1000 saved
540/1000 saved
550/1000 saved
560/1000 saved
570/1000 saved
580/1000 saved
590/1000 saved
600/1000 saved
610/1000 saved
620/1000 saved
630/1000 saved
640/1000 saved
650/1000 saved
660/1000 saved
670/1000 saved
680/

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


D = "/content/drive/MyDrive/hedge_run"

LABELLED_CSV = (
    f"{D}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
)

FINAL_JSON = (
    f"{D}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_LABELLED_1000.json"
)

OOF_CSV = (
    f"{D}/"
    "hotpot_experiment1_oof_predictions.csv"
)

EXP1_JSON = (
    f"{D}/"
    "hotpot_experiment1_grouped_auroc.json"
)



REASONING = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]

CONFIDENCE = [
    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]

FINAL_VERIFICATION = [
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]

ANSWER_LEVEL = (
    CONFIDENCE
    +
    FINAL_VERIFICATION
)

FULL17 = (
    REASONING
    +
    CONFIDENCE
    +
    FINAL_VERIFICATION
)

GROUPS = {
    "Confidence": CONFIDENCE,
    "Final verification": FINAL_VERIFICATION,
    "Reasoning": REASONING,
    "Answer-level combined": ANSWER_LEVEL,
    "Full 17": FULL17,
}



df = pd.read_csv(
    LABELLED_CSV
)

oof = pd.read_csv(
    OOF_CSV
)

with open(
    FINAL_JSON,
    encoding="utf-8"
) as f:
    records = json.load(f)

with open(
    EXP1_JSON,
    encoding="utf-8"
) as f:
    saved_exp = json.load(f)


print("=" * 80)
print("1. BASIC DATA INTEGRITY")
print("=" * 80)

print("CSV rows:", len(df))
print("JSON rows:", len(records))
print("OOF rows:", len(oof))

assert len(df) == 1000
assert len(records) == 1000
assert len(oof) == 800

print()
print(df["split"].value_counts())

assert (df["split"] == "dev").sum() == 800
assert (df["split"] == "test").sum() == 200

assert df["question_index"].nunique() == 1000
assert oof["question_index"].nunique() == 800

print("\nUnique question indices: YES")



dev_ids = set(
    df.loc[
        df["split"] == "dev",
        "question_index"
    ].astype(int)
)

test_ids = set(
    df.loc[
        df["split"] == "test",
        "question_index"
    ].astype(int)
)

intersection = (
    dev_ids
    &
    test_ids
)

print()
print("DEV/TEST overlap:", len(intersection))

assert len(intersection) == 0



print()
print("=" * 80)
print("2. LABEL INTEGRITY")
print("=" * 80)

assert np.all(
    df["is_wrong"].astype(int).to_numpy()
    ==
    (
        1
        -
        df["judge_correct"].astype(int).to_numpy()
    )
)

print(
    "is_wrong = 1 - judge_correct: YES"
)

dev = (
    df[
        df["split"] == "dev"
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "DEV correct:",
    int(dev["judge_correct"].sum())
)

print(
    "DEV wrong:",
    int(dev["is_wrong"].sum())
)

print(
    "DEV accuracy:",
    round(
        float(
            dev["judge_correct"].mean()
        ),
        4,
    )
)

assert int(dev["judge_correct"].sum()) == 505
assert int(dev["is_wrong"].sum()) == 295



print()
print("=" * 80)
print("3. FEATURE INTEGRITY")
print("=" * 80)

for feature in FULL17:

    assert feature in df.columns

    values = (
        pd.to_numeric(
            df[feature],
            errors="raise"
        )
        .to_numpy(dtype=float)
    )

    assert np.all(
        np.isfinite(values)
    )


print("All 17 features finite: YES")


bounded_01 = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
    "self_consistency",
    "answer_norm_entropy",
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]

bad_ranges = {}

for col in bounded_01:

    v = df[col].astype(float)

    if (
        (v < -1e-8).any()
        or
        (v > 1 + 1e-8).any()
    ):
        bad_ranges[col] = (
            float(v.min()),
            float(v.max())
        )


print(
    "Features outside expected [0,1] range:",
    bad_ranges
)

assert not bad_ranges


nonzero = (
    df["num_steps"]
    >
    0
)

fraction_sum = (
    df.loc[
        nonzero,
        [
            "frac_supported",
            "frac_contradicted",
            "frac_unclear",
        ]
    ]
    .sum(axis=1)
)

max_fraction_error = float(
    np.abs(
        fraction_sum
        -
        1.0
    ).max()
)

print(
    "Maximum reasoning-label fraction error:",
    max_fraction_error
)

assert max_fraction_error < 1e-6



zero = (
    df["num_steps"]
    ==
    0
)

print(
    "Zero-step trajectories:",
    int(zero.sum())
)

if zero.any():

    print(
        "Zero-step mean frac_unclear:",
        float(
            df.loc[
                zero,
                "frac_unclear"
            ].mean()
        )
    )



print()
print("=" * 80)
print("4. YES/NO PATCH INTEGRITY")
print("=" * 80)

PATCH_IDS = {
    260,
    267,
    441,
    495,
    522,
    784,
    795,
    830,
    874,
    949,
    981,
    983,
}

by_id = {
    int(r["question_index"]): r
    for r in records
}

missing_patch_ids = (
    PATCH_IDS
    -
    set(by_id)
)

assert not missing_patch_ids

for qi in sorted(PATCH_IDS):

    pred = str(
        by_id[qi]["majority_answer"]
    ).strip().lower()

    if pred in {"yes", "no"}:

        raise RuntimeError(
            f"Patched QI {qi} still has Yes/No prediction."
        )


print(
    "All 12 patched predictions are non-Yes/No: YES"
)



print()
print("=" * 80)
print("5. FINAL CLAIM INTEGRITY")
print("=" * 80)

critical_claims = {
    2: "actor common",
    17: "1969 satire novel",
    281: "adopted the name",
    495: "biggest commercial success was on",
}

for qi, phrase in critical_claims.items():

    claim = str(
        by_id[qi]["final_claim"]
    )

    assert (
        phrase.lower()
        in
        claim.lower()
    )


claim_784 = str(
    by_id[784]["final_claim"]
).strip()

print(
    "QI784 claim:",
    claim_784
)

assert (
    claim_784
    ==
    "Vertical Horizon started their band first."
)

print(
    "Critical corrected claims present: YES"
)



print()
print("=" * 80)
print("6. LEAKAGE CHECK")
print("=" * 80)

for forbidden in [
    "judge_correct",
    "is_wrong",
    "gold_answer",
]:

    assert forbidden not in FULL17


print(
    "Gold/correctness labels absent from 17-feature vector: YES"
)



print()
print("=" * 80)
print("7. OOF ALIGNMENT")
print("=" * 80)

dev_by_id = (
    dev
    .set_index(
        "question_index"
    )
)

oof_by_id = (
    oof
    .set_index(
        "question_index"
    )
)

assert set(
    dev_by_id.index.astype(int)
) == set(
    oof_by_id.index.astype(int)
)

aligned = (
    dev_by_id
    .loc[
        oof_by_id.index
    ]
)

assert np.all(
    aligned[
        "is_wrong"
    ].astype(int).to_numpy()
    ==
    oof_by_id[
        "is_wrong"
    ].astype(int).to_numpy()
)

print(
    "OOF predictions align to the correct development labels: YES"
)



print()
print("=" * 80)
print("8. SAVED OOF AUROC RECOMPUTATION")
print("=" * 80)

y_oof = (
    oof[
        "is_wrong"
    ]
    .astype(int)
    .to_numpy()
)


OOF_COLUMNS = {
    "Confidence":
        "risk_confidence",

    "Final verification":
        "risk_final_verification",

    "Reasoning":
        "risk_reasoning",

    "Answer-level combined":
        "risk_answer_level_combined",

    "Full 17":
        "risk_full_17",
}


recomputed_auc = {}


for name, col in OOF_COLUMNS.items():

    auc = float(
        roc_auc_score(
            y_oof,
            oof[col].to_numpy()
        )
    )

    recomputed_auc[name] = auc

    saved_auc = float(
        saved_exp[
            "feature_group_results"
        ][
            name
        ][
            "oof_auroc"
        ]
    )

    print(
        f"{name:24s}",
        f"recomputed={auc:.6f}",
        f"saved={saved_auc:.6f}",
        f"diff={abs(auc-saved_auc):.10f}"
    )

    assert abs(
        auc
        -
        saved_auc
    ) < 1e-10


print(
    "\nSaved AUROCs reproduce exactly: YES"
)



print()
print("=" * 80)
print("9. FULL OOF RECOMPUTATION FROM FEATURES")
print("=" * 80)

SEED = 42

y = (
    dev[
        "is_wrong"
    ]
    .astype(int)
    .to_numpy()
)


skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)


folds = list(
    skf.split(
        np.zeros(len(y)),
        y,
    )
)


def make_model():

    return Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    random_state=SEED,
                    max_iter=5000,
                    solver="lbfgs",
                ),
            ),
        ]
    )


fresh_oof = {}


for name, features in GROUPS.items():

    X = (
        dev[
            features
        ]
        .astype(float)
        .to_numpy()
    )

    scores = np.full(
        len(dev),
        np.nan
    )

    fold_aucs = []


    for fold_id, (
        train_idx,
        val_idx
    ) in enumerate(
        folds,
        start=1
    ):

        # Explicit disjointness check.
        assert len(
            set(train_idx)
            &
            set(val_idx)
        ) == 0


        model = make_model()

        model.fit(
            X[train_idx],
            y[train_idx]
        )

        risk = (
            model.predict_proba(
                X[val_idx]
            )[:, 1]
        )

        scores[val_idx] = risk

        fold_aucs.append(
            roc_auc_score(
                y[val_idx],
                risk
            )
        )


    assert np.all(
        np.isfinite(scores)
    )


    auc = roc_auc_score(
        y,
        scores
    )

    fresh_oof[name] = scores


    print()
    print(name)
    print(
        "pooled OOF AUROC:",
        round(float(auc), 6)
    )

    print(
        "fold AUROCs:",
        [
            round(float(x), 4)
            for x in fold_aucs
        ]
    )

    print(
        "mean fold AUROC:",
        round(
            float(
                np.mean(fold_aucs)
            ),
            6
        )
    )

    print(
        "saved OOF difference:",
        np.max(
            np.abs(
                scores
                -
                oof[
                    OOF_COLUMNS[name]
                ].to_numpy()
            )
        )
    )



print()
print("=" * 80)
print("10. POOLED OOF VS MEAN-FOLD NOTE")
print("=" * 80)

print(
    """
Pooled OOF AUROC and mean fold AUROC do not have to be identical.

Each validation fold is predicted by a separately fitted logistic-regression
model. Probability scales can therefore differ slightly between folds.

For the dissertation, use ONE convention consistently across both datasets.
The current primary metric is pooled OOF AUROC over all 800 development
examples. Fold AUROCs are diagnostics.
""".strip()
)




print()
print("=" * 80)
print("11. HOTPOT PAIRED FULL-vs-ANSWER-LEVEL TEST")
print("=" * 80)

answer_scores = (
    fresh_oof[
        "Answer-level combined"
    ]
)

full_scores = (
    fresh_oof[
        "Full 17"
    ]
)


auc_answer = roc_auc_score(
    y,
    answer_scores
)

auc_full = roc_auc_score(
    y,
    full_scores
)

observed_delta = (
    auc_full
    -
    auc_answer
)


rng = np.random.default_rng(
    42
)

N_BOOT = 10000

deltas = []


for _ in range(
    N_BOOT
):

    idx = rng.integers(
        0,
        len(y),
        size=len(y)
    )

    if np.unique(
        y[idx]
    ).size < 2:
        continue


    delta = (
        roc_auc_score(
            y[idx],
            full_scores[idx]
        )
        -
        roc_auc_score(
            y[idx],
            answer_scores[idx]
        )
    )

    deltas.append(
        delta
    )


deltas = np.asarray(
    deltas
)


ci_low = float(
    np.percentile(
        deltas,
        2.5
    )
)

ci_high = float(
    np.percentile(
        deltas,
        97.5
    )
)

p_le_zero = float(
    np.mean(
        deltas <= 0
    )
)

p_two = min(
    1.0,
    2 * min(
        np.mean(deltas <= 0),
        np.mean(deltas >= 0)
    )
)


print(
    "Answer-level AUROC:",
    round(
        float(auc_answer),
        4
    )
)

print(
    "Full-17 AUROC:",
    round(
        float(auc_full),
        4
    )
)

print(
    "Delta:",
    round(
        float(observed_delta),
        4
    )
)

print(
    "95% paired CI:",
    (
        round(ci_low, 4),
        round(ci_high, 4)
    )
)

print(
    "P(delta <= 0):",
    round(p_le_zero, 4)
)

print(
    "Approx. two-sided p:",
    round(p_two, 4)
)



print()
print("=" * 80)
print("12. BASIC FEATURE/LABEL SANITY")
print("=" * 80)


for col in [
    "fa_support_retr",
    "fa_contradiction_retr",
    "self_consistency",
    "max_contradiction_score",
    "mean_support",
]:

    correct_mean = float(
        dev.loc[
            dev["is_wrong"] == 0,
            col
        ].mean()
    )

    wrong_mean = float(
        dev.loc[
            dev["is_wrong"] == 1,
            col
        ].mean()
    )


    print()
    print(col)
    print(
        " correct mean:",
        round(
            correct_mean,
            4
        )
    )
    print(
        " wrong mean:  ",
        round(
            wrong_mean,
            4
        )
    )



print()
print("=" * 80)
print("13. FINAL VALIDITY STATUS")
print("=" * 80)

assert not any(
    qi in test_ids
    for qi in oof[
        "question_index"
    ].astype(int)
)

print(
    "No test question appears in OOF classifier evaluation: YES"
)

print(
    "Gold/correctness absent from model feature vector: YES"
)

print(
    "Saved OOF predictions exactly reproducible: YES"
)

print()
print("AUDIT COMPLETE.")

1. BASIC DATA INTEGRITY
CSV rows: 1000
JSON rows: 1000
OOF rows: 800

split
dev     800
test    200
Name: count, dtype: int64

Unique question indices: YES

DEV/TEST overlap: 0

2. LABEL INTEGRITY
is_wrong = 1 - judge_correct: YES
DEV correct: 505
DEV wrong: 295
DEV accuracy: 0.6312

3. FEATURE INTEGRITY
All 17 features finite: YES
Features outside expected [0,1] range: {}
Maximum reasoning-label fraction error: 0.0
Zero-step trajectories: 11
Zero-step mean frac_unclear: 1.0

4. YES/NO PATCH INTEGRITY
All 12 patched predictions are non-Yes/No: YES

5. FINAL CLAIM INTEGRITY
QI784 claim: Vertical Horizon started their band first.
Critical corrected claims present: YES

6. LEAKAGE CHECK
Gold/correctness labels absent from 17-feature vector: YES

7. OOF ALIGNMENT
OOF predictions align to the correct development labels: YES

8. SAVED OOF AUROC RECOMPUTATION
Confidence               recomputed=0.579580 saved=0.579580 diff=0.0000000000
Final verification       recomputed=0.736567 saved=0.7365

WHY DOES REASONING HELP MORE ON 2WIKI THAN HOTPOT?

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr


D = "/content/drive/MyDrive/hedge_run"

HOTPOT_PATH = (
    f"{D}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
)

WIKI_PATH = (
    f"{D}/"
    "2wiki_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
)



REASONING = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]


CONFIDENCE = [
    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]


FINAL_VERIFICATION = [
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]


ANSWER_LEVEL = (
    CONFIDENCE
    +
    FINAL_VERIFICATION
)


def load_dev(path):

    df = pd.read_csv(path)

    dev = (
        df[
            df["split"] == "dev"
        ]
        .copy()
        .reset_index(drop=True)
    )

    assert len(dev) == 800

    return dev


hotpot = load_dev(
    HOTPOT_PATH
)

wiki = load_dev(
    WIKI_PATH
)


print(
    "Hotpot dev:",
    len(hotpot)
)

print(
    "2Wiki dev:",
    len(wiki)
)



def analyse_feature(
    df,
    feature,
):

    y = (
        df["is_wrong"]
        .astype(int)
        .to_numpy()
    )

    x = (
        df[feature]
        .astype(float)
        .to_numpy()
    )


    # Constant feature safeguard.
    if np.unique(x).size <= 1:

        raw_auc = 0.5

    else:

        raw_auc = float(
            roc_auc_score(
                y,
                x,
            )
        )


    discrimination = max(
        raw_auc,
        1.0 - raw_auc,
    )


    correct_mean = float(
        x[
            y == 0
        ].mean()
    )


    wrong_mean = float(
        x[
            y == 1
        ].mean()
    )


    if wrong_mean > correct_mean:

        direction = (
            "higher -> wrong"
        )

    elif wrong_mean < correct_mean:

        direction = (
            "higher -> correct"
        )

    else:

        direction = "equal"


    return {

        "feature":
            feature,

        "raw_error_auc":
            raw_auc,

        "discrimination_auc":
            discrimination,

        "correct_mean":
            correct_mean,

        "wrong_mean":
            wrong_mean,

        "direction":
            direction,
    }



hotpot_uni = pd.DataFrame(
    [
        analyse_feature(
            hotpot,
            feature
        )
        for feature
        in REASONING
    ]
)


wiki_uni = pd.DataFrame(
    [
        analyse_feature(
            wiki,
            feature
        )
        for feature
        in REASONING
    ]
)


comparison = (
    hotpot_uni[
        [
            "feature",
            "discrimination_auc",
            "direction",
        ]
    ]
    .rename(
        columns={
            "discrimination_auc":
                "Hotpot AUC",

            "direction":
                "Hotpot direction",
        }
    )
    .merge(

        wiki_uni[
            [
                "feature",
                "discrimination_auc",
                "direction",
            ]
        ]
        .rename(
            columns={
                "discrimination_auc":
                    "2Wiki AUC",

                "direction":
                    "2Wiki direction",
            }
        ),

        on="feature",
    )
)


comparison[
    "2Wiki - Hotpot"
] = (
    comparison[
        "2Wiki AUC"
    ]
    -
    comparison[
        "Hotpot AUC"
    ]
)


comparison = (
    comparison
    .sort_values(
        "2Wiki AUC",
        ascending=False,
    )
    .reset_index(drop=True)
)


print()
print("=" * 90)
print("INDIVIDUAL REASONING FEATURE DISCRIMINATION")
print("=" * 90)


print(
    comparison.to_string(

        index=False,

        formatters={

            "Hotpot AUC":
                lambda x:
                    f"{x:.4f}",

            "2Wiki AUC":
                lambda x:
                    f"{x:.4f}",

            "2Wiki - Hotpot":
                lambda x:
                    f"{x:+.4f}",
        },
    )
)



def reasoning_answer_correlations(
    df,
):

    rows = []


    for reasoning_feature in REASONING:

        correlations = {}


        for answer_feature in ANSWER_LEVEL:

            rho, _ = spearmanr(

                df[
                    reasoning_feature
                ].astype(float),

                df[
                    answer_feature
                ].astype(float),
            )


            if np.isnan(rho):
                rho = 0.0


            correlations[
                answer_feature
            ] = float(
                rho
            )


        max_feature = max(

            correlations,

            key=lambda key:
                abs(
                    correlations[
                        key
                    ]
                )
        )


        rows.append(
            {

                "reasoning_feature":
                    reasoning_feature,

                "max_abs_answer_corr":
                    abs(
                        correlations[
                            max_feature
                        ]
                    ),

                "most_correlated_with":
                    max_feature,

                "signed_rho":
                    correlations[
                        max_feature
                    ],
            }
        )


    return pd.DataFrame(
        rows
    )


hotpot_corr = (
    reasoning_answer_correlations(
        hotpot
    )
)


wiki_corr = (
    reasoning_answer_correlations(
        wiki
    )
)


corr_comparison = (
    hotpot_corr
    .rename(
        columns={
            "max_abs_answer_corr":
                "Hotpot max |rho|",

            "most_correlated_with":
                "Hotpot closest signal",

            "signed_rho":
                "Hotpot rho",
        }
    )
    .merge(

        wiki_corr.rename(
            columns={
                "max_abs_answer_corr":
                    "2Wiki max |rho|",

                "most_correlated_with":
                    "2Wiki closest signal",

                "signed_rho":
                    "2Wiki rho",
            }
        ),

        on="reasoning_feature",
    )
)


corr_comparison[
    "Hotpot-2Wiki redundancy"
] = (

    corr_comparison[
        "Hotpot max |rho|"
    ]

    -

    corr_comparison[
        "2Wiki max |rho|"
    ]
)


print()
print("=" * 90)
print("REASONING ↔ ANSWER-LEVEL REDUNDANCY")
print("=" * 90)


print(
    corr_comparison
    .sort_values(
        "Hotpot-2Wiki redundancy",
        ascending=False,
    )
    .to_string(

        index=False,

        formatters={

            "Hotpot max |rho|":
                lambda x:
                    f"{x:.3f}",

            "Hotpot rho":
                lambda x:
                    f"{x:+.3f}",

            "2Wiki max |rho|":
                lambda x:
                    f"{x:.3f}",

            "2Wiki rho":
                lambda x:
                    f"{x:+.3f}",

            "Hotpot-2Wiki redundancy":
                lambda x:
                    f"{x:+.3f}",
        },
    )
)


def cross_corr_matrix(
    df,
):

    matrix = pd.DataFrame(
        index=REASONING,
        columns=ANSWER_LEVEL,
        dtype=float,
    )


    for r in REASONING:

        for a in ANSWER_LEVEL:

            rho, _ = spearmanr(
                df[r].astype(float),
                df[a].astype(float),
            )

            matrix.loc[
                r,
                a
            ] = (
                0.0
                if np.isnan(rho)
                else rho
            )


    return matrix


hotpot_matrix = (
    cross_corr_matrix(
        hotpot
    )
)


wiki_matrix = (
    cross_corr_matrix(
        wiki
    )
)


print()
print("=" * 90)
print("HOTPOT REASONING ↔ ANSWER-LEVEL SPEARMAN")
print("=" * 90)

print(
    hotpot_matrix.round(3)
)


print()
print("=" * 90)
print("2WIKI REASONING ↔ ANSWER-LEVEL SPEARMAN")
print("=" * 90)

print(
    wiki_matrix.round(3)
)


def error_correlations(
    df,
):

    y = (
        df[
            "is_wrong"
        ]
        .astype(float)
    )


    out = []


    for feature in REASONING:

        rho, _ = spearmanr(
            df[
                feature
            ].astype(float),
            y,
        )


        if np.isnan(rho):
            rho = 0.0


        out.append(
            {
                "feature":
                    feature,

                "rho_error":
                    float(
                        rho
                    ),
            }
        )


    return pd.DataFrame(
        out
    )


h_error = (
    error_correlations(
        hotpot
    )
    .rename(
        columns={
            "rho_error":
                "Hotpot rho(error)"
        }
    )
)


w_error = (
    error_correlations(
        wiki
    )
    .rename(
        columns={
            "rho_error":
                "2Wiki rho(error)"
        }
    )
)


error_comparison = (
    h_error.merge(
        w_error,
        on="feature",
    )
)


print()
print("=" * 90)
print("REASONING FEATURE CORRELATION WITH ERROR")
print("=" * 90)


print(
    error_comparison.to_string(

        index=False,

        formatters={

            "Hotpot rho(error)":
                lambda x:
                    f"{x:+.3f}",

            "2Wiki rho(error)":
                lambda x:
                    f"{x:+.3f}",
        },
    )
)



print()
print("=" * 90)
print("DATASET-LEVEL SUMMARY")
print("=" * 90)


print(
    "Mean individual reasoning discrimination AUROC"
)


print(
    "Hotpot:",
    round(
        float(
            comparison[
                "Hotpot AUC"
            ].mean()
        ),
        4,
    )
)


print(
    "2Wiki:",
    round(
        float(
            comparison[
                "2Wiki AUC"
            ].mean()
        ),
        4,
    )
)


print()
print(
    "Mean maximum |correlation| with answer-level features"
)


print(
    "Hotpot:",
    round(
        float(
            hotpot_corr[
                "max_abs_answer_corr"
            ].mean()
        ),
        4,
    )
)


print(
    "2Wiki:",
    round(
        float(
            wiki_corr[
                "max_abs_answer_corr"
            ].mean()
        ),
        4,
    )
)


print()
print("DONE.")

Hotpot dev: 800
2Wiki dev: 800

INDIVIDUAL REASONING FEATURE DISCRIMINATION
                feature Hotpot AUC  Hotpot direction 2Wiki AUC   2Wiki direction 2Wiki - Hotpot
              num_steps     0.5073   higher -> wrong    0.6887 higher -> correct        +0.1814
               conflict     0.5008   higher -> wrong    0.6582 higher -> correct        +0.1573
         support_spread     0.5332   higher -> wrong    0.6332 higher -> correct        +0.1001
max_contradiction_score     0.5532   higher -> wrong    0.6285 higher -> correct        +0.0753
         frac_supported     0.6087 higher -> correct    0.5553 higher -> correct        -0.0533
           mean_support     0.6143 higher -> correct    0.5536 higher -> correct        -0.0607
     mean_contradiction     0.5498   higher -> wrong    0.5352 higher -> correct        -0.0146
      frac_contradicted     0.5796   higher -> wrong    0.5244   higher -> wrong        -0.0551
      min_support_score     0.6149 higher -> correct    0.52

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score


D = "/content/drive/MyDrive/hedge_run"

HOTPOT_PATH = (
    f"{D}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
)

WIKI_PATH = (
    f"{D}/"
    "2wiki_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
)




REASONING = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]


CONFIDENCE = [
    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]


FINAL_VERIFICATION = [
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]


ANSWER_LEVEL = (
    CONFIDENCE
    +
    FINAL_VERIFICATION
)


REASONING_NO_LENGTH = [
    x
    for x in REASONING
    if x != "num_steps"
]



SEMANTIC_REASONING = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
]


FULL17 = (
    ANSWER_LEVEL
    +
    REASONING
)


GROUPS = {

    "Answer-level":
        ANSWER_LEVEL,

    "Answer-level + num_steps":
        ANSWER_LEVEL
        +
        ["num_steps"],

    "Answer-level + reasoning (no num_steps)":
        ANSWER_LEVEL
        +
        REASONING_NO_LENGTH,

    "Answer-level + semantic reasoning":
        ANSWER_LEVEL
        +
        SEMANTIC_REASONING,

    "Full 17":
        FULL17,
}

def load_dev(path):

    df = pd.read_csv(path)

    dev = (
        df[
            df["split"] == "dev"
        ]
        .copy()
        .reset_index(drop=True)
    )

    assert len(dev) == 800

    return dev


datasets = {

    "Hotpot":
        load_dev(
            HOTPOT_PATH
        ),

    "2Wiki":
        load_dev(
            WIKI_PATH
        ),
}



SEED = 42

N_SPLITS = 5


def make_model():

    return Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),

            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    random_state=SEED,
                    max_iter=5000,
                    solver="lbfgs",
                ),
            ),
        ]
    )


def make_folds(y):

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED,
    )

    return list(
        skf.split(
            np.zeros(
                len(y)
            ),
            y,
        )
    )



def oof_scores(
    df,
    features,
):

    y = (
        df[
            "is_wrong"
        ]
        .astype(int)
        .to_numpy()
    )


    X = (
        df[
            features
        ]
        .astype(float)
        .to_numpy()
    )


    folds = make_folds(
        y
    )


    oof = np.full(
        len(y),
        np.nan,
    )


    fold_aucs = []


    for train_idx, val_idx in folds:

        model = make_model()

        model.fit(
            X[train_idx],
            y[train_idx],
        )

        risk = (
            model.predict_proba(
                X[val_idx]
            )[:, 1]
        )

        oof[
            val_idx
        ] = risk


        fold_aucs.append(
            roc_auc_score(
                y[val_idx],
                risk,
            )
        )


    assert np.all(
        np.isfinite(
            oof
        )
    )


    return (
        y,
        oof,
        float(
            roc_auc_score(
                y,
                oof,
            )
        ),
        fold_aucs,
    )


def paired_delta_ci(
    y,
    baseline,
    alternative,
    n_boot=10000,
    seed=42,
):

    observed = (
        roc_auc_score(
            y,
            alternative,
        )
        -
        roc_auc_score(
            y,
            baseline,
        )
    )


    rng = np.random.default_rng(
        seed
    )


    deltas = []


    for _ in range(
        n_boot
    ):

        idx = rng.integers(
            0,
            len(y),
            size=len(y),
        )


        if np.unique(
            y[idx]
        ).size < 2:

            continue


        delta = (
            roc_auc_score(
                y[idx],
                alternative[idx],
            )
            -
            roc_auc_score(
                y[idx],
                baseline[idx],
            )
        )


        deltas.append(
            delta
        )


    deltas = np.asarray(
        deltas
    )


    return {

        "delta":
            float(
                observed
            ),

        "low":
            float(
                np.percentile(
                    deltas,
                    2.5,
                )
            ),

        "high":
            float(
                np.percentile(
                    deltas,
                    97.5,
                )
            ),

        "p_le_zero":
            float(
                np.mean(
                    deltas <= 0
                )
            ),
    }



all_results = {}


for dataset_name, df in (
    datasets.items()
):

    print()
    print("=" * 90)
    print(dataset_name.upper())
    print("=" * 90)


    predictions = {}

    result_rows = []


    for name, features in (
        GROUPS.items()
    ):

        (
            y,
            scores,
            auc,
            fold_aucs,
        ) = oof_scores(
            df,
            features,
        )


        predictions[
            name
        ] = scores


        result_rows.append(
            {

                "Model":
                    name,

                "Features":
                    len(features),

                "OOF AUROC":
                    auc,

                "Mean fold AUROC":
                    float(
                        np.mean(
                            fold_aucs
                        )
                    ),
            }
        )


    result_table = pd.DataFrame(
        result_rows
    )


    print()
    print(
        result_table.to_string(
            index=False,
            formatters={
                "OOF AUROC":
                    lambda x:
                        f"{x:.4f}",

                "Mean fold AUROC":
                    lambda x:
                        f"{x:.4f}",
            },
        )
    )



    baseline = (
        predictions[
            "Answer-level"
        ]
    )


    print()
    print(
        "PAIRED DELTAS VS ANSWER-LEVEL"
    )
    print("-" * 90)


    for name in [
        "Answer-level + num_steps",
        "Answer-level + reasoning (no num_steps)",
        "Answer-level + semantic reasoning",
        "Full 17",
    ]:

        stats = paired_delta_ci(
            y,
            baseline,
            predictions[
                name
            ],
        )


        print()
        print(name)

        print(
            "Delta:",
            f"{stats['delta']:+.4f}"
        )

        print(
            "95% CI:",
            (
                round(
                    stats[
                        "low"
                    ],
                    4,
                ),
                round(
                    stats[
                        "high"
                    ],
                    4,
                ),
            )
        )

        print(
            "P(delta <= 0):",
            round(
                stats[
                    "p_le_zero"
                ],
                4,
            )
        )


    all_results[
        dataset_name
    ] = {
        "table":
            result_table,

        "predictions":
            predictions,
    }



print()
print("=" * 90)
print("ACCURACY BY TRAJECTORY LENGTH")
print("=" * 90)


for dataset_name, df in datasets.items():

    print()
    print(dataset_name)
    print("-" * 70)


    length_table = (
        df
        .groupby(
            "num_steps"
        )
        .agg(
            n=(
                "judge_correct",
                "size",
            ),

            correct=(
                "judge_correct",
                "sum",
            ),

            accuracy=(
                "judge_correct",
                "mean",
            ),
        )
        .reset_index()
    )


    print(
        length_table.to_string(
            index=False,

            formatters={
                "accuracy":
                    lambda x:
                        f"{x:.4f}"
            },
        )
    )



print()
print("=" * 90)
print("CORRELATION WITH NUM_STEPS")
print("=" * 90)


for dataset_name, df in datasets.items():

    print()
    print(dataset_name)
    print("-" * 70)


    correlations = []


    for feature in REASONING:

        if feature == "num_steps":
            continue


        rho = (
            df[
                [
                    "num_steps",
                    feature,
                ]
            ]
            .corr(
                method="spearman"
            )
            .iloc[
                0,
                1
            ]
        )


        correlations.append(
            (
                feature,
                float(
                    rho
                ),
            )
        )


    correlations.sort(
        key=lambda x:
            abs(
                x[1]
            ),
        reverse=True,
    )


    for feature, rho in correlations:

        print(
            f"{feature:28s}",
            f"{rho:+.3f}",
        )




print()
print("=" * 90)
print("INCREMENTAL VALUE OF EACH REASONING FEATURE")
print("=" * 90)


for dataset_name, df in datasets.items():

    print()
    print(dataset_name)
    print("-" * 90)


    y, base_scores, base_auc, _ = (
        oof_scores(
            df,
            ANSWER_LEVEL,
        )
    )


    incremental = []


    for feature in REASONING:

        (
            _,
            scores,
            auc,
            _,
        ) = oof_scores(
            df,
            ANSWER_LEVEL
            +
            [feature],
        )


        incremental.append(
            {

                "feature":
                    feature,

                "AUROC":
                    auc,

                "delta":
                    auc
                    -
                    base_auc,
            }
        )


    inc_df = (
        pd.DataFrame(
            incremental
        )
        .sort_values(
            "delta",
            ascending=False,
        )
    )


    print(
        inc_df.to_string(
            index=False,

            formatters={
                "AUROC":
                    lambda x:
                        f"{x:.4f}",

                "delta":
                    lambda x:
                        f"{x:+.4f}",
            },
        )
    )


print()
print("DONE.")


HOTPOT

                                  Model  Features OOF AUROC Mean fold AUROC
                           Answer-level         6    0.7480          0.7685
               Answer-level + num_steps         7    0.7428          0.7587
Answer-level + reasoning (no num_steps)        16    0.7516          0.7567
      Answer-level + semantic reasoning        14    0.7520          0.7602
                                Full 17        17    0.7511          0.7556

PAIRED DELTAS VS ANSWER-LEVEL
------------------------------------------------------------------------------------------

Answer-level + num_steps
Delta: -0.0052
95% CI: (-0.0168, 0.0064)
P(delta <= 0): 0.8134

Answer-level + reasoning (no num_steps)
Delta: +0.0036
95% CI: (-0.0181, 0.0254)
P(delta <= 0): 0.3654

Answer-level + semantic reasoning
Delta: +0.0040
95% CI: (-0.0175, 0.0258)
P(delta <= 0): 0.3556

Full 17
Delta: +0.0031
95% CI: (-0.0183, 0.0249)
P(delta <= 0): 0.3843

2WIKI

                                  Model  F

Experiment 2

In [ ]:


import os
import gc
import json
import numpy as np
import pandas as pd
import torch



for name in [
    "model",
    "base_model",
    "trainer",
]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


D = "/content/drive/MyDrive/hedge_run"

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"


PAIR_JSONL = (
    f"{D}/hotpot_dpo_pairs_FINAL_CLEAN.jsonl"
)

PAIR_METADATA = (
    f"{D}/hotpot_dpo_pairs_FINAL_CLEAN_metadata.csv"
)

OOF_CSV = (
    f"{D}/hotpot_experiment1_oof_predictions.csv"
)

LABELLED_CSV = (
    f"{D}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
)

FINAL_JSON = (
    f"{D}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_LABELLED_1000.json"
)

TEST_CLASSIFIER_CSV = (
    f"{D}/"
    "hotpot_dpo_test_classifier_FINAL_CLEAN.csv"
)

OUT_DIR = (
    f"{D}/"
    "hotpot_dpo_reliability_FINAL_CLEAN_adapter"
)

GEN_PATH = (
    f"{D}/"
    "hotpot_dpo_FINAL_CLEAN_v2_test_generation.json"
)

JUDGE_PATH = (
    f"{D}/"
    "hotpot_dpo_FINAL_CLEAN_v2_test_judgments.json"
)

RESULT_CSV = (
    f"{D}/"
    "hotpot_dpo_FINAL_CLEAN_v2_test_results.csv"
)

RESULT_JSON = (
    f"{D}/"
    "hotpot_dpo_FINAL_CLEAN_v2_test_summary.json"
)


required = {
    "pairs": PAIR_JSONL,
    "metadata": PAIR_METADATA,
    "OOF": OOF_CSV,
    "labelled CSV": LABELLED_CSV,
    "final JSON": FINAL_JSON,
    "test classifier": TEST_CLASSIFIER_CSV,
}


for name, path in required.items():

    print(
        f"{name:20s}",
        os.path.exists(path),
        path
    )

    assert os.path.exists(path), (
        f"MISSING FILE: {path}"
    )



with open(
    PAIR_JSONL,
    "r",
    encoding="utf-8"
) as f:

    pairs = [
        json.loads(line)
        for line in f
        if line.strip()
    ]


meta = pd.read_csv(
    PAIR_METADATA
)


print()
print("=" * 80)
print("PREFERENCE DATA")
print("=" * 80)

print(
    "Training pairs:",
    len(pairs)
)

print()
print(
    meta["kind"].value_counts()
)


assert len(pairs) == 694

assert (
    meta["kind"]
    .eq("answer_preferred")
    .sum()
    ==
    374
)

assert (
    meta["kind"]
    .eq("abstain_preferred")
    .sum()
    ==
    320
)

assert (
    meta["kind"]
    .eq("excluded_low_risk_wrong")
    .sum()
    ==
    106
)



oof = pd.read_csv(
    OOF_CSV
)


RISK_THRESHOLD = float(
    np.percentile(
        oof[
            "risk_full_17"
        ].astype(float),
        60,
    )
)


print()
print(
    "Frozen risk threshold:",
    RISK_THRESHOLD
)


assert abs(
    RISK_THRESHOLD
    -
    0.4696021204774299
) < 1e-8


test_clf = pd.read_csv(
    TEST_CLASSIFIER_CSV
)


assert len(test_clf) == 200


print()
print("=" * 80)
print("TEST CLASSIFIER")
print("=" * 80)

print(
    test_clf[
        "classifier_action"
    ].value_counts()
)


assert (
    test_clf[
        "classifier_action"
    ]
    .eq("answer")
    .sum()
    ==
    136
)

assert (
    test_clf[
        "classifier_action"
    ]
    .eq("abstain")
    .sum()
    ==
    64
)


print()
print("=" * 80)
print("CELL 1 COMPLETE")
print("=" * 80)

print(
    "All frozen RQ2 inputs are valid."
)

pairs                True /content/drive/MyDrive/hedge_run/hotpot_dpo_pairs_FINAL_CLEAN.jsonl
metadata             True /content/drive/MyDrive/hedge_run/hotpot_dpo_pairs_FINAL_CLEAN_metadata.csv
OOF                  True /content/drive/MyDrive/hedge_run/hotpot_experiment1_oof_predictions.csv
labelled CSV         True /content/drive/MyDrive/hedge_run/hotpot_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv
final JSON           True /content/drive/MyDrive/hedge_run/hotpot_passive_iterative_v2_FINAL_CLEAN_LABELLED_1000.json
test classifier      True /content/drive/MyDrive/hedge_run/hotpot_dpo_test_classifier_FINAL_CLEAN.csv

PREFERENCE DATA
Training pairs: 694

kind
answer_preferred           374
abstain_preferred          320
excluded_low_risk_wrong    106
Name: count, dtype: int64

Frozen risk threshold: 0.4696021204774299

TEST CLASSIFIER
classifier_action
answer     136
abstain     64
Name: count, dtype: int64

CELL 1 COMPLETE
All frozen RQ2 inputs are valid.


In [ ]:
import os
import gc
import math
import shutil
import inspect

import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
)

from trl import (
    DPOConfig,
    DPOTrainer,
)

import trl
import transformers
import peft


SEED = 42

BETA = 0.05

LEARNING_RATE = 1e-5

EPOCHS = 3

BATCH_SIZE = 2

GRAD_ACC = 8

MAX_LENGTH = 1280



print("=" * 80)
print("ENVIRONMENT")
print("=" * 80)

print(
    "TRL:",
    trl.__version__
)

print(
    "Transformers:",
    transformers.__version__
)

print(
    "PEFT:",
    peft.__version__
)

print(
    "PyTorch:",
    torch.__version__
)


assert torch.cuda.is_available(), (
    "GPU is required."
)


print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "BF16 supported:",
    torch.cuda.is_bf16_supported()
)


assert torch.cuda.is_bf16_supported(), (
    "The frozen experiment uses BF16 computation."
)


adapter_config_path = os.path.join(
    OUT_DIR,
    "adapter_config.json"
)


if os.path.exists(
    OUT_DIR
):

    if os.path.exists(
        adapter_config_path
    ):

        print()
        print(
            "Existing COMPLETE adapter found."
        )

        print(
            "Deleting it so this is a clean rerun..."
        )


    else:

        print()
        print(
            "Removing incomplete previous output directory..."
        )


    shutil.rmtree(
        OUT_DIR
    )


os.makedirs(
    OUT_DIR,
    exist_ok=True
)



ds = load_dataset(
    "json",
    data_files=PAIR_JSONL,
    split="train",
)


assert len(ds) == 694


print()
print(
    "Preference pairs:",
    len(ds)
)



micro_batches_per_epoch = math.ceil(
    len(ds)
    /
    BATCH_SIZE
)


updates_per_epoch = math.ceil(
    micro_batches_per_epoch
    /
    GRAD_ACC
)


total_updates = (
    updates_per_epoch
    *
    EPOCHS
)


WARMUP_STEPS = int(
    round(
        0.10
        *
        total_updates
    )
)


print(
    "Micro-batches per epoch:",
    micro_batches_per_epoch
)

print(
    "Optimiser updates per epoch:",
    updates_per_epoch
)

print(
    "Approx. total optimiser updates:",
    total_updates
)

print(
    "Warm-up steps:",
    WARMUP_STEPS
)


assert WARMUP_STEPS == 13




config_params = (
    inspect.signature(
        DPOConfig.__init__
    ).parameters
)


print()
print("=" * 80)
print("INSTALLED DPOCONFIG SUPPORT")
print("=" * 80)


for param in [
    "beta",
    "max_length",
    "warmup_steps",
    "warmup_ratio",
    "disable_dropout",
    "truncation_mode",
]:

    print(
        f"{param:20s}",
        param in config_params
    )


# These are required for our frozen experiment.
for required_param in [
    "beta",
    "max_length",
    "warmup_steps",
]:

    if required_param not in config_params:

        raise RuntimeError(
            f"Installed TRL does not support "
            f"required DPOConfig argument: "
            f"{required_param}"
        )


tokenizer = (
    AutoTokenizer
    .from_pretrained(
        BASE_MODEL
    )
)


if tokenizer.pad_token is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


tokenizer.padding_side = "left"




bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type=
        "nf4",

    bnb_4bit_compute_dtype=
        torch.bfloat16,

    bnb_4bit_use_double_quant=
        True,
)


print()
print("=" * 80)
print("LOADING QWEN")
print("=" * 80)


model = (
    AutoModelForCausalLM
    .from_pretrained(

        BASE_MODEL,

        quantization_config=
            bnb_config,

        device_map={
            "": 0
        },

        dtype=
            torch.bfloat16,
    )
)


model.config.use_cache = False


model = (
    prepare_model_for_kbit_training(
        model
    )
)



peft_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


config_kwargs = {

    "output_dir":
        OUT_DIR,

    # DPO
    "beta":
        BETA,

    # Optimisation
    "learning_rate":
        LEARNING_RATE,

    "num_train_epochs":
        EPOCHS,

    "per_device_train_batch_size":
        BATCH_SIZE,

    "gradient_accumulation_steps":
        GRAD_ACC,

    # Fixed equivalent of 10% warm-up
    "warmup_steps":
        WARMUP_STEPS,

    "lr_scheduler_type":
        "cosine",

    # Sequence
    "max_length":
        MAX_LENGTH,

    # Precision
    "bf16":
        True,

    "fp16":
        False,

    # Memory
    "gradient_checkpointing":
        True,

    # Logging / save
    "logging_steps":
        10,

    "save_strategy":
        "epoch",

    "save_total_limit":
        3,

    "report_to":
        "none",

    # Reproducibility
    "seed":
        SEED,
}




optional_kwargs = {

    "loss_type":
        "sigmoid",

    "truncation_mode":
        "keep_start",

    "disable_dropout":
        False,

    "data_seed":
        SEED,

    "gradient_checkpointing_kwargs":
        {
            "use_reentrant":
                False
        },
}


for key, value in optional_kwargs.items():

    if key in config_params:

        config_kwargs[
            key
        ] = value


training_args = DPOConfig(
    **config_kwargs
)


print()
print(
    "DPOConfig successfully created."
)


trainer_params = (
    inspect.signature(
        DPOTrainer.__init__
    ).parameters
)


trainer_kwargs = {

    "model":
        model,

    "ref_model":
        None,

    "args":
        training_args,

    "train_dataset":
        ds,

    "peft_config":
        peft_config,
}


# Newer TRL:
if "processing_class" in trainer_params:

    trainer_kwargs[
        "processing_class"
    ] = tokenizer


# Older TRL:
elif "tokenizer" in trainer_params:

    trainer_kwargs[
        "tokenizer"
    ] = tokenizer


else:

    raise RuntimeError(
        "Cannot find tokenizer/processing_class "
        "argument in installed DPOTrainer."
    )


trainer = DPOTrainer(
    **trainer_kwargs
)



print()
print("=" * 80)
print("TRAINABLE PARAMETERS")
print("=" * 80)


if hasattr(
    trainer.model,
    "print_trainable_parameters",
):

    trainer.model.print_trainable_parameters()



print()
print("=" * 80)
print("STARTING CLEAN HOTPOT DPO TRAINING")
print("=" * 80)


train_result = (
    trainer.train()
)


trainer.save_model(
    OUT_DIR
)

tokenizer.save_pretrained(
    OUT_DIR
)


adapter_config_path = os.path.join(
    OUT_DIR,
    "adapter_config.json"
)


print()
print("=" * 80)
print("DPO TRAINING COMPLETE")
print("=" * 80)

print(
    "Adapter directory:",
    OUT_DIR
)

print(
    "adapter_config.json exists:",
    os.path.exists(
        adapter_config_path
    )
)


print()
print("Files:")

for filename in os.listdir(
    OUT_DIR
):

    print(
        " ",
        filename
    )


assert os.path.exists(
    adapter_config_path
), (
    "Training completed but adapter_config.json "
    "was not saved."
)


print()
print(
    "TRAINING SUCCESSFUL — SAFE TO EVALUATE."
)

ENVIRONMENT
TRL: 1.10.0
Transformers: 5.15.0
PEFT: 0.20.0
PyTorch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-80GB
BF16 supported: True

Preference pairs: 694
Micro-batches per epoch: 347
Optimiser updates per epoch: 44
Approx. total optimiser updates: 132
Warm-up steps: 13

INSTALLED DPOCONFIG SUPPORT
beta                 True
max_length           True
warmup_steps         True
warmup_ratio         False
disable_dropout      True
truncation_mode      True

LOADING QWEN


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


DPOConfig successfully created.


Tokenizing train dataset:   0%|          | 0/694 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/694 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



TRAINABLE PARAMETERS
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273

STARTING CLEAN HOTPOT DPO TRAINING


Step,Training Loss
10,0.685932
20,0.633988
30,0.622962
40,0.654191
50,0.628185
60,0.545093
70,0.582837
80,0.513594
90,0.521533
100,0.519196



DPO TRAINING COMPLETE
Adapter directory: /content/drive/MyDrive/hedge_run/hotpot_dpo_reliability_FINAL_CLEAN_adapter
adapter_config.json exists: True

Files:
  README.md
  checkpoint-44
  checkpoint-88
  checkpoint-132
  adapter_model.safetensors
  adapter_config.json
  chat_template.jinja
  tokenizer_config.json
  tokenizer.json
  training_args.bin

TRAINING SUCCESSFUL — SAFE TO EVALUATE.


hotpot

In [ ]:
import os
import gc
import re
import json

import torch

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel



adapter_config_path = os.path.join(
    OUT_DIR,
    "adapter_config.json"
)


assert os.path.exists(
    adapter_config_path
)


print(
    "Adapter found:",
    adapter_config_path
)




for name in [
    "trainer",
    "model",
]:

    if name in globals():

        del globals()[
            name
        ]


gc.collect()

torch.cuda.empty_cache()


print(
    "CUDA allocated GB after cleanup:",
    round(
        torch.cuda.memory_allocated()
        /
        1e9,
        3
    )
)




with open(
    FINAL_JSON,
    "r",
    encoding="utf-8"
) as f:

    all_records = json.load(f)


test_records = [

    r

    for r in all_records

    if r["split"] == "test"
]


assert len(test_records) == 200


print(
    "Test questions:",
    len(test_records)
)



INSTRUCTION = """You are given a question, retrieved evidence, and an existing reasoning trace.

Decide whether the question can be answered reliably from this information.

If it can be answered reliably, respond only in this form:
FINAL ANSWER: <answer>

If it cannot be answered reliably, respond exactly:
I don't know.

Do not provide any additional explanation."""


def as_list(value):

    if value is None:

        return []


    if isinstance(
        value,
        (list, tuple),
    ):

        return [

            str(x).strip()

            for x in value

            if str(x).strip()
        ]


    return [
        str(value).strip()
    ]


def make_prompt(record):

    question = str(
        record[
            "question"
        ]
    ).strip()


    reasoning = as_list(
        record.get(
            "selected_steps",
            []
        )
    )


    evidence = as_list(
        record.get(
            "question_evidence",
            []
        )
    )


    if not evidence:

        raise RuntimeError(
            "Missing frozen evidence for "
            f"QI={record['question_index']}"
        )


    if reasoning:

        reasoning_text = "\n".join(

            f"{i}. {step}"

            for i, step
            in enumerate(
                reasoning,
                start=1
            )
        )

    else:

        reasoning_text = (
            "No intermediate reasoning was produced."
        )


    evidence_text = "\n".join(

        f"E{i}: {sentence}"

        for i, sentence
        in enumerate(
            evidence,
            start=1
        )
    )


    return f"""{INSTRUCTION}

QUESTION:
{question}

REASONING TRACE:
{reasoning_text}

RETRIEVED EVIDENCE:
{evidence_text}""".strip()


tokenizer = (
    AutoTokenizer
    .from_pretrained(
        BASE_MODEL
    )
)


if tokenizer.pad_token is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


tokenizer.padding_side = "left"


bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type=
        "nf4",

    bnb_4bit_compute_dtype=
        torch.bfloat16,

    bnb_4bit_use_double_quant=
        True,
)


print()
print(
    "Loading base Qwen..."
)


base_model = (
    AutoModelForCausalLM
    .from_pretrained(

        BASE_MODEL,

        quantization_config=
            bnb_config,

        device_map={
            "": 0
        },

        dtype=
            torch.bfloat16,
    )
)


print(
    "Loading DPO LoRA adapter..."
)


model = (
    PeftModel
    .from_pretrained(

        base_model,

        OUT_DIR,
    )
)


model.eval()

model.config.use_cache = True


MODEL_DEVICE = (
    next(
        model.parameters()
    ).device
)


print(
    "Model device:",
    MODEL_DEVICE
)



ABSTAIN_PATTERNS = [

    r"^i don't know\b",

    r"^i do not know\b",

    r"^cannot answer\b",

    r"^can't answer\b",

    r"^unable to answer\b",

    r"^not enough information\b",
]


def parse_output(text):

    text = str(
        text
    ).strip()


    without_prefix = re.sub(

        r"^\s*FINAL\s+ANSWER\s*:\s*",

        "",

        text,

        flags=re.I,
    ).strip()


    low = without_prefix.lower()


    for pattern in ABSTAIN_PATTERNS:

        if re.search(
            pattern,
            low
        ):

            return (
                "abstain",
                None,
                True,
            )



    match = re.search(

        r"FINAL\s+ANSWER\s*:\s*(.+)",

        text,

        flags=re.I | re.S,
    )


    if match:

        answer = (

            match
            .group(1)
            .strip()
            .splitlines()[0]
            .strip()
        )


        return (
            "answer",
            answer,
            True,
        )



    return (
        "answer",
        text,
        False,
    )


@torch.inference_mode()
def generate_one(record):

    prompt = make_prompt(
        record
    )


    messages = [
        {
            "role":
                "user",

            "content":
                prompt,
        }
    ]


    rendered = tokenizer.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True,
    )


    inputs = tokenizer(

        rendered,

        return_tensors="pt",

        add_special_tokens=False,
    )


    inputs = {

        key:
            value.to(
                MODEL_DEVICE
            )

        for key, value
        in inputs.items()
    }


    input_length = (
        inputs[
            "input_ids"
        ].shape[1]
    )


    output = model.generate(

        **inputs,

        max_new_tokens=
            64,

        do_sample=
            False,

        pad_token_id=
            tokenizer.pad_token_id,

        eos_token_id=
            tokenizer.eos_token_id,
    )


    generated_ids = (
        output[
            0,
            input_length:
        ]
    )


    text = tokenizer.decode(

        generated_ids,

        skip_special_tokens=True,
    ).strip()


    return text




if os.path.exists(
    GEN_PATH
):

    with open(
        GEN_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        generations = json.load(f)

else:

    generations = {}


print()
print(
    "Existing completed generations:",
    len(generations)
)


for record in tqdm(
    test_records
):

    qi = str(
        int(
            record[
                "question_index"
            ]
        )
    )


    if qi in generations:

        continue


    raw = generate_one(
        record
    )


    (
        action,
        answer,
        format_ok,
    ) = parse_output(
        raw
    )


    generations[
        qi
    ] = {

        "question_index":
            int(qi),

        "raw_response":
            raw,

        "dpo_action":
            action,

        "dpo_answer":
            answer,

        "format_ok":
            bool(
                format_ok
            ),
    }


    # Save after every question.
    with open(
        GEN_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            generations,
            f,
            indent=2,
            ensure_ascii=False,
        )


assert len(
    generations
) == 200


N_ANSWER = sum(

    item[
        "dpo_action"
    ]
    ==
    "answer"

    for item
    in generations.values()
)


N_ABSTAIN = (
    200
    -
    N_ANSWER
)


N_BAD_FORMAT = sum(

    not item[
        "format_ok"
    ]

    for item
    in generations.values()
)


print()
print("=" * 80)
print("DPO TEST GENERATION COMPLETE")
print("=" * 80)

print(
    "Answer:",
    N_ANSWER
)

print(
    "Abstain:",
    N_ABSTAIN
)

print(
    "Non-standard format:",
    N_BAD_FORMAT
)

print(
    "Total:",
    len(generations)
)

print()
print(
    "Saved:",
    GEN_PATH
)

Adapter found: /content/drive/MyDrive/hedge_run/hotpot_dpo_reliability_FINAL_CLEAN_adapter/adapter_config.json
CUDA allocated GB after cleanup: 7.842
Test questions: 200

Loading base Qwen...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading DPO LoRA adapter...
Model device: cuda:0

Existing completed generations: 0


  0%|          | 0/200 [00:00<?, ?it/s]


DPO TEST GENERATION COMPLETE
Answer: 114
Abstain: 86
Non-standard format: 0
Total: 200

Saved: /content/drive/MyDrive/hedge_run/hotpot_dpo_FINAL_CLEAN_v2_test_generation.json


In [ ]:
!pip -q install -U openai

import os
import json
import time
import pandas as pd
import numpy as np

from tqdm.auto import tqdm
from openai import OpenAI



D = "/content/drive/MyDrive/hedge_run"


FINAL_JSON = (
    f"{D}/"
    "hotpot_passive_iterative_v2_FINAL_CLEAN_LABELLED_1000.json"
)


TEST_CLASSIFIER_CSV = (
    f"{D}/"
    "hotpot_dpo_test_classifier_FINAL_CLEAN.csv"
)


GEN_PATH = (
    f"{D}/"
    "hotpot_dpo_FINAL_CLEAN_v2_test_generation.json"
)


JUDGE_PATH = (
    f"{D}/"
    "hotpot_dpo_FINAL_CLEAN_v2_test_judgments.json"
)


RESULT_CSV = (
    f"{D}/"
    "hotpot_dpo_FINAL_CLEAN_v2_test_results.csv"
)


RESULT_JSON = (
    f"{D}/"
    "hotpot_dpo_FINAL_CLEAN_v2_test_summary.json"
)




for path in [
    FINAL_JSON,
    TEST_CLASSIFIER_CSV,
    GEN_PATH,
]:

    assert os.path.exists(path), (
        f"Missing file:\n{path}"
    )



with open(
    FINAL_JSON,
    "r",
    encoding="utf-8",
) as f:

    all_records = json.load(f)


test_records = [
    r
    for r in all_records
    if r["split"] == "test"
]


assert len(test_records) == 200


records_by_qi = {
    str(int(r["question_index"])): r
    for r in test_records
}



with open(
    GEN_PATH,
    "r",
    encoding="utf-8",
) as f:

    generations = json.load(f)


assert len(generations) == 200


n_answer = sum(
    x["dpo_action"] == "answer"
    for x in generations.values()
)

n_abstain = sum(
    x["dpo_action"] == "abstain"
    for x in generations.values()
)


print("=" * 80)
print("DPO GENERATIONS")
print("=" * 80)

print("Total:", len(generations))
print("Answer:", n_answer)
print("Abstain:", n_abstain)


assert n_answer + n_abstain == 200


assert n_answer == 114
assert n_abstain == 86




classifier_df = pd.read_csv(
    TEST_CLASSIFIER_CSV
)


assert len(classifier_df) == 200


classifier_by_qi = {
    int(row.question_index): row
    for row in classifier_df.itertuples()
}


print()
print("=" * 80)
print("FROZEN CLASSIFIER")
print("=" * 80)

print(
    classifier_df[
        "classifier_action"
    ].value_counts()
)


OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)


if not OPENAI_API_KEY:

    try:
        from google.colab import userdata

        OPENAI_API_KEY = userdata.get(
            "OPENAI_API_KEY"
        )

    except Exception:
        OPENAI_API_KEY = None


if not OPENAI_API_KEY:

    raise RuntimeError(
        "OPENAI_API_KEY not found. "
        "Add it to Colab Secrets or the environment."
    )


client = OpenAI(
    api_key=OPENAI_API_KEY
)


JUDGE_MODEL = "gpt-4o-mini"


JUDGE_INSTRUCTIONS = """
You are an offline correctness evaluator for a question-answering experiment.

You receive:
1. a question,
2. a reference answer,
3. a model prediction.

Determine whether the prediction should be counted as correct.

Accept:
- semantically equivalent paraphrases,
- aliases,
- spelling variants,
- capitalization differences,
- harmless formatting differences.

Reject:
- factually different answers,
- answers to a different relation,
- reversed comparisons,
- broader or narrower answers that change what was asked.

Judge only the correctness of the final model prediction.

Do not use or infer any hidden reasoning, confidence score, retrieved evidence,
classifier output, or reliability feature.

Return a binary correctness judgment.
""".strip()


JUDGE_SCHEMA = {
    "type": "object",

    "properties": {
        "correct": {
            "type": "boolean"
        },

        "reason": {
            "type": "string"
        },
    },

    "required": [
        "correct",
        "reason",
    ],

    "additionalProperties": False,
}


def judge_answer(
    question,
    gold,
    prediction,
):

    prompt = f"""QUESTION:
{question}

REFERENCE ANSWER:
{gold}

MODEL PREDICTION:
{prediction}"""


    for attempt in range(5):

        try:

            response = client.responses.create(

                model=JUDGE_MODEL,

                instructions=JUDGE_INSTRUCTIONS,

                input=prompt,

                temperature=0,

                max_output_tokens=120,

                store=False,

                text={
                    "format": {
                        "type": "json_schema",
                        "name": "qa_correctness",
                        "strict": True,
                        "schema": JUDGE_SCHEMA,
                    }
                },
            )


            result = json.loads(
                response.output_text
            )


            assert isinstance(
                result["correct"],
                bool,
            )


            return result


        except Exception as exc:

            if attempt == 4:
                raise RuntimeError(
                    f"Judge failed after 5 attempts:\n{exc}"
                )

            time.sleep(
                2 ** attempt
            )


if os.path.exists(
    JUDGE_PATH
):

    with open(
        JUDGE_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        judgments = json.load(f)

else:

    judgments = {}


answered_ids = [
    qi

    for qi, output
    in generations.items()

    if output["dpo_action"] == "answer"
]


print()
print("=" * 80)
print("CORRECTNESS JUDGING")
print("=" * 80)

print(
    "Returned answers:",
    len(answered_ids)
)

print(
    "Already judged:",
    sum(
        qi in judgments
        for qi in answered_ids
    )
)

print(
    "Remaining:",
    sum(
        qi not in judgments
        for qi in answered_ids
    )
)



for qi in tqdm(
    answered_ids,
    desc="Judging DPO answers",
):

    if qi in judgments:
        continue


    record = records_by_qi[
        qi
    ]


    prediction = generations[
        qi
    ][
        "dpo_answer"
    ]


    if prediction is None:
        raise RuntimeError(
            f"DPO action=answer but answer=None for QI={qi}"
        )


    result = judge_answer(

        question=record[
            "question"
        ],

        gold=record[
            "gold_answer"
        ],

        prediction=prediction,
    )


    judgments[
        qi
    ] = {

        "correct": bool(
            result["correct"]
        ),

        "reason": str(
            result["reason"]
        ).strip(),
    }



    with open(
        JUDGE_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            judgments,
            f,
            indent=2,
            ensure_ascii=False,
        )


assert all(
    qi in judgments
    for qi in answered_ids
)


print()
print(
    "All returned answers judged:",
    len(answered_ids)
)


rows = []


for record in test_records:

    qi = int(
        record["question_index"]
    )

    key = str(qi)


    gen = generations[
        key
    ]


    dpo_action = gen[
        "dpo_action"
    ]


    dpo_answered = (
        dpo_action == "answer"
    )


    if dpo_answered:

        dpo_correct = bool(
            judgments[
                key
            ][
                "correct"
            ]
        )

        judge_reason = judgments[
            key
        ][
            "reason"
        ]

    else:

        # Abstention is not a correct answer.
        # It is handled separately by selective metrics.
        dpo_correct = False

        judge_reason = None


    clf = classifier_by_qi[
        qi
    ]


    rows.append(
        {

            "question_index":
                qi,

            "question":
                record[
                    "question"
                ],

            "gold_answer":
                record[
                    "gold_answer"
                ],



            "base_answer":
                record[
                    "majority_answer"
                ],

            "base_correct":
                int(
                    bool(
                        record[
                            "judge_correct"
                        ]
                    )
                ),



            "classifier_risk":
                float(
                    clf.classifier_risk
                ),

            "classifier_action":
                str(
                    clf.classifier_action
                ),



            "dpo_action":
                dpo_action,

            "dpo_answer":
                gen[
                    "dpo_answer"
                ],

            "dpo_raw_response":
                gen[
                    "raw_response"
                ],

            "format_ok":
                int(
                    bool(
                        gen[
                            "format_ok"
                        ]
                    )
                ),

            "dpo_correct":
                int(
                    dpo_correct
                ),

            "judge_reason":
                judge_reason,
        }
    )


result_df = pd.DataFrame(
    rows
)


assert len(result_df) == 200



N = len(
    result_df
)


N_BASE_CORRECT = int(
    result_df[
        "base_correct"
    ].sum()
)


N_BASE_WRONG = (
    N
    -
    N_BASE_CORRECT
)


BASE_ACCURACY = (
    N_BASE_CORRECT
    /
    N
)


# Current frozen Hotpot test should be:
assert N_BASE_CORRECT == 128
assert N_BASE_WRONG == 72



dpo_answer_mask = (
    result_df[
        "dpo_action"
    ]
    ==
    "answer"
)


dpo_abstain_mask = (
    result_df[
        "dpo_action"
    ]
    ==
    "abstain"
)


N_DPO_ANSWER = int(
    dpo_answer_mask.sum()
)


N_DPO_ABSTAIN = int(
    dpo_abstain_mask.sum()
)


N_DPO_CORRECT = int(

    result_df.loc[
        dpo_answer_mask,
        "dpo_correct"
    ].sum()
)


N_DPO_WRONG = (
    N_DPO_ANSWER
    -
    N_DPO_CORRECT
)


COVERAGE = (
    N_DPO_ANSWER
    /
    N
)


SELECTIVE_ACCURACY = (
    N_DPO_CORRECT
    /
    N_DPO_ANSWER
)


CONFIDENT_ERROR = (
    N_DPO_WRONG
    /
    N
)



base_correct_mask = (
    result_df[
        "base_correct"
    ]
    ==
    1
)


N_OVER_ABSTAIN = int(

    (
        base_correct_mask
        &
        dpo_abstain_mask
    ).sum()
)


OVER_ABSTENTION = (
    N_OVER_ABSTAIN
    /
    N_BASE_CORRECT
)


Pc0 = (
    N_BASE_CORRECT
    /
    N
)


Pw0 = (
    N_BASE_WRONG
    /
    N
)


Pc = (
    N_DPO_CORRECT
    /
    N
)


Pw = (
    N_DPO_WRONG
    /
    N
)


THS = (
    (
        Pc * Pw0
        -
        Pw * Pc0
    )
    /
    Pw0
)


THS100 = (
    100 * THS
)


clf_answer_mask = (
    result_df[
        "classifier_action"
    ]
    ==
    "answer"
)


clf_abstain_mask = (
    result_df[
        "classifier_action"
    ]
    ==
    "abstain"
)


N_CLF_ANSWER = int(
    clf_answer_mask.sum()
)


N_CLF_ABSTAIN = int(
    clf_abstain_mask.sum()
)


N_CLF_CORRECT = int(

    result_df.loc[
        clf_answer_mask,
        "base_correct"
    ].sum()
)


N_CLF_WRONG = (
    N_CLF_ANSWER
    -
    N_CLF_CORRECT
)


CLF_COVERAGE = (
    N_CLF_ANSWER
    /
    N
)


CLF_SELECTIVE_ACCURACY = (
    N_CLF_CORRECT
    /
    N_CLF_ANSWER
)


CLF_CONFIDENT_ERROR = (
    N_CLF_WRONG
    /
    N
)



same_action = (
    result_df[
        "classifier_action"
    ]
    ==
    result_df[
        "dpo_action"
    ]
)


FIDELITY = float(
    same_action.mean()
)


when_classifier_answers = (
    result_df[
        "classifier_action"
    ]
    ==
    "answer"
)


when_classifier_abstains = (
    result_df[
        "classifier_action"
    ]
    ==
    "abstain"
)


FIDELITY_ANSWER = float(

    (
        result_df.loc[
            when_classifier_answers,
            "dpo_action"
        ]
        ==
        "answer"
    ).mean()
)


FIDELITY_ABSTAIN = float(

    (
        result_df.loc[
            when_classifier_abstains,
            "dpo_action"
        ]
        ==
        "abstain"
    ).mean()
)


fidelity_table = pd.crosstab(

    result_df[
        "classifier_action"
    ],

    result_df[
        "dpo_action"
    ],

    rownames=[
        "Classifier"
    ],

    colnames=[
        "DPO"
    ],
)



FORMAT_COMPLIANCE = float(
    result_df[
        "format_ok"
    ].mean()
)



result_df.to_csv(
    RESULT_CSV,
    index=False,
)


summary = {

    "dataset":
        "HotpotQA",

    "test_n":
        N,

    "baseline": {

        "correct":
            N_BASE_CORRECT,

        "wrong":
            N_BASE_WRONG,

        "accuracy":
            BASE_ACCURACY,
    },

    "classifier": {

        "answered":
            N_CLF_ANSWER,

        "abstained":
            N_CLF_ABSTAIN,

        "correct_returned":
            N_CLF_CORRECT,

        "wrong_returned":
            N_CLF_WRONG,

        "coverage":
            CLF_COVERAGE,

        "selective_accuracy":
            CLF_SELECTIVE_ACCURACY,

        "confident_error":
            CLF_CONFIDENT_ERROR,
    },

    "dpo": {

        "answered":
            N_DPO_ANSWER,

        "abstained":
            N_DPO_ABSTAIN,

        "correct_returned":
            N_DPO_CORRECT,

        "wrong_returned":
            N_DPO_WRONG,

        "coverage":
            COVERAGE,

        "selective_accuracy":
            SELECTIVE_ACCURACY,

        "confident_error":
            CONFIDENT_ERROR,

        "over_abstention_count":
            N_OVER_ABSTAIN,

        "over_abstention":
            OVER_ABSTENTION,

        "THS":
            THS,

        "THSx100":
            THS100,

        "format_compliance":
            FORMAT_COMPLIANCE,
    },

    "fidelity": {

        "overall":
            FIDELITY,

        "when_classifier_answers":
            FIDELITY_ANSWER,

        "when_classifier_abstains":
            FIDELITY_ABSTAIN,
    },
}


with open(
    RESULT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
    )



print()
print("=" * 80)
print("FINAL CLEAN HOTPOT RQ2 RESULTS")
print("=" * 80)


print()
print("BASE MODEL")
print("-" * 50)

print(
    "Correct:",
    N_BASE_CORRECT
)

print(
    "Wrong:",
    N_BASE_WRONG
)

print(
    "Accuracy:",
    round(
        BASE_ACCURACY,
        4
    )
)


print()
print("FROZEN RELIABILITY CLASSIFIER")
print("-" * 50)

print(
    "Answered:",
    N_CLF_ANSWER
)

print(
    "Abstained:",
    N_CLF_ABSTAIN
)

print(
    "Correct returned:",
    N_CLF_CORRECT
)

print(
    "Wrong returned:",
    N_CLF_WRONG
)

print(
    "Coverage:",
    round(
        CLF_COVERAGE,
        4
    )
)

print(
    "Selective accuracy:",
    round(
        CLF_SELECTIVE_ACCURACY,
        4
    )
)

print(
    "Confident error:",
    round(
        CLF_CONFIDENT_ERROR,
        4
    )
)


print()
print("DPO")
print("-" * 50)

print(
    "Answered:",
    N_DPO_ANSWER
)

print(
    "Abstained:",
    N_DPO_ABSTAIN
)

print(
    "Correct returned:",
    N_DPO_CORRECT
)

print(
    "Wrong returned:",
    N_DPO_WRONG
)

print(
    "Coverage:",
    round(
        COVERAGE,
        4
    )
)

print(
    "Selective accuracy:",
    round(
        SELECTIVE_ACCURACY,
        4
    )
)

print(
    "Confident error:",
    round(
        CONFIDENT_ERROR,
        4
    )
)

print(
    "Over-abstention count:",
    N_OVER_ABSTAIN
)

print(
    "Over-abstention:",
    round(
        OVER_ABSTENTION,
        4
    )
)

print(
    "THS x100:",
    round(
        THS100,
        2
    )
)

print(
    "Format compliance:",
    round(
        FORMAT_COMPLIANCE,
        4
    )
)


print()
print("CLASSIFIER ↔ DPO FIDELITY")
print("-" * 50)

print(
    "Overall:",
    round(
        FIDELITY,
        4
    )
)

print(
    "When classifier answers:",
    round(
        FIDELITY_ANSWER,
        4
    )
)

print(
    "When classifier abstains:",
    round(
        FIDELITY_ABSTAIN,
        4
    )
)


print()
print(
    fidelity_table
)


print()
print("=" * 80)
print("SAVED")
print("=" * 80)

print(
    "Judgments:",
    JUDGE_PATH
)

print(
    "Detailed results:",
    RESULT_CSV
)

print(
    "Summary:",
    RESULT_JSON
)

print()
print("DONE.")

DPO GENERATIONS
Total: 200
Answer: 114
Abstain: 86

FROZEN CLASSIFIER
classifier_action
answer     136
abstain     64
Name: count, dtype: int64

CORRECTNESS JUDGING
Returned answers: 114
Already judged: 0
Remaining: 114


Judging DPO answers:   0%|          | 0/114 [00:00<?, ?it/s]


All returned answers judged: 114

FINAL CLEAN HOTPOT RQ2 RESULTS

BASE MODEL
--------------------------------------------------
Correct: 128
Wrong: 72
Accuracy: 0.64

FROZEN RELIABILITY CLASSIFIER
--------------------------------------------------
Answered: 136
Abstained: 64
Correct returned: 96
Wrong returned: 40
Coverage: 0.68
Selective accuracy: 0.7059
Confident error: 0.2

DPO
--------------------------------------------------
Answered: 114
Abstained: 86
Correct returned: 86
Wrong returned: 28
Coverage: 0.57
Selective accuracy: 0.7544
Confident error: 0.14
Over-abstention count: 47
Over-abstention: 0.3672
THS x100: 18.11
Format compliance: 1.0

CLASSIFIER ↔ DPO FIDELITY
--------------------------------------------------
Overall: 0.67
When classifier answers: 0.6765
When classifier abstains: 0.6562

DPO         abstain  answer
Classifier                 
abstain          42      22
answer           44      92

SAVED
Judgments: /content/drive/MyDrive/hedge_run/hotpot_dpo_FINAL_CLEAN

2wiki

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


D = "/content/drive/MyDrive/hedge_run"

SEED = 42
RISK_PERCENTILE = 60

random.seed(SEED)
np.random.seed(SEED)


FINAL_JSON = (
    f"{D}/"
    "2wiki_passive_iterative_v2_FINAL_CLEAN_LABELLED_1000.json"
)

LABELLED_CSV = (
    f"{D}/"
    "2wiki_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
)

OOF_CSV = (
    f"{D}/"
    "2wiki_experiment1_oof_predictions.csv"
)

PAIR_JSONL = (
    f"{D}/"
    "2wiki_dpo_pairs_FINAL_CLEAN.jsonl"
)

PAIR_METADATA = (
    f"{D}/"
    "2wiki_dpo_pairs_FINAL_CLEAN_metadata.csv"
)

TEST_CLASSIFIER_CSV = (
    f"{D}/"
    "2wiki_dpo_test_classifier_FINAL_CLEAN.csv"
)

OUT_DIR = (
    f"{D}/"
    "2wiki_dpo_reliability_FINAL_CLEAN_adapter"
)



REASONING = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]

CONFIDENCE = [
    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]

FINAL_VERIFICATION = [
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]

FULL17 = (
    REASONING
    + CONFIDENCE
    + FINAL_VERIFICATION
)

assert len(FULL17) == 17



with open(
    FINAL_JSON,
    "r",
    encoding="utf-8",
) as f:
    records = json.load(f)


df = pd.read_csv(
    LABELLED_CSV
)

oof = pd.read_csv(
    OOF_CSV
)


assert len(records) == 1000
assert len(df) == 1000
assert len(oof) == 800


by_qi = {
    int(r["question_index"]): r
    for r in records
}


dev = (
    df[df["split"] == "dev"]
    .copy()
    .reset_index(drop=True)
)

test = (
    df[df["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)


assert len(dev) == 800
assert len(test) == 200



dev = dev.merge(
    oof[
        [
            "question_index",
            "risk_full_17",
        ]
    ],
    on="question_index",
    how="left",
    validate="one_to_one",
)


assert dev["risk_full_17"].notna().all()



RISK_THRESHOLD = float(
    np.percentile(
        oof["risk_full_17"].astype(float),
        RISK_PERCENTILE,
    )
)


print("=" * 80)
print("2WIKI FROZEN DPO THRESHOLD")
print("=" * 80)

print(
    "Threshold:",
    RISK_THRESHOLD
)


# Should be around 0.65418.
assert 0.64 < RISK_THRESHOLD < 0.67



INSTRUCTION = """You are given a question, retrieved evidence, and an existing reasoning trace.

Decide whether the question can be answered reliably from this information.

If it can be answered reliably, respond only in this form:
FINAL ANSWER: <answer>

If it cannot be answered reliably, respond exactly:
I don't know.

Do not provide any additional explanation."""


def as_list(x):

    if x is None:
        return []

    if isinstance(x, (list, tuple)):
        return [
            str(v).strip()
            for v in x
            if str(v).strip()
        ]

    return [str(x).strip()]


def make_prompt(record):

    question = str(
        record["question"]
    ).strip()

    evidence = as_list(
        record.get(
            "question_evidence",
            []
        )
    )

    reasoning = as_list(
        record.get(
            "selected_steps",
            []
        )
    )

    if not evidence:
        raise RuntimeError(
            f"Missing question evidence QI={record['question_index']}"
        )

    evidence_text = "\n".join(
        f"E{i}: {e}"
        for i, e in enumerate(
            evidence,
            start=1,
        )
    )

    if reasoning:
        reasoning_text = "\n".join(
            f"{i}. {s}"
            for i, s in enumerate(
                reasoning,
                start=1,
            )
        )
    else:
        reasoning_text = (
            "No intermediate reasoning was produced."
        )

    return f"""{INSTRUCTION}

QUESTION:
{question}

REASONING TRACE:
{reasoning_text}

RETRIEVED EVIDENCE:
{evidence_text}""".strip()


ABSTAIN = "I don't know."


def answer_completion(answer):
    return (
        "FINAL ANSWER: "
        + str(answer).strip()
    )


pairs = []
metadata = []

answer_preferred = 0
abstain_preferred = 0
excluded = 0


for _, row in dev.iterrows():

    qi = int(
        row["question_index"]
    )

    record = by_qi[qi]

    risk = float(
        row["risk_full_17"]
    )

    correct = bool(
        int(
            row["judge_correct"]
        )
    )

    answer = str(
        record["majority_answer"]
    ).strip()

    prompt = make_prompt(
        record
    )

    answer_text = answer_completion(
        answer
    )


    # HIGH RISK -> abstain
    if risk >= RISK_THRESHOLD:

        chosen = ABSTAIN
        rejected = answer_text
        kind = "abstain_preferred"

        abstain_preferred += 1


    # LOW RISK + CORRECT -> answer
    elif correct:

        chosen = answer_text
        rejected = ABSTAIN
        kind = "answer_preferred"

        answer_preferred += 1


    # LOW RISK + WRONG -> exclude
    else:

        excluded += 1

        metadata.append({
            "question_index": qi,
            "risk": risk,
            "judge_correct": 0,
            "kind": "excluded_low_risk_wrong",
        })

        continue


    pairs.append({
        "prompt": [
            {
                "role": "user",
                "content": prompt,
            }
        ],

        "chosen": [
            {
                "role": "assistant",
                "content": chosen,
            }
        ],

        "rejected": [
            {
                "role": "assistant",
                "content": rejected,
            }
        ],
    })


    metadata.append({
        "question_index": qi,
        "risk": risk,
        "judge_correct": int(correct),
        "kind": kind,
        "majority_answer": answer,
    })


random.Random(
    SEED
).shuffle(
    pairs
)


with open(
    PAIR_JSONL,
    "w",
    encoding="utf-8",
) as f:

    for pair in pairs:
        f.write(
            json.dumps(
                pair,
                ensure_ascii=False,
            )
            + "\n"
        )


pd.DataFrame(
    metadata
).to_csv(
    PAIR_METADATA,
    index=False,
)


print()
print("=" * 80)
print("2WIKI CLEAN DPO PAIRS")
print("=" * 80)

print(
    "Answer-preferred:",
    answer_preferred
)

print(
    "Abstain-preferred:",
    abstain_preferred
)

print(
    "Excluded low-risk wrong:",
    excluded
)

print(
    "Total pairs:",
    len(pairs)
)

print(
    "Check:",
    answer_preferred
    + abstain_preferred
    + excluded
)


assert (
    answer_preferred
    + abstain_preferred
    + excluded
    ==
    800
)



assert answer_preferred == 279
assert abstain_preferred == 320
assert excluded == 201
assert len(pairs) == 599


clf = Pipeline([
    (
        "scaler",
        StandardScaler(),
    ),

    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            random_state=SEED,
            max_iter=5000,
            solver="lbfgs",
        ),
    ),
])


X_dev = (
    dev[FULL17]
    .astype(float)
    .to_numpy()
)

y_dev = (
    1
    -
    dev["judge_correct"]
    .astype(int)
    .to_numpy()
)

X_test = (
    test[FULL17]
    .astype(float)
    .to_numpy()
)


clf.fit(
    X_dev,
    y_dev,
)


test_risk = (
    clf.predict_proba(
        X_test
    )[:, 1]
)


test_classifier = (
    test[
        [
            "question_index",
            "question",
            "gold_answer",
            "majority_answer",
            "judge_correct",
        ]
    ]
    .copy()
)


test_classifier[
    "classifier_risk"
] = test_risk


test_classifier[
    "classifier_action"
] = np.where(
    test_risk >= RISK_THRESHOLD,
    "abstain",
    "answer",
)


test_classifier.to_csv(
    TEST_CLASSIFIER_CSV,
    index=False,
)


print()
print("=" * 80)
print("2WIKI TEST CLASSIFIER")
print("=" * 80)

print(
    test_classifier[
        "classifier_action"
    ].value_counts()
)


print()
print(
    "Saved pairs:",
    PAIR_JSONL
)

print(
    "Saved classifier:",
    TEST_CLASSIFIER_CSV
)

2WIKI FROZEN DPO THRESHOLD
Threshold: 0.6541802787173193

2WIKI CLEAN DPO PAIRS
Answer-preferred: 279
Abstain-preferred: 320
Excluded low-risk wrong: 201
Total pairs: 599
Check: 800

2WIKI TEST CLASSIFIER
classifier_action
answer     129
abstain     71
Name: count, dtype: int64

Saved pairs: /content/drive/MyDrive/hedge_run/2wiki_dpo_pairs_FINAL_CLEAN.jsonl
Saved classifier: /content/drive/MyDrive/hedge_run/2wiki_dpo_test_classifier_FINAL_CLEAN.csv


In [ ]:
import os
import gc
import math
import shutil
import inspect
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
)

from trl import (
    DPOConfig,
    DPOTrainer,
)

import trl
import transformers
import peft




D = "/content/drive/MyDrive/hedge_run"

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

PAIR_JSONL = (
    f"{D}/2wiki_dpo_pairs_FINAL_CLEAN.jsonl"
)

OUT_DIR = (
    f"{D}/2wiki_dpo_reliability_FINAL_CLEAN_adapter"
)


SEED = 42

BETA = 0.05
LEARNING_RATE = 1e-5

EPOCHS = 3

BATCH_SIZE = 2
GRAD_ACC = 8


MAX_LENGTH = 1536




for name in [
    "model",
    "base_model",
    "trainer",
]:

    if name in globals():
        del globals()[name]


gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()



print("=" * 80)
print("ENVIRONMENT")
print("=" * 80)

print("TRL:", trl.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("PyTorch:", torch.__version__)


assert torch.cuda.is_available()

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "BF16 supported:",
    torch.cuda.is_bf16_supported()
)

assert torch.cuda.is_bf16_supported()



ds = load_dataset(
    "json",
    data_files=PAIR_JSONL,
    split="train",
)


assert len(ds) == 599


print()
print(
    "Preference pairs:",
    len(ds)
)



micro_batches_per_epoch = math.ceil(
    len(ds) / BATCH_SIZE
)

updates_per_epoch = math.ceil(
    micro_batches_per_epoch / GRAD_ACC
)

total_updates = (
    updates_per_epoch * EPOCHS
)

WARMUP_STEPS = int(
    round(
        0.10 * total_updates
    )
)


print(
    "Micro-batches per epoch:",
    micro_batches_per_epoch
)

print(
    "Optimiser updates per epoch:",
    updates_per_epoch
)

print(
    "Approx total optimiser updates:",
    total_updates
)

print(
    "Warm-up steps:",
    WARMUP_STEPS
)


assert WARMUP_STEPS == 11


config_params = (
    inspect.signature(
        DPOConfig.__init__
    ).parameters
)


print()
print("=" * 80)
print("DPOCONFIG SUPPORT")
print("=" * 80)


for param in [
    "beta",
    "max_length",
    "warmup_steps",
    "disable_dropout",
    "truncation_mode",
]:

    print(
        f"{param:20s}",
        param in config_params
    )


for required in [
    "beta",
    "max_length",
    "warmup_steps",
]:

    assert required in config_params



tokenizer = (
    AutoTokenizer
    .from_pretrained(
        BASE_MODEL
    )
)


if tokenizer.pad_token is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


tokenizer.padding_side = "left"


bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type=
        "nf4",

    bnb_4bit_compute_dtype=
        torch.bfloat16,

    bnb_4bit_use_double_quant=
        True,
)



print()
print("=" * 80)
print("LOADING QWEN")
print("=" * 80)


model = (
    AutoModelForCausalLM
    .from_pretrained(

        BASE_MODEL,

        quantization_config=
            bnb_config,

        device_map={
            "": 0
        },

        dtype=
            torch.bfloat16,
    )
)


model.config.use_cache = False


model = (
    prepare_model_for_kbit_training(
        model
    )
)



peft_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)



if os.path.exists(
    OUT_DIR
):

    print()
    print(
        "Removing existing adapter directory:"
    )

    print(
        OUT_DIR
    )

    shutil.rmtree(
        OUT_DIR
    )


config_kwargs = {

    "output_dir":
        OUT_DIR,

    "beta":
        BETA,

    "learning_rate":
        LEARNING_RATE,

    "num_train_epochs":
        EPOCHS,

    "per_device_train_batch_size":
        BATCH_SIZE,

    "gradient_accumulation_steps":
        GRAD_ACC,

    "warmup_steps":
        WARMUP_STEPS,

    "lr_scheduler_type":
        "cosine",

    "max_length":
        MAX_LENGTH,

    "bf16":
        True,

    "fp16":
        False,

    "gradient_checkpointing":
        True,

    "logging_steps":
        10,

    "save_strategy":
        "epoch",

    "save_total_limit":
        3,

    "report_to":
        "none",

    "seed":
        SEED,
}


optional_kwargs = {

    "loss_type":
        "sigmoid",

    "truncation_mode":
        "keep_start",

    "disable_dropout":
        False,

    "data_seed":
        SEED,

    "gradient_checkpointing_kwargs":
        {
            "use_reentrant":
                False
        },
}


for key, value in optional_kwargs.items():

    if key in config_params:

        config_kwargs[
            key
        ] = value


training_args = DPOConfig(
    **config_kwargs
)


print()
print(
    "DPOConfig successfully created."
)



trainer_params = (
    inspect.signature(
        DPOTrainer.__init__
    ).parameters
)


trainer_kwargs = {

    "model":
        model,

    "ref_model":
        None,

    "args":
        training_args,

    "train_dataset":
        ds,

    "peft_config":
        peft_config,
}


if "processing_class" in trainer_params:

    trainer_kwargs[
        "processing_class"
    ] = tokenizer

elif "tokenizer" in trainer_params:

    trainer_kwargs[
        "tokenizer"
    ] = tokenizer

else:

    raise RuntimeError(
        "No tokenizer/processing_class argument found."
    )


trainer = DPOTrainer(
    **trainer_kwargs
)



print()
print("=" * 80)
print("TRAINABLE PARAMETERS")
print("=" * 80)


if hasattr(
    trainer.model,
    "print_trainable_parameters"
):

    trainer.model.print_trainable_parameters()


print()
print("=" * 80)
print("STARTING CLEAN 2WIKI DPO TRAINING")
print("=" * 80)


train_result = trainer.train()



trainer.save_model(
    OUT_DIR
)

tokenizer.save_pretrained(
    OUT_DIR
)


adapter_config_path = (
    os.path.join(
        OUT_DIR,
        "adapter_config.json"
    )
)


print()
print("=" * 80)
print("2WIKI DPO TRAINING COMPLETE")
print("=" * 80)


print(
    "Adapter:",
    OUT_DIR
)

print(
    "adapter_config exists:",
    os.path.exists(
        adapter_config_path
    )
)


print()
print("Files:")

for filename in os.listdir(
    OUT_DIR
):

    print(
        " ",
        filename
    )


assert os.path.exists(
    adapter_config_path
)


print()
print(
    "TRAINING SUCCESSFUL — SAFE TO EVALUATE."
)

ENVIRONMENT
TRL: 1.10.0
Transformers: 5.15.0
PEFT: 0.20.0
PyTorch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-80GB
BF16 supported: True


Generating train split: 0 examples [00:00, ? examples/s]


Preference pairs: 599
Micro-batches per epoch: 300
Optimiser updates per epoch: 38
Approx total optimiser updates: 114
Warm-up steps: 11

DPOCONFIG SUPPORT
beta                 True
max_length           True
warmup_steps         True
disable_dropout      True
truncation_mode      True


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

...

Dropping fully truncated examples from train dataset:   0%|          | 0/599 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



TRAINABLE PARAMETERS
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273

STARTING CLEAN 2WIKI DPO TRAINING


Step,Training Loss
10,0.673721
20,0.569039
30,0.442050
40,0.448516
50,0.361290
60,0.365109
70,0.367735
80,0.414712
90,0.369123
100,0.404466



2WIKI DPO TRAINING COMPLETE
Adapter: /content/drive/MyDrive/hedge_run/2wiki_dpo_reliability_FINAL_CLEAN_adapter
adapter_config exists: True

Files:
  README.md
  checkpoint-38
  checkpoint-76
  checkpoint-114
  adapter_model.safetensors
  adapter_config.json
  chat_template.jinja
  tokenizer_config.json
  tokenizer.json
  training_args.bin

TRAINING SUCCESSFUL — SAFE TO EVALUATE.


In [ ]:
import os
import gc
import re
import json

import torch

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel


FINAL_JSON = (
    f"{D}/"
    "2wiki_passive_iterative_v2_FINAL_CLEAN_LABELLED_1000.json"
)


TEST_CLASSIFIER_CSV = (
    f"{D}/"
    "2wiki_dpo_test_classifier_FINAL_CLEAN.csv"
)


GEN_PATH = (
    f"{D}/"
    "2wiki_dpo_FINAL_CLEAN_test_generation.json"
)




adapter_config_path = os.path.join(
    OUT_DIR,
    "adapter_config.json"
)


assert os.path.exists(
    adapter_config_path
)


print(
    "Adapter found:",
    adapter_config_path
)



for name in [
    "trainer",
    "model",
]:

    if name in globals():

        del globals()[name]


gc.collect()

torch.cuda.empty_cache()


print(
    "CUDA allocated GB after cleanup:",
    round(
        torch.cuda.memory_allocated()
        / 1e9,
        3
    )
)



with open(
    FINAL_JSON,
    "r",
    encoding="utf-8",
) as f:

    all_records = json.load(f)


test_records = [

    r

    for r in all_records

    if r["split"] == "test"
]


assert len(test_records) == 200


print(
    "Test questions:",
    len(test_records)
)



INSTRUCTION = """You are given a question, retrieved evidence, and an existing reasoning trace.

Decide whether the question can be answered reliably from this information.

If it can be answered reliably, respond only in this form:
FINAL ANSWER: <answer>

If it cannot be answered reliably, respond exactly:
I don't know.

Do not provide any additional explanation."""


def as_list(value):

    if value is None:
        return []

    if isinstance(
        value,
        (list, tuple)
    ):

        return [
            str(x).strip()
            for x in value
            if str(x).strip()
        ]

    return [
        str(value).strip()
    ]


def make_prompt(record):

    question = str(
        record["question"]
    ).strip()


    reasoning = as_list(
        record.get(
            "selected_steps",
            []
        )
    )


    evidence = as_list(
        record.get(
            "question_evidence",
            []
        )
    )


    if not evidence:

        raise RuntimeError(
            "Missing frozen evidence for "
            f"QI={record['question_index']}"
        )


    if reasoning:

        reasoning_text = "\n".join(

            f"{i}. {step}"

            for i, step in enumerate(
                reasoning,
                start=1,
            )
        )

    else:

        reasoning_text = (
            "No intermediate reasoning was produced."
        )


    evidence_text = "\n".join(

        f"E{i}: {sentence}"

        for i, sentence in enumerate(
            evidence,
            start=1,
        )
    )


    return f"""{INSTRUCTION}

QUESTION:
{question}

REASONING TRACE:
{reasoning_text}

RETRIEVED EVIDENCE:
{evidence_text}""".strip()



tokenizer = (
    AutoTokenizer
    .from_pretrained(
        BASE_MODEL
    )
)


if tokenizer.pad_token is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


tokenizer.padding_side = "left"


bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type=
        "nf4",

    bnb_4bit_compute_dtype=
        torch.bfloat16,

    bnb_4bit_use_double_quant=
        True,
)


print()
print(
    "Loading base Qwen..."
)


base_model = (
    AutoModelForCausalLM
    .from_pretrained(

        BASE_MODEL,

        quantization_config=
            bnb_config,

        device_map={
            "": 0
        },

        dtype=
            torch.bfloat16,
    )
)


print(
    "Loading 2Wiki DPO adapter..."
)


model = (
    PeftModel
    .from_pretrained(

        base_model,

        OUT_DIR,
    )
)


model.eval()

model.config.use_cache = True


MODEL_DEVICE = (
    next(
        model.parameters()
    ).device
)


print(
    "Model device:",
    MODEL_DEVICE
)



ABSTAIN_PATTERNS = [

    r"^i don't know\b",
    r"^i do not know\b",
    r"^cannot answer\b",
    r"^can't answer\b",
    r"^unable to answer\b",
    r"^not enough information\b",
]


def parse_output(text):

    text = str(
        text
    ).strip()


    without_prefix = re.sub(

        r"^\s*FINAL\s+ANSWER\s*:\s*",

        "",

        text,

        flags=re.I,
    ).strip()


    low = without_prefix.lower()


    for pattern in ABSTAIN_PATTERNS:

        if re.search(
            pattern,
            low,
        ):

            return (
                "abstain",
                None,
                True,
            )


    match = re.search(

        r"FINAL\s+ANSWER\s*:\s*(.+)",

        text,

        flags=re.I | re.S,
    )


    if match:

        answer = (
            match
            .group(1)
            .strip()
            .splitlines()[0]
            .strip()
        )


        return (
            "answer",
            answer,
            True,
        )


    return (
        "answer",
        text,
        False,
    )



@torch.inference_mode()
def generate_one(record):

    prompt = make_prompt(
        record
    )


    messages = [
        {
            "role":
                "user",

            "content":
                prompt,
        }
    ]


    rendered = tokenizer.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True,
    )


    inputs = tokenizer(

        rendered,

        return_tensors="pt",

        add_special_tokens=False,
    )


    inputs = {

        key:
            value.to(
                MODEL_DEVICE
            )

        for key, value
        in inputs.items()
    }


    input_length = (
        inputs[
            "input_ids"
        ].shape[1]
    )


    output = model.generate(

        **inputs,

        max_new_tokens=
            64,

        do_sample=
            False,

        pad_token_id=
            tokenizer.pad_token_id,

        eos_token_id=
            tokenizer.eos_token_id,
    )


    generated_ids = (
        output[
            0,
            input_length:
        ]
    )


    return (
        tokenizer.decode(

            generated_ids,

            skip_special_tokens=True,

        ).strip()
    )



if os.path.exists(
    GEN_PATH
):

    with open(
        GEN_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        generations = json.load(f)

else:

    generations = {}


print()
print(
    "Already generated:",
    len(generations)
)


for record in tqdm(
    test_records
):

    qi = str(
        int(
            record[
                "question_index"
            ]
        )
    )


    if qi in generations:
        continue


    raw = generate_one(
        record
    )


    (
        action,
        answer,
        format_ok,
    ) = parse_output(
        raw
    )


    generations[
        qi
    ] = {

        "question_index":
            int(qi),

        "raw_response":
            raw,

        "dpo_action":
            action,

        "dpo_answer":
            answer,

        "format_ok":
            bool(
                format_ok
            ),
    }


    with open(
        GEN_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            generations,
            f,
            indent=2,
            ensure_ascii=False,
        )


assert len(generations) == 200



N_ANSWER = sum(

    x[
        "dpo_action"
    ]
    ==
    "answer"

    for x in
    generations.values()
)


N_ABSTAIN = (
    200 - N_ANSWER
)


N_BAD_FORMAT = sum(

    not x[
        "format_ok"
    ]

    for x in
    generations.values()
)


print()
print("=" * 80)
print("2WIKI DPO TEST GENERATION COMPLETE")
print("=" * 80)

print(
    "Answer:",
    N_ANSWER
)

print(
    "Abstain:",
    N_ABSTAIN
)

print(
    "Non-standard format:",
    N_BAD_FORMAT
)

print(
    "Total:",
    len(generations)
)

print()
print(
    "Saved:",
    GEN_PATH
)

Adapter found: /content/drive/MyDrive/hedge_run/2wiki_dpo_reliability_FINAL_CLEAN_adapter/adapter_config.json
CUDA allocated GB after cleanup: 13.564
Test questions: 200

Loading base Qwen...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading 2Wiki DPO adapter...
Model device: cuda:0

Already generated: 200


  0%|          | 0/200 [00:00<?, ?it/s]


2WIKI DPO TEST GENERATION COMPLETE
Answer: 89
Abstain: 111
Non-standard format: 0
Total: 200

Saved: /content/drive/MyDrive/hedge_run/2wiki_dpo_FINAL_CLEAN_test_generation.json


In [ ]:


!pip -q install -U openai

import os
import json
import time
import pandas as pd

from tqdm.auto import tqdm
from openai import OpenAI



JUDGE_PATH = (
    f"{D}/"
    "2wiki_dpo_FINAL_CLEAN_test_judgments.json"
)


RESULT_CSV = (
    f"{D}/"
    "2wiki_dpo_FINAL_CLEAN_test_results.csv"
)


RESULT_JSON = (
    f"{D}/"
    "2wiki_dpo_FINAL_CLEAN_test_summary.json"
)


with open(
    GEN_PATH,
    "r",
    encoding="utf-8",
) as f:

    generations = json.load(f)


assert len(generations) == 200


with open(
    FINAL_JSON,
    "r",
    encoding="utf-8",
) as f:

    all_records = json.load(f)


test_records = [

    r

    for r in all_records

    if r["split"] == "test"
]


assert len(test_records) == 200


records_by_qi = {

    str(
        int(
            r[
                "question_index"
            ]
        )
    ):
        r

    for r in test_records
}



classifier_df = pd.read_csv(
    TEST_CLASSIFIER_CSV
)


assert len(classifier_df) == 200


classifier_by_qi = {

    int(
        row.question_index
    ):
        row

    for row in
    classifier_df.itertuples()
}



OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)


if not OPENAI_API_KEY:

    try:

        from google.colab import userdata

        OPENAI_API_KEY = userdata.get(
            "OPENAI_API_KEY"
        )

    except Exception:

        OPENAI_API_KEY = None


if not OPENAI_API_KEY:

    raise RuntimeError(
        "OPENAI_API_KEY not found."
    )


client = OpenAI(
    api_key=
        OPENAI_API_KEY
)



JUDGE_MODEL = "gpt-4o-mini"


JUDGE_INSTRUCTIONS = """
You are an offline correctness evaluator for a question-answering experiment.

You receive:
1. a question,
2. a reference answer,
3. a model prediction.

Determine whether the prediction should be counted as correct.

Accept:
- semantically equivalent paraphrases,
- aliases,
- spelling variants,
- capitalization differences,
- harmless formatting differences.

Reject:
- factually different answers,
- answers to a different relation,
- reversed comparisons,
- broader or narrower answers that change what was asked.

Judge only the correctness of the final model prediction.

Do not use or infer hidden reasoning, confidence scores, retrieved evidence,
classifier outputs, or reliability features.

Return a binary correctness judgment.
""".strip()


JUDGE_SCHEMA = {

    "type":
        "object",

    "properties": {

        "correct": {
            "type":
                "boolean"
        },

        "reason": {
            "type":
                "string"
        },
    },

    "required": [
        "correct",
        "reason",
    ],

    "additionalProperties":
        False,
}


def judge_answer(
    question,
    gold,
    prediction,
):

    prompt = f"""QUESTION:
{question}

REFERENCE ANSWER:
{gold}

MODEL PREDICTION:
{prediction}"""


    for attempt in range(5):

        try:

            response = (
                client.responses.create(

                    model=
                        JUDGE_MODEL,

                    instructions=
                        JUDGE_INSTRUCTIONS,

                    input=
                        prompt,

                    temperature=
                        0,

                    max_output_tokens=
                        120,

                    store=
                        False,

                    text={
                        "format": {

                            "type":
                                "json_schema",

                            "name":
                                "qa_correctness",

                            "strict":
                                True,

                            "schema":
                                JUDGE_SCHEMA,
                        }
                    },
                )
            )


            return json.loads(
                response.output_text
            )


        except Exception as exc:

            if attempt == 4:

                raise RuntimeError(
                    f"Judge failed: {exc}"
                )


            time.sleep(
                2 ** attempt
            )



if os.path.exists(
    JUDGE_PATH
):

    with open(
        JUDGE_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        judgments = json.load(f)

else:

    judgments = {}


answered_ids = [

    qi

    for qi, item
    in generations.items()

    if item[
        "dpo_action"
    ]
    ==
    "answer"
]


print("=" * 80)
print("2WIKI CORRECTNESS JUDGING")
print("=" * 80)

print(
    "Returned answers:",
    len(answered_ids)
)

print(
    "Already judged:",
    sum(
        qi in judgments
        for qi in answered_ids
    )
)


for qi in tqdm(
    answered_ids,
    desc="Judging 2Wiki DPO answers",
):

    if qi in judgments:
        continue


    record = records_by_qi[
        qi
    ]


    prediction = generations[
        qi
    ][
        "dpo_answer"
    ]


    result = judge_answer(

        question=
            record[
                "question"
            ],

        gold=
            record[
                "gold_answer"
            ],

        prediction=
            prediction,
    )


    judgments[
        qi
    ] = {

        "correct":
            bool(
                result[
                    "correct"
                ]
            ),

        "reason":
            str(
                result[
                    "reason"
                ]
            ).strip(),
    }


    with open(
        JUDGE_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            judgments,
            f,
            indent=2,
            ensure_ascii=False,
        )



rows = []


for record in test_records:

    qi = int(
        record[
            "question_index"
        ]
    )

    key = str(qi)


    gen = generations[
        key
    ]


    dpo_answered = (
        gen[
            "dpo_action"
        ]
        ==
        "answer"
    )


    dpo_correct = (

        bool(
            judgments[
                key
            ][
                "correct"
            ]
        )

        if dpo_answered

        else False
    )


    clf = classifier_by_qi[
        qi
    ]


    rows.append({

        "question_index":
            qi,

        "question":
            record[
                "question"
            ],

        "gold_answer":
            record[
                "gold_answer"
            ],

        "base_answer":
            record[
                "majority_answer"
            ],

        "base_correct":
            int(
                bool(
                    record[
                        "judge_correct"
                    ]
                )
            ),

        "classifier_risk":
            float(
                clf.classifier_risk
            ),

        "classifier_action":
            str(
                clf.classifier_action
            ),

        "dpo_action":
            gen[
                "dpo_action"
            ],

        "dpo_answer":
            gen[
                "dpo_answer"
            ],

        "dpo_raw_response":
            gen[
                "raw_response"
            ],

        "format_ok":
            int(
                bool(
                    gen[
                        "format_ok"
                    ]
                )
            ),

        "dpo_correct":
            int(
                dpo_correct
            ),
    })


result_df = pd.DataFrame(
    rows
)


assert len(result_df) == 200


N = 200


N_BASE_CORRECT = int(
    result_df[
        "base_correct"
    ].sum()
)


N_BASE_WRONG = (
    N - N_BASE_CORRECT
)


BASE_ACCURACY = (
    N_BASE_CORRECT / N
)


# Frozen 2Wiki test result.
assert N_BASE_CORRECT == 84
assert N_BASE_WRONG == 116



dpo_answer = (
    result_df[
        "dpo_action"
    ]
    ==
    "answer"
)


dpo_abstain = (
    ~dpo_answer
)


N_DPO_ANSWER = int(
    dpo_answer.sum()
)


N_DPO_ABSTAIN = (
    N - N_DPO_ANSWER
)


N_DPO_CORRECT = int(

    result_df.loc[
        dpo_answer,
        "dpo_correct"
    ].sum()
)


N_DPO_WRONG = (
    N_DPO_ANSWER
    -
    N_DPO_CORRECT
)


COVERAGE = (
    N_DPO_ANSWER / N
)


SELECTIVE_ACCURACY = (

    N_DPO_CORRECT
    /
    N_DPO_ANSWER

    if N_DPO_ANSWER

    else float("nan")
)


CONFIDENT_ERROR = (
    N_DPO_WRONG / N
)



base_correct = (
    result_df[
        "base_correct"
    ]
    ==
    1
)


N_OVER_ABSTAIN = int(

    (
        base_correct
        &
        dpo_abstain
    ).sum()
)


OVER_ABSTENTION = (

    N_OVER_ABSTAIN
    /
    N_BASE_CORRECT
)




Pc0 = N_BASE_CORRECT / N
Pw0 = N_BASE_WRONG / N

Pc = N_DPO_CORRECT / N
Pw = N_DPO_WRONG / N


THS = (
    (
        Pc * Pw0
        -
        Pw * Pc0
    )
    /
    Pw0
)


THS100 = (
    THS * 100
)



clf_answer = (
    result_df[
        "classifier_action"
    ]
    ==
    "answer"
)


clf_abstain = (
    ~clf_answer
)


N_CLF_ANSWER = int(
    clf_answer.sum()
)


N_CLF_ABSTAIN = (
    N - N_CLF_ANSWER
)


N_CLF_CORRECT = int(

    result_df.loc[
        clf_answer,
        "base_correct"
    ].sum()
)


N_CLF_WRONG = (
    N_CLF_ANSWER
    -
    N_CLF_CORRECT
)


CLF_COVERAGE = (
    N_CLF_ANSWER / N
)


CLF_SELECTIVE_ACCURACY = (

    N_CLF_CORRECT
    /
    N_CLF_ANSWER
)


CLF_CONFIDENT_ERROR = (
    N_CLF_WRONG / N
)



same = (

    result_df[
        "classifier_action"
    ]
    ==
    result_df[
        "dpo_action"
    ]
)


FIDELITY = float(
    same.mean()
)


FIDELITY_ANSWER = float(

    (
        result_df.loc[
            clf_answer,
            "dpo_action"
        ]
        ==
        "answer"
    ).mean()
)


FIDELITY_ABSTAIN = float(

    (
        result_df.loc[
            clf_abstain,
            "dpo_action"
        ]
        ==
        "abstain"
    ).mean()
)


fidelity_table = pd.crosstab(

    result_df[
        "classifier_action"
    ],

    result_df[
        "dpo_action"
    ],

    rownames=[
        "Classifier"
    ],

    colnames=[
        "DPO"
    ],
)


FORMAT_COMPLIANCE = float(
    result_df[
        "format_ok"
    ].mean()
)



result_df.to_csv(
    RESULT_CSV,
    index=False,
)


summary = {

    "dataset":
        "2WikiMultiHopQA",

    "test_n":
        N,

    "baseline": {

        "correct":
            N_BASE_CORRECT,

        "wrong":
            N_BASE_WRONG,

        "accuracy":
            BASE_ACCURACY,
    },

    "classifier": {

        "answered":
            N_CLF_ANSWER,

        "abstained":
            N_CLF_ABSTAIN,

        "correct_returned":
            N_CLF_CORRECT,

        "wrong_returned":
            N_CLF_WRONG,

        "coverage":
            CLF_COVERAGE,

        "selective_accuracy":
            CLF_SELECTIVE_ACCURACY,

        "confident_error":
            CLF_CONFIDENT_ERROR,
    },

    "dpo": {

        "answered":
            N_DPO_ANSWER,

        "abstained":
            N_DPO_ABSTAIN,

        "correct_returned":
            N_DPO_CORRECT,

        "wrong_returned":
            N_DPO_WRONG,

        "coverage":
            COVERAGE,

        "selective_accuracy":
            SELECTIVE_ACCURACY,

        "confident_error":
            CONFIDENT_ERROR,

        "over_abstention_count":
            N_OVER_ABSTAIN,

        "over_abstention":
            OVER_ABSTENTION,

        "THS":
            THS,

        "THSx100":
            THS100,

        "format_compliance":
            FORMAT_COMPLIANCE,
    },

    "fidelity": {

        "overall":
            FIDELITY,

        "when_classifier_answers":
            FIDELITY_ANSWER,

        "when_classifier_abstains":
            FIDELITY_ABSTAIN,
    },
}


with open(
    RESULT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
    )



print()
print("=" * 80)
print("FINAL CLEAN 2WIKI RQ2 RESULTS")
print("=" * 80)


print()
print("BASE MODEL")
print("-" * 50)

print(
    "Correct:",
    N_BASE_CORRECT
)

print(
    "Wrong:",
    N_BASE_WRONG
)

print(
    "Accuracy:",
    round(
        BASE_ACCURACY,
        4,
    )
)


print()
print("FROZEN RELIABILITY CLASSIFIER")
print("-" * 50)

print(
    "Answered:",
    N_CLF_ANSWER
)

print(
    "Abstained:",
    N_CLF_ABSTAIN
)

print(
    "Correct returned:",
    N_CLF_CORRECT
)

print(
    "Wrong returned:",
    N_CLF_WRONG
)

print(
    "Coverage:",
    round(
        CLF_COVERAGE,
        4,
    )
)

print(
    "Selective accuracy:",
    round(
        CLF_SELECTIVE_ACCURACY,
        4,
    )
)

print(
    "Confident error:",
    round(
        CLF_CONFIDENT_ERROR,
        4,
    )
)


print()
print("DPO")
print("-" * 50)

print(
    "Answered:",
    N_DPO_ANSWER
)

print(
    "Abstained:",
    N_DPO_ABSTAIN
)

print(
    "Correct returned:",
    N_DPO_CORRECT
)

print(
    "Wrong returned:",
    N_DPO_WRONG
)

print(
    "Coverage:",
    round(
        COVERAGE,
        4,
    )
)

print(
    "Selective accuracy:",
    round(
        SELECTIVE_ACCURACY,
        4,
    )
)

print(
    "Confident error:",
    round(
        CONFIDENT_ERROR,
        4,
    )
)

print(
    "Over-abstention count:",
    N_OVER_ABSTAIN
)

print(
    "Over-abstention:",
    round(
        OVER_ABSTENTION,
        4,
    )
)

print(
    "THS x100:",
    round(
        THS100,
        2,
    )
)

print(
    "Format compliance:",
    round(
        FORMAT_COMPLIANCE,
        4,
    )
)


print()
print("CLASSIFIER ↔ DPO FIDELITY")
print("-" * 50)

print(
    "Overall:",
    round(
        FIDELITY,
        4,
    )
)

print(
    "When classifier answers:",
    round(
        FIDELITY_ANSWER,
        4,
    )
)

print(
    "When classifier abstains:",
    round(
        FIDELITY_ABSTAIN,
        4,
    )
)


print()
print(
    fidelity_table
)


print()
print(
    "Saved detailed results:",
    RESULT_CSV
)

print(
    "Saved summary:",
    RESULT_JSON
)

print()
print("DONE.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.4/94.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.0/83.0 kB 8.8 MB/s eta 0:00:00
2WIKI CORRECTNESS JUDGING
Returned answers: 89
Already judged: 0


Judging 2Wiki DPO answers:   0%|          | 0/89 [00:00<?, ?it/s]


FINAL CLEAN 2WIKI RQ2 RESULTS

BASE MODEL
--------------------------------------------------
Correct: 84
Wrong: 116
Accuracy: 0.42

FROZEN RELIABILITY CLASSIFIER
--------------------------------------------------
Answered: 129
Abstained: 71
Correct returned: 72
Wrong returned: 57
Coverage: 0.645
Selective accuracy: 0.5581
Confident error: 0.285

DPO
--------------------------------------------------
Answered: 89
Abstained: 111
Correct returned: 61
Wrong returned: 28
Coverage: 0.445
Selective accuracy: 0.6854
Confident error: 0.14
Over-abstention count: 25
Over-abstention: 0.2976
THS x100: 20.36
Format compliance: 1.0

CLASSIFIER ↔ DPO FIDELITY
--------------------------------------------------
Overall: 0.68
When classifier answers: 0.5969
When classifier abstains: 0.831

DPO         abstain  answer
Classifier                 
abstain          59      12
answer           52      77

Saved detailed results: /content/drive/MyDrive/hedge_run/2wiki_dpo_FINAL_CLEAN_test_results.csv
Saved su